# Furas Data Pipeline — Bronze to Silver

## Purpose
This notebook transforms raw tender data from the Bronze layer into clean, standardized, and reliable Silver datasets.

## Input
- Raw tender data stored in Azure Data Lake Storage (ADLS) Bronze layer.
- Data is collected from multiple tender sources with different structures and formats.

## Main Processing
- Read raw data from each source.
- Clean and standardize column names and data types.
- Standardize dates, statuses, tender types, categories, and regions.
- Generate a unique `tender_key` for each tender.
- Map all sources to a common Silver schema.
- Use Delta Lake `MERGE` for incremental source updates.
- Combine the sources into a unified Silver dataset.
- Handle known cross-source duplicate tenders.

## Output
- Source-specific Silver Delta tables.
- One unified and deduplicated Silver tender dataset.

## Technologies
Azure Databricks | PySpark | Delta Lake | ADLS Gen2

## Pipeline Position
Bronze (Raw) → **Silver (Clean & Standardized)** → Gold (Business-Ready)

In [0]:
#checking that Databricks can read the files in bronze layer
#verifies that Databricks can access our Bronze layer in Azure Data Lake

bronze_path = "abfss://bronze@tenderdatalake11.dfs.core.windows.net/"

# list the files:
files = dbutils.fs.ls(bronze_path)

display(files)

path,name,size,modificationTime
abfss://bronze@tenderdatalake11.dfs.core.windows.net/ETIMAD/,ETIMAD/,0,0
abfss://bronze@tenderdatalake11.dfs.core.windows.net/FORSAH/,FORSAH/,0,0
abfss://bronze@tenderdatalake11.dfs.core.windows.net/GEO/,GEO/,0,0
abfss://bronze@tenderdatalake11.dfs.core.windows.net/SAR/,SAR/,0,0
abfss://bronze@tenderdatalake11.dfs.core.windows.net/TAHAKOM/,TAHAKOM/,0,0
abfss://bronze@tenderdatalake11.dfs.core.windows.net/TAQEEM/,TAQEEM/,0,0
abfss://bronze@tenderdatalake11.dfs.core.windows.net/Tanafos/,Tanafos/,0,0
abfss://bronze@tenderdatalake11.dfs.core.windows.net/accredited_valuers/,accredited_valuers/,0,0
abfss://bronze@tenderdatalake11.dfs.core.windows.net/cst/,cst/,0,0
abfss://bronze@tenderdatalake11.dfs.core.windows.net/nupco/,nupco/,0,0


In [0]:
# This function handles incremental loading into the Silver layer
# It uses the unique tender key to check whether a tender already exists. Existing tenders are updated, new tenders are inserted, and if the Silver table does  # not exist yet, the function creates it as a Delta table."

from delta.tables import DeltaTable

def merge_to_silver(source_df, silver_path, source_name):
    """
    Incrementally upsert a source DataFrame into its Silver Delta table.
    """

    if DeltaTable.isDeltaTable(spark, silver_path):
        target = DeltaTable.forPath(spark, silver_path)

        (
            target.alias("target")
            .merge(
                source_df.alias("source"),
                "target.tender_key = source.tender_key"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"{source_name}: Silver MERGE completed.")

    else:
        # First-ever load
        (
            source_df.write
            .format("delta")
            .mode("overwrite")
            .save(silver_path)
        )

        print(f"{source_name}: Silver table created.")

## Source NO.1 : Nupco

In [0]:
# ============================================================
# STEP: Locate NUPCO Bronze data
# Purpose:
# Define the NUPCO Bronze path and verify that the raw
# ingestion files are available before transformation.
# ============================================================

nupco_path = bronze_path + "nupco/"

display(dbutils.fs.ls(nupco_path))

path,name,size,modificationTime
abfss://bronze@tenderdatalake11.dfs.core.windows.net/nupco/2026-09-14/,2026-09-14/,0,1789902679000
abfss://bronze@tenderdatalake11.dfs.core.windows.net/nupco/2026-09-23/,2026-09-23/,0,1790173454000
abfss://bronze@tenderdatalake11.dfs.core.windows.net/nupco/2026-09-27/,2026-09-27/,0,1790494136000
abfss://bronze@tenderdatalake11.dfs.core.windows.net/nupco/2026-09-28/,2026-09-28/,0,1790616008000


In [0]:
#Each ingestion is stored in a date-based Bronze folder. This cell automatically selects the latest NUPCO ingestion folder, allowing the pipeline to process  the newest data without manually changing the path.

nupco_folders = dbutils.fs.ls(nupco_path)

latest_nupco_folder = sorted(
    [f.path for f in nupco_folders if f.isDir()],
    reverse=True
)[0]

print("Latest NUPCO folder:", latest_nupco_folder)

display(dbutils.fs.ls(latest_nupco_folder))

Latest NUPCO folder: abfss://bronze@tenderdatalake11.dfs.core.windows.net/nupco/2026-09-28/


path,name,size,modificationTime
abfss://bronze@tenderdatalake11.dfs.core.windows.net/nupco/2026-09-28/nupco_20260928_172007.json,nupco_20260928_172007.json,4449,1790616008000


In [0]:

# # Get JSON files from the latest NUPCO folder
# nupco_files = [
#     f for f in dbutils.fs.ls(latest_nupco_folder)
#     if not f.isDir() and f.name.endswith(".json")
# ]

# # Select the latest JSON file
# latest_nupco_file = max(
#     nupco_files,
#     key=lambda f: f.modificationTime
# )

# nupco_file = latest_nupco_file.path

# print("Latest NUPCO file:", nupco_file)

# nupco_raw = spark.read.option("multiline", "true").json(nupco_file)

# nupco_raw.printSchema()


# new update for this cell:
# ============================================================
# STEP: Load latest NUPCO Bronze file
# Purpose:
# Find the latest JSON file from the newest NUPCO ingestion
# and load the raw data into a Spark DataFrame.
# ============================================================

nupco_files = [
    f for f in dbutils.fs.ls(latest_nupco_folder)
    if not f.isDir() and f.name.endswith(".json")
]

if not nupco_files:
    raise ValueError(
        f"No JSON files found in {latest_nupco_folder}"
    )

latest_nupco_file = max(
    nupco_files,
    key=lambda f: f.modificationTime
)

nupco_file = latest_nupco_file.path

print("Latest NUPCO file:", nupco_file)

nupco_raw = (
    spark.read
    .option("multiline", "true")
    .json(nupco_file)
)

nupco_raw.printSchema()


Latest NUPCO file: abfss://bronze@tenderdatalake11.dfs.core.windows.net/nupco/2026-09-28/nupco_20260928_172007.json
root
 |-- data: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- bid_opening: string (nullable = true)
 |    |    |-- submission_deadline: string (nullable = true)
 |    |    |-- tender_id: string (nullable = true)
 |    |    |-- tender_name: string (nullable = true)
 |    |    |-- tender_type: string (nullable = true)
 |    |    |-- tender_url: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- pages_scraped: long (nullable = true)
 |-- record_count: long (nullable = true)
 |-- source: string (nullable = true)
 |-- source_url: string (nullable = true)



In [0]:
#NUPCO stores its tender records inside a nested JSON array called data.
#This cell uses Spark's explode function to convert that array into individual rows, giving us one tender per row while preserving the source and ingestion timestamp.

from pyspark.sql.functions import explode, col

nupco_flat = (
    nupco_raw
    .select(
        explode("data").alias("tender"),
        col("source"),
        col("ingestion_timestamp")
    )
)

display(nupco_flat)

tender,source,ingestion_timestamp
"List(Mon, 05/10/2026, Mon, 05/10/2026, NDP0903/26, تقديم نتائج فحوصات مخبرية عن طريق إرسالها لمختبرات مرجعية, Direct Purchase, https://www.nupco.com/en/tender/%d8%aa%d9%82%d8%af%d9%8a%d9%85-%d9%86%d8%aa%d8%a7%d8%a6%d8%ac-%d9%81%d8%ad%d9%88%d8%b5%d8%a7%d8%aa-%d9%85%d8%ae%d8%a8%d8%b1%d9%8a%d8%a9-%d8%b9%d9%86-%d8%b7%d8%b1%d9%8a%d9%82-%d8%a5%d8%b1%d8%b3%d8%a7/)",NUPCO,2026-09-28T17:20:07
"List(Mon, 05/10/2026, Mon, 05/10/2026, NDP0888/26, تأمين أجهزة الطب الإتصالي لصالح تجمع المدينة المنورة الصحي, Direct Purchase, https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a3%d8%ac%d9%87%d8%b2%d8%a9-%d8%a7%d9%84%d8%b7%d8%a8-%d8%a7%d9%84%d8%a5%d8%aa%d8%b5%d8%a7%d9%84%d9%8a-%d9%84%d8%b5%d8%a7%d9%84%d8%ad-%d8%aa%d8%ac%d9%85%d8%b9-%d8%a7/)",NUPCO,2026-09-28T17:20:07
"List(Sun, 04/10/2026, Sun, 04/10/2026, NDP0885/26, تأمين أجهزة لبرنامج التغذية الإكلينيكية – جامعة الملك سعود بن عبدالعزيز للعلوم الصحية في الرياض, Direct Purchase, https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a3%d8%ac%d9%87%d8%b2%d8%a9-%d9%84%d8%a8%d8%b1%d9%86%d8%a7%d9%85%d8%ac-%d8%a7%d9%84%d8%aa%d8%ba%d8%b0%d9%8a%d8%a9-%d8%a7%d9%84%d8%a5%d9%83%d9%84%d9%8a%d9%86%d9%8a/)",NUPCO,2026-09-28T17:20:07
"List(Sun, 04/10/2026, Sat, 10/10/2026, NDP0852/26, تأمين أجهزة طبية – رئاسة أمن الدولة, Direct Purchase, https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a3%d8%ac%d9%87%d8%b2%d8%a9-%d8%b7%d8%a8%d9%8a%d8%a9-%d8%b1%d8%a6%d8%a7%d8%b3%d8%a9-%d8%a3%d9%85%d9%86-%d8%a7%d9%84%d8%af%d9%88%d9%84%d8%a9/)",NUPCO,2026-09-28T17:20:07
"List(Wed, 30/09/2026, Wed, 30/09/2026, NDP0877/26, تأمين مستلزمات طبية – INUPCO, Direct Purchase, https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d9%85%d8%b3%d8%aa%d9%84%d8%b2%d9%85%d8%a7%d8%aa-%d8%b7%d8%a8%d9%8a%d8%a9-inupco/)",NUPCO,2026-09-28T17:20:07
"List(Wed, 07/10/2026, Wed, 07/10/2026, NDP0889/26, تأمين مستلزمات العيون والجلدية, Direct Purchase, https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d9%85%d8%b3%d8%aa%d9%84%d8%b2%d9%85%d8%a7%d8%aa-%d8%a7%d9%84%d8%b9%d9%8a%d9%88%d9%86-%d9%88%d8%a7%d9%84%d8%ac%d9%84%d8%af%d9%8a%d8%a9/)",NUPCO,2026-09-28T17:20:07
"List(Mon, 05/10/2026, Mon, 05/10/2026, NDP0887/26, INUPCO – تأمين بنود أدوية لمنصة السوق الالكتروني من خلال منصة, Direct Purchase, https://www.nupco.com/en/tender/inupco-%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d8%a3%d8%af%d9%88%d9%8a%d8%a9-%d9%84%d9%85%d9%86%d8%b5%d8%a9-%d8%a7%d9%84%d8%b3%d9%88%d9%82-%d8%a7%d9%84%d8%a7%d9%84%d9%83%d8%aa%d8%b1-6/)",NUPCO,2026-09-28T17:20:07
"List(Sun, 04/10/2026, Sun, 04/10/2026, NDP0886/26, INUPCO – تأمين بنود أدوية لمنصة السوق الالكتروني من خلال منصة, Direct Purchase, https://www.nupco.com/en/tender/inupco-%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d8%a3%d8%af%d9%88%d9%8a%d8%a9-%d9%84%d9%85%d9%86%d8%b5%d8%a9-%d8%a7%d9%84%d8%b3%d9%88%d9%82-%d8%a7%d9%84%d8%a7%d9%84%d9%83%d8%aa%d8%b1/)",NUPCO,2026-09-28T17:20:07


In [0]:
#After flattening the NUPCO tender array, this cell extracts the required fields from each nested tender object.
#It also preserves the source and ingestion timestamp for traceability.

nupco_flat = nupco_flat.select(
    col("tender.tender_id").alias("tender_id"),
    col("tender.tender_name").alias("tender_name"),
    col("tender.tender_type").alias("tender_type"),
    col("tender.submission_deadline").alias("submission_deadline"),
    col("tender.bid_opening").alias("bid_opening"),
    col("tender.tender_url").alias("tender_url"),
    col("source"),
    col("ingestion_timestamp")
)

display(nupco_flat)

tender_id,tender_name,tender_type,submission_deadline,bid_opening,tender_url,source,ingestion_timestamp
NDP0903/26,تقديم نتائج فحوصات مخبرية عن طريق إرسالها لمختبرات مرجعية,Direct Purchase,"Mon, 05/10/2026","Mon, 05/10/2026",https://www.nupco.com/en/tender/%d8%aa%d9%82%d8%af%d9%8a%d9%85-%d9%86%d8%aa%d8%a7%d8%a6%d8%ac-%d9%81%d8%ad%d9%88%d8%b5%d8%a7%d8%aa-%d9%85%d8%ae%d8%a8%d8%b1%d9%8a%d8%a9-%d8%b9%d9%86-%d8%b7%d8%b1%d9%8a%d9%82-%d8%a5%d8%b1%d8%b3%d8%a7/,NUPCO,2026-09-28T17:20:07
NDP0888/26,تأمين أجهزة الطب الإتصالي لصالح تجمع المدينة المنورة الصحي,Direct Purchase,"Mon, 05/10/2026","Mon, 05/10/2026",https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a3%d8%ac%d9%87%d8%b2%d8%a9-%d8%a7%d9%84%d8%b7%d8%a8-%d8%a7%d9%84%d8%a5%d8%aa%d8%b5%d8%a7%d9%84%d9%8a-%d9%84%d8%b5%d8%a7%d9%84%d8%ad-%d8%aa%d8%ac%d9%85%d8%b9-%d8%a7/,NUPCO,2026-09-28T17:20:07
NDP0885/26,تأمين أجهزة لبرنامج التغذية الإكلينيكية – جامعة الملك سعود بن عبدالعزيز للعلوم الصحية في الرياض,Direct Purchase,"Sun, 04/10/2026","Sun, 04/10/2026",https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a3%d8%ac%d9%87%d8%b2%d8%a9-%d9%84%d8%a8%d8%b1%d9%86%d8%a7%d9%85%d8%ac-%d8%a7%d9%84%d8%aa%d8%ba%d8%b0%d9%8a%d8%a9-%d8%a7%d9%84%d8%a5%d9%83%d9%84%d9%8a%d9%86%d9%8a/,NUPCO,2026-09-28T17:20:07
NDP0852/26,تأمين أجهزة طبية – رئاسة أمن الدولة,Direct Purchase,"Sat, 10/10/2026","Sun, 04/10/2026",https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a3%d8%ac%d9%87%d8%b2%d8%a9-%d8%b7%d8%a8%d9%8a%d8%a9-%d8%b1%d8%a6%d8%a7%d8%b3%d8%a9-%d8%a3%d9%85%d9%86-%d8%a7%d9%84%d8%af%d9%88%d9%84%d8%a9/,NUPCO,2026-09-28T17:20:07
NDP0877/26,تأمين مستلزمات طبية – INUPCO,Direct Purchase,"Wed, 30/09/2026","Wed, 30/09/2026",https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d9%85%d8%b3%d8%aa%d9%84%d8%b2%d9%85%d8%a7%d8%aa-%d8%b7%d8%a8%d9%8a%d8%a9-inupco/,NUPCO,2026-09-28T17:20:07
NDP0889/26,تأمين مستلزمات العيون والجلدية,Direct Purchase,"Wed, 07/10/2026","Wed, 07/10/2026",https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d9%85%d8%b3%d8%aa%d9%84%d8%b2%d9%85%d8%a7%d8%aa-%d8%a7%d9%84%d8%b9%d9%8a%d9%88%d9%86-%d9%88%d8%a7%d9%84%d8%ac%d9%84%d8%af%d9%8a%d8%a9/,NUPCO,2026-09-28T17:20:07
NDP0887/26,INUPCO – تأمين بنود أدوية لمنصة السوق الالكتروني من خلال منصة,Direct Purchase,"Mon, 05/10/2026","Mon, 05/10/2026",https://www.nupco.com/en/tender/inupco-%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d8%a3%d8%af%d9%88%d9%8a%d8%a9-%d9%84%d9%85%d9%86%d8%b5%d8%a9-%d8%a7%d9%84%d8%b3%d9%88%d9%82-%d8%a7%d9%84%d8%a7%d9%84%d9%83%d8%aa%d8%b1-6/,NUPCO,2026-09-28T17:20:07
NDP0886/26,INUPCO – تأمين بنود أدوية لمنصة السوق الالكتروني من خلال منصة,Direct Purchase,"Sun, 04/10/2026","Sun, 04/10/2026",https://www.nupco.com/en/tender/inupco-%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d8%a3%d8%af%d9%88%d9%8a%d8%a9-%d9%84%d9%85%d9%86%d8%b5%d8%a9-%d8%a7%d9%84%d8%b3%d9%88%d9%82-%d8%a7%d9%84%d8%a7%d9%84%d9%83%d8%aa%d8%b1/,NUPCO,2026-09-28T17:20:07


In [0]:
nupco_flat.count()

8

In [0]:
#This cell checks the structure and data types of the extracted NUPCO data before we standardize it into the Silver schema.

nupco_flat.printSchema()

root
 |-- tender_id: string (nullable = true)
 |-- tender_name: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- submission_deadline: string (nullable = true)
 |-- bid_opening: string (nullable = true)
 |-- tender_url: string (nullable = true)
 |-- source: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)



In [0]:
#This validation cell checks the number of extracted NUPCO tenders and displays their current schema before standardization.

print("Number of tenders:", nupco_flat.count())

nupco_flat.printSchema()

In [0]:
#This cell inspects sample NUPCO date and timestamp values so we can understand their raw formats before converting them to standardized Silver data types.

nupco_flat.select(
    "submission_deadline",
    "bid_opening",
    "ingestion_timestamp"
).show(10, truncate=False)

In [0]:
#This cell performs a data-quality check by counting the number of missing values in every extracted NUPCO column.

from pyspark.sql.functions import col, sum as spark_sum

nupco_flat.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in nupco_flat.columns
]).show()

+---------+-----------+-----------+-------------------+-----------+----------+------+-------------------+
|tender_id|tender_name|tender_type|submission_deadline|bid_opening|tender_url|source|ingestion_timestamp|
+---------+-----------+-----------+-------------------+-----------+----------+------+-------------------+
|        0|          0|          0|                  0|          0|         0|     0|                  0|
+---------+-----------+-----------+-------------------+-----------+----------+------+-------------------+



In [0]:
#This cell performs the main NUPCO cleaning and standardization.
#It cleans text fields, converts raw date strings into proper date types, standardizes the source name.
#and generates a unique SHA-256 tender key. The result is a cleaned NUPCO dataset that can then be mapped to the complete common Silver schema.

from pyspark.sql.functions import (
    col,
    trim,
    to_date,
    to_timestamp,
    lit,
    sha2,
    concat_ws,
    regexp_replace
)

nupco_silver = (
    nupco_flat

    # Clean text
    .withColumn("source_tender_id", trim(col("tender_id")))
    .withColumn("tender_name", trim(col("tender_name")))
    .withColumn("tender_type", trim(col("tender_type")))

    # Remove weekday and convert to Date
    # Example: "Thu, 01/10/2026" -> "01/10/2026"
    .withColumn(
        "closing_date",
        to_date(
            regexp_replace(col("submission_deadline"), r"^[A-Za-z]{3},\s*", ""),
            "dd/MM/yyyy"
        )
    )
    .withColumn(
        "opening_date",
        to_date(
            regexp_replace(col("bid_opening"), r"^[A-Za-z]{3},\s*", ""),
            "dd/MM/yyyy"
        )
    )

    # Convert ingestion timestamp
    .withColumn(
        "ingestion_timestamp",
        to_timestamp(col("ingestion_timestamp"))
    )

    # Standard source name
    .withColumn("source", lit("nupco"))

    # Generate unique Furas key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # Select Silver columns
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "tender_name",
        "tender_type",
        "closing_date",
        "opening_date",
        "tender_url",
        "ingestion_timestamp"
    )
)

In [0]:
display(nupco_silver)

tender_key,source,source_tender_id,tender_name,tender_type,closing_date,opening_date,tender_url,ingestion_timestamp
f60b3383d0bbb56ac9e5fc7ac1df0d92a43e8ae80995e20b739e28e22798cb08,nupco,NDP0903/26,تقديم نتائج فحوصات مخبرية عن طريق إرسالها لمختبرات مرجعية,Direct Purchase,2026-10-05,2026-10-05,https://www.nupco.com/en/tender/%d8%aa%d9%82%d8%af%d9%8a%d9%85-%d9%86%d8%aa%d8%a7%d8%a6%d8%ac-%d9%81%d8%ad%d9%88%d8%b5%d8%a7%d8%aa-%d9%85%d8%ae%d8%a8%d8%b1%d9%8a%d8%a9-%d8%b9%d9%86-%d8%b7%d8%b1%d9%8a%d9%82-%d8%a5%d8%b1%d8%b3%d8%a7/,2026-09-28T17:20:07.000Z
3f2eeed20f502b980f184ffc8b5b1308bb524c97e003f32b7bed8f98c3bcff0a,nupco,NDP0888/26,تأمين أجهزة الطب الإتصالي لصالح تجمع المدينة المنورة الصحي,Direct Purchase,2026-10-05,2026-10-05,https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a3%d8%ac%d9%87%d8%b2%d8%a9-%d8%a7%d9%84%d8%b7%d8%a8-%d8%a7%d9%84%d8%a5%d8%aa%d8%b5%d8%a7%d9%84%d9%8a-%d9%84%d8%b5%d8%a7%d9%84%d8%ad-%d8%aa%d8%ac%d9%85%d8%b9-%d8%a7/,2026-09-28T17:20:07.000Z
7350c77ed19e997b20ea25be665560bce88f84813b0375b671429f78260b49f1,nupco,NDP0885/26,تأمين أجهزة لبرنامج التغذية الإكلينيكية – جامعة الملك سعود بن عبدالعزيز للعلوم الصحية في الرياض,Direct Purchase,2026-10-04,2026-10-04,https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a3%d8%ac%d9%87%d8%b2%d8%a9-%d9%84%d8%a8%d8%b1%d9%86%d8%a7%d9%85%d8%ac-%d8%a7%d9%84%d8%aa%d8%ba%d8%b0%d9%8a%d8%a9-%d8%a7%d9%84%d8%a5%d9%83%d9%84%d9%8a%d9%86%d9%8a/,2026-09-28T17:20:07.000Z
466d1f4087a1400071a59abfdae634ba0da1b2d30960e05e97d2ac4e7a357fa9,nupco,NDP0852/26,تأمين أجهزة طبية – رئاسة أمن الدولة,Direct Purchase,2026-10-10,2026-10-04,https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a3%d8%ac%d9%87%d8%b2%d8%a9-%d8%b7%d8%a8%d9%8a%d8%a9-%d8%b1%d8%a6%d8%a7%d8%b3%d8%a9-%d8%a3%d9%85%d9%86-%d8%a7%d9%84%d8%af%d9%88%d9%84%d8%a9/,2026-09-28T17:20:07.000Z
bcf136854aafc2a03e47fbf782e4a60be4b0baba540a4e6d04e11310f83fd5b8,nupco,NDP0877/26,تأمين مستلزمات طبية – INUPCO,Direct Purchase,2026-09-30,2026-09-30,https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d9%85%d8%b3%d8%aa%d9%84%d8%b2%d9%85%d8%a7%d8%aa-%d8%b7%d8%a8%d9%8a%d8%a9-inupco/,2026-09-28T17:20:07.000Z
a44c65099c3e0e1f5e00831e0d7507ded6920f15a830d680b707555120fff15d,nupco,NDP0889/26,تأمين مستلزمات العيون والجلدية,Direct Purchase,2026-10-07,2026-10-07,https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d9%85%d8%b3%d8%aa%d9%84%d8%b2%d9%85%d8%a7%d8%aa-%d8%a7%d9%84%d8%b9%d9%8a%d9%88%d9%86-%d9%88%d8%a7%d9%84%d8%ac%d9%84%d8%af%d9%8a%d8%a9/,2026-09-28T17:20:07.000Z
333d058df68b99bcd935b22ba11e31628cf1d1b546825e7fa351905382f5aa7b,nupco,NDP0887/26,INUPCO – تأمين بنود أدوية لمنصة السوق الالكتروني من خلال منصة,Direct Purchase,2026-10-05,2026-10-05,https://www.nupco.com/en/tender/inupco-%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d8%a3%d8%af%d9%88%d9%8a%d8%a9-%d9%84%d9%85%d9%86%d8%b5%d8%a9-%d8%a7%d9%84%d8%b3%d9%88%d9%82-%d8%a7%d9%84%d8%a7%d9%84%d9%83%d8%aa%d8%b1-6/,2026-09-28T17:20:07.000Z
c758733c0c43b36cce1644c05e3af8847ba6cb629ba6f7da7cafd454f1fb256c,nupco,NDP0886/26,INUPCO – تأمين بنود أدوية لمنصة السوق الالكتروني من خلال منصة,Direct Purchase,2026-10-04,2026-10-04,https://www.nupco.com/en/tender/inupco-%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d8%a3%d8%af%d9%88%d9%8a%d8%a9-%d9%84%d9%85%d9%86%d8%b5%d8%a9-%d8%a7%d9%84%d8%b3%d9%88%d9%82-%d8%a7%d9%84%d8%a7%d9%84%d9%83%d8%aa%d8%b1/,2026-09-28T17:20:07.000Z


In [0]:
#This cell checks whether the previous transformation produced a healthy nupco_silver dataset before we continue to the complete Silver schema and write it.

nupco_silver.printSchema()

print(
    "Null closing dates:",
    nupco_silver.filter(col("closing_date").isNull()).count()
)

print(
    "Null opening dates:",
    nupco_silver.filter(col("opening_date").isNull()).count()
)

print("Total rows:", nupco_silver.count())

print(
    "Unique tender keys:",
    nupco_silver.select("tender_key").distinct().count()
)

root
 |-- tender_key: string (nullable = true)
 |-- source: string (nullable = false)
 |-- source_tender_id: string (nullable = true)
 |-- tender_name: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- closing_date: date (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- tender_url: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

Null closing dates: 0
Null opening dates: 0
Total rows: 8
Unique tender keys: 8


In [0]:
#This cell defines the standard schema that all tender sources must follow in the Silver layer.
#Because every tender source has a different structure, we created one common Silver schema containing 24 standardized columns.
#Each source is transformed to match this schema before the datasets are combined.
#It standardizes identifiers, dates, statuses, categories, and other tender information while still preserving important original source values.

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
    ArrayType,
    DecimalType,
    BooleanType
)

silver_schema = StructType([
    StructField("tender_key", StringType(), True),
    StructField("source", StringType(), False),
    StructField("source_tender_id", StringType(), True),
    StructField("reference_number", StringType(), True),

    StructField("tender_name", StringType(), True),
    StructField("tender_name_en", StringType(), True),
    StructField("description", StringType(), True),
    StructField("agency_name", StringType(), True),

    StructField("source_tender_type", StringType(), True),
    StructField("tender_type", StringType(), True),

    StructField("source_status", StringType(), True),
    StructField("status", StringType(), True),

    StructField("publish_date", DateType(), True),
    StructField("closing_date", DateType(), True),
    StructField("opening_date", DateType(), True),

    StructField("categories", ArrayType(StringType()), True),
    StructField("regions", ArrayType(StringType()), True),

    StructField("estimated_value_min", DecimalType(18, 2), True),
    StructField("estimated_value_max", DecimalType(18, 2), True),
    StructField("document_price", DecimalType(18, 2), True),
    StructField("currency", StringType(), True),

    StructField("tender_url", StringType(), True),
    StructField("is_tech", BooleanType(), True),

    StructField("ingestion_timestamp", TimestampType(), True)
])

In [0]:
# NUPCO does not provide every field required by our common Silver schema. This cell maps the cleaned NUPCO data to the complete 24-column schema.

from pyspark.sql.functions import col, lit

nupco_silver_final = (
    nupco_silver

    # Identification
    .withColumn("reference_number", lit(None).cast("string"))

    # Tender information not provided by NUPCO
    .withColumn("tender_name_en", lit(None).cast("string"))
    .withColumn("description", lit(None).cast("string"))
    .withColumn("agency_name", lit(None).cast("string"))

    # Preserve original NUPCO type
    .withColumn("source_tender_type", col("tender_type"))

    # Status
    .withColumn("source_status", lit(None).cast("string"))
    .withColumn("status", lit(None).cast("string"))

    # Dates
    .withColumn("publish_date", lit(None).cast("date"))

    # Classification
    .withColumn("categories", lit(None).cast("array<string>"))
    .withColumn("regions", lit(None).cast("array<string>"))

    # Financial information
    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn("currency", lit(None).cast("string"))

    # Furas classification
    .withColumn("is_tech", lit(None).cast("boolean"))

    # Put columns in canonical Silver order
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
nupco_silver_final.printSchema()

root
 |-- tender_key: string (nullable = true)
 |-- source: string (nullable = false)
 |-- source_tender_id: string (nullable = true)
 |-- reference_number: string (nullable = true)
 |-- tender_name: string (nullable = true)
 |-- tender_name_en: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- source_tender_type: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- source_status: string (nullable = true)
 |-- status: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- closing_date: date (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- regions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- estimated_value_min: decimal(18,2) (nullable = true)
 |-- estimated_value_max: decimal(18,2) (nullable = true)
 |-- document_price: decimal(18,2) (nullable = tru

In [0]:
print("Rows:", nupco_silver_final.count())
print("Columns:", len(nupco_silver_final.columns))

Rows: 8
Columns: 24


In [0]:
display(nupco_silver_final)

tender_key,source,source_tender_id,reference_number,tender_name,tender_name_en,description,agency_name,source_tender_type,tender_type,source_status,status,publish_date,closing_date,opening_date,categories,regions,estimated_value_min,estimated_value_max,document_price,currency,tender_url,is_tech,ingestion_timestamp
f60b3383d0bbb56ac9e5fc7ac1df0d92a43e8ae80995e20b739e28e22798cb08,nupco,NDP0903/26,null,تقديم نتائج فحوصات مخبرية عن طريق إرسالها لمختبرات مرجعية,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-10-05,2026-10-05,null,null,null,null,null,null,https://www.nupco.com/en/tender/%d8%aa%d9%82%d8%af%d9%8a%d9%85-%d9%86%d8%aa%d8%a7%d8%a6%d8%ac-%d9%81%d8%ad%d9%88%d8%b5%d8%a7%d8%aa-%d9%85%d8%ae%d8%a8%d8%b1%d9%8a%d8%a9-%d8%b9%d9%86-%d8%b7%d8%b1%d9%8a%d9%82-%d8%a5%d8%b1%d8%b3%d8%a7/,null,2026-09-28T17:20:07.000Z
3f2eeed20f502b980f184ffc8b5b1308bb524c97e003f32b7bed8f98c3bcff0a,nupco,NDP0888/26,null,تأمين أجهزة الطب الإتصالي لصالح تجمع المدينة المنورة الصحي,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-10-05,2026-10-05,null,null,null,null,null,null,https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a3%d8%ac%d9%87%d8%b2%d8%a9-%d8%a7%d9%84%d8%b7%d8%a8-%d8%a7%d9%84%d8%a5%d8%aa%d8%b5%d8%a7%d9%84%d9%8a-%d9%84%d8%b5%d8%a7%d9%84%d8%ad-%d8%aa%d8%ac%d9%85%d8%b9-%d8%a7/,null,2026-09-28T17:20:07.000Z
7350c77ed19e997b20ea25be665560bce88f84813b0375b671429f78260b49f1,nupco,NDP0885/26,null,تأمين أجهزة لبرنامج التغذية الإكلينيكية – جامعة الملك سعود بن عبدالعزيز للعلوم الصحية في الرياض,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-10-04,2026-10-04,null,null,null,null,null,null,https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a3%d8%ac%d9%87%d8%b2%d8%a9-%d9%84%d8%a8%d8%b1%d9%86%d8%a7%d9%85%d8%ac-%d8%a7%d9%84%d8%aa%d8%ba%d8%b0%d9%8a%d8%a9-%d8%a7%d9%84%d8%a5%d9%83%d9%84%d9%8a%d9%86%d9%8a/,null,2026-09-28T17:20:07.000Z
466d1f4087a1400071a59abfdae634ba0da1b2d30960e05e97d2ac4e7a357fa9,nupco,NDP0852/26,null,تأمين أجهزة طبية – رئاسة أمن الدولة,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-10-10,2026-10-04,null,null,null,null,null,null,https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a3%d8%ac%d9%87%d8%b2%d8%a9-%d8%b7%d8%a8%d9%8a%d8%a9-%d8%b1%d8%a6%d8%a7%d8%b3%d8%a9-%d8%a3%d9%85%d9%86-%d8%a7%d9%84%d8%af%d9%88%d9%84%d8%a9/,null,2026-09-28T17:20:07.000Z
bcf136854aafc2a03e47fbf782e4a60be4b0baba540a4e6d04e11310f83fd5b8,nupco,NDP0877/26,null,تأمين مستلزمات طبية – INUPCO,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-09-30,2026-09-30,null,null,null,null,null,null,https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d9%85%d8%b3%d8%aa%d9%84%d8%b2%d9%85%d8%a7%d8%aa-%d8%b7%d8%a8%d9%8a%d8%a9-inupco/,null,2026-09-28T17:20:07.000Z
a44c65099c3e0e1f5e00831e0d7507ded6920f15a830d680b707555120fff15d,nupco,NDP0889/26,null,تأمين مستلزمات العيون والجلدية,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-10-07,2026-10-07,null,null,null,null,null,null,https://www.nupco.com/en/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d9%85%d8%b3%d8%aa%d9%84%d8%b2%d9%85%d8%a7%d8%aa-%d8%a7%d9%84%d8%b9%d9%8a%d9%88%d9%86-%d9%88%d8%a7%d9%84%d8%ac%d9%84%d8%af%d9%8a%d8%a9/,null,2026-09-28T17:20:07.000Z
333d058df68b99bcd935b22ba11e31628cf1d1b546825e7fa351905382f5aa7b,nupco,NDP0887/26,null,INUPCO – تأمين بنود أدوية لمنصة السوق الالكتروني من خلال منصة,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-10-05,2026-10-05,null,null,null,null,null,null,https://www.nupco.com/en/tender/inupco-%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d8%a3%d8%af%d9%88%d9%8a%d8%a9-%d9%84%d9%85%d9%86%d8%b5%d8%a9-%d8%a7%d9%84%d8%b3%d9%88%d9%82-%d8%a7%d9%84%d8%a7%d9%84%d9%83%d8%aa%d8%b1-6/,null,2026-09-28T17:20:07.000Z
c758733c0c43b36cce1644c05e3af8847ba6cb629ba6f7da7cafd454f1fb256c,nupco,NDP0886/26,null,INUPCO – تأمين بنود أدوية لمنصة السوق الالكتروني من خلال منصة,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-1

In [0]:
# After mapping NUPCO to the common Silver schema, this cell performs a final completeness check on the most important NUPCO fields.

from pyspark.sql.functions import col, sum as spark_sum

important_columns = [
    "tender_key",
    "source",
    "source_tender_id",
    "tender_name",
    "closing_date",
    "tender_url",
    "ingestion_timestamp"
]

nupco_silver_final.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in important_columns
]).show()

+----------+------+----------------+-----------+------------+----------+-------------------+
|tender_key|source|source_tender_id|tender_name|closing_date|tender_url|ingestion_timestamp|
+----------+------+----------------+-----------+------------+----------+-------------------+
|         0|     0|               0|          0|           0|         0|                  0|
+----------+------+----------------+-----------+------------+----------+-------------------+



In [0]:
# Check for Duplicate NUPCO Tenders

nupco_silver_final \
    .groupBy("source", "source_tender_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+------+----------------+-----+
|source|source_tender_id|count|
+------+----------------+-----+
+------+----------------+-----+



In [0]:
# This cell is the final NUPCO loading step. It sends the standardized NUPCO dataset to its Silver Delta table.
# Our reusable merge function updates existing tenders using their unique tender key and inserts new tenders. On the first run, it creates the Delta table #automatically.

silver_nupco_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/nupco"
)

merge_to_silver(
    nupco_silver_final,
    silver_nupco_path,
    "NUPCO"
)

NUPCO: Silver MERGE completed.


In [0]:
# After merging NUPCO into Silver, this cell reads the stored Delta table back from ADLS and verifies its row count and number of columns.
# This confirms that the Silver write was successful and that the stored dataset follows our 24-column schema.

nupco_silver_check = (
    spark.read
    .format("delta")
    .load(silver_nupco_path)
)

print("Silver rows:", nupco_silver_check.count())
print("Silver columns:", len(nupco_silver_check.columns))

display(nupco_silver_check)

Silver rows: 154
Silver columns: 24


tender_key,source,source_tender_id,reference_number,tender_name,tender_name_en,description,agency_name,source_tender_type,tender_type,source_status,status,publish_date,closing_date,opening_date,categories,regions,estimated_value_min,estimated_value_max,document_price,currency,tender_url,is_tech,ingestion_timestamp
d8d31c7c2f9aa6bd23d76e2c5d5cf33484c2d976e09fefdedb2d9a8b4194bc68,nupco,NDP0849/26,null,تأمين بنود مستلزمات طبية للتجمعات الصحية,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-10-01,2026-10-01,null,null,null,null,null,null,https://www.nupco.com/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d9%85%d8%b3%d8%aa%d9%84%d8%b2%d9%85%d8%a7%d8%aa-%d8%b7%d8%a8%d9%8a%d8%a9-%d9%84%d9%84%d8%aa%d8%ac%d9%85%d8%b9%d8%a7%d8%aa-%d8%a7%d9%84%d8%b5-2/,null,2026-09-14T23:51:02.000Z
c6dfa221776bfe1d82136fd9abe6c3c8b799ad3e114b19982f526a2eacc9046b,nupco,NDP0846/26,null,تأمين بنود مستلزمات طبية للتجمعات الصحية,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-10-01,2026-10-01,null,null,null,null,null,null,https://www.nupco.com/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d9%85%d8%b3%d8%aa%d9%84%d8%b2%d9%85%d8%a7%d8%aa-%d8%b7%d8%a8%d9%8a%d8%a9-%d9%84%d9%84%d8%aa%d8%ac%d9%85%d8%b9%d8%a7%d8%aa-%d8%a7%d9%84%d8%b5/,null,2026-09-14T23:51:02.000Z
d4e6941be36d630063031c072fbf9121235cb5412e4d9c9d0e06ccc401595137,nupco,NDP0845/26,null,تأمين اكسسوارات طبية– الشؤون الصحية بوزارة الحرس الوطني,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-09-21,2026-09-21,null,null,null,null,null,null,https://www.nupco.com/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a7%d9%83%d8%b3%d8%b3%d9%88%d8%a7%d8%b1%d8%a7%d8%aa-%d8%b7%d8%a8%d9%8a%d8%a9-%d8%a7%d9%84%d8%b4%d8%a4%d9%88%d9%86-%d8%a7%d9%84%d8%b5%d8%ad%d9%8a%d8%a9/,null,2026-09-14T23:51:02.000Z
f41e7d0bb3c8acb1aefd7c273f17d2afb6300aca48cd35e5be2eb571efb765cc,nupco,NDP0847-26,null,INUPCO – تأمين بنود أدوية لمنصة السوق الالكتروني من خلال منصة,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-09-22,2026-09-22,null,null,null,null,null,null,https://www.nupco.com/tender/inupco-%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d8%a3%d8%af%d9%88%d9%8a%d8%a9-%d9%84%d9%85%d9%86%d8%b5%d8%a9-%d8%a7%d9%84%d8%b3%d9%88%d9%82-%d8%a7%d9%84%d8%a7%d9%84%d9%83%d8%aa%d8%b1-7/,null,2026-09-14T23:51:02.000Z
6bf06fc426eaa5b4c604d3258c3f7a1af98e9a1a6b00633195773efa4f140f97,nupco,NDP0848/26,null,تأمين بنود أدوية للتجمعات الصحية,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-10-01,2026-10-01,null,null,null,null,null,null,https://www.nupco.com/tender/%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d8%a3%d8%af%d9%88%d9%8a%d8%a9-%d9%84%d9%84%d8%aa%d8%ac%d9%85%d8%b9%d8%a7%d8%aa-%d8%a7%d9%84%d8%b5%d8%ad%d9%8a%d8%a9/,null,2026-09-14T23:51:02.000Z
9813468dcffac36cb6f8ee71a3b4e150fed197bf6b5b2680f55cff6bc24346b7,nupco,NDP0844/26,null,INUPCO – تأمين بنود أدوية لمنصة السوق الالكتروني من خلال منصة,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-09-20,2026-09-20,null,null,null,null,null,null,https://www.nupco.com/tender/inupco-%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d8%a3%d8%af%d9%88%d9%8a%d8%a9-%d9%84%d9%85%d9%86%d8%b5%d8%a9-%d8%a7%d9%84%d8%b3%d9%88%d9%82-%d8%a7%d9%84%d8%a7%d9%84%d9%83%d8%aa%d8%b1-6/,null,2026-09-14T23:51:02.000Z
a2cea132df561c9a843eefa6480b5121c1c8ed73d888301ec596272b46260a2e,nupco,NDP0843/26,null,INUPCO – تأمين بنود أدوية لمنصة السوق الالكتروني من خلال منصة,null,null,null,Direct Purchase,Direct Purchase,null,null,null,2026-09-20,2026-09-20,null,null,null,null,null,null,https://www.nupco.com/tender/inupco-%d8%aa%d8%a3%d9%85%d9%8a%d9%86-%d8%a8%d9%86%d9%88%d8%af-%d8%a3%d8%af%d9%88%d9%8a%d8%a9-%d9%84%d9%85%d9%86%d8%b5%d8%a9-%d8%a7%d9%84%d8%b3%d9%88%d9%82-%d8%a7%d9%84%d8%a7%d9%84%d9%83%d8%aa%d8%b1-5/,null,2026-09-14T23:51:02.000Z
6f21a1cf23ee39a88821aa76cc592934ffb252cd7d427025583e1af984474255,nupco,NPT0032/26,null,الإتفاقية الإطارية المفتوحة لتأمين مستلزمات الأسنان ال

# Source NO.2 : STC Bronze → Silver

In [0]:
# Read the stc path in the bronze layer

stc_path = bronze_path + "stc/"
display(dbutils.fs.ls(stc_path))

path,name,size,modificationTime
abfss://bronze@tenderdatalake11.dfs.core.windows.net/stc/2026-09-14/,2026-09-14/,0,1789902679000
abfss://bronze@tenderdatalake11.dfs.core.windows.net/stc/2026-09-23/,2026-09-23/,0,1790175858000


In [0]:
# checking and use the latest stc folder 

stc_folders = dbutils.fs.ls(stc_path)

latest_stc_folder = sorted(
    [f.path for f in stc_folders if f.isDir()],
    reverse=True
)[0]

print("Latest STC folder:", latest_stc_folder)

display(dbutils.fs.ls(latest_stc_folder))

Latest STC folder: abfss://bronze@tenderdatalake11.dfs.core.windows.net/stc/2026-09-23/


path,name,size,modificationTime
abfss://bronze@tenderdatalake11.dfs.core.windows.net/stc/2026-09-23/stc_20260923_150418.json,stc_20260923_150418.json,148878,1790175859000


In [0]:

stc_files = [
    f for f in dbutils.fs.ls(latest_stc_folder)
    if not f.isDir() and f.name.endswith(".json")
]


latest_stc_file = max(
    stc_files,
    key=lambda f: f.modificationTime
)

stc_file = latest_stc_file.path

print("Latest STC file:", stc_file)
stc_raw = (
    spark.read
    .option("multiline", "true")
    .json(stc_file)
)

stc_raw.printSchema()

# new cell here:
# ============================================================
# STEP: Load the latest STC Bronze file
# Purpose:
# Find the newest JSON file in the latest STC ingestion folder
# and load it into Spark for transformation.


Latest STC file: abfss://bronze@tenderdatalake11.dfs.core.windows.net/stc/2026-09-23/stc_20260923_150418.json
root
 |-- data: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- agency_name: string (nullable = true)
 |    |    |-- confidence: double (nullable = true)
 |    |    |-- criticality_level: string (nullable = true)
 |    |    |-- details_url: string (nullable = true)
 |    |    |-- portfolio_domain: string (nullable = true)
 |    |    |-- portfolio_fit_level: string (nullable = true)
 |    |    |-- publish_date: string (nullable = true)
 |    |    |-- reason: string (nullable = true)
 |    |    |-- reference_number: string (nullable = true)
 |    |    |-- status: string (nullable = true)
 |    |    |-- status_ar: string (nullable = true)
 |    |    |-- stc_portfolio_company: string (nullable = true)
 |    |    |-- stc_product: string (nullable = true)
 |    |    |-- tender_activity_name: string (nullable = true)
 |    |    |-- tender_name: s

In [0]:
display(stc_raw)

data ingestion_timestamp record_count source source_url List(List(20251231SAUDI IRRIGATION ORGANIZATION (S, 0.98, High, https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=Z2pJczVTIefeWrCAJ43iNQ==, Systems Integration, Strong, 2026-09-19, The tender explicitly procures Oracle database software licences. Under the approved third-party enterprise software procurement rule, this maps to solutions by stc; no sovereign cloud hosting or stc connectivity service is specified., 2000002893, open, مفتوح, solutions by stc, Oracle Database Software Licensing & Enablement, تقنية المعلومات, توريد رخص قواعد بيانات أوراكل, منافسة عامة), List(صندوق التنمية العقارية, 0.85, High, https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=5BQuVrbCXegdQ91MJ7qMSA==, Cybersecurity, Strong, 2026-09-17, Tender is for renewal of licenses and support for cybersecurity event management and monitoring (SIEM-type capability), directly aligning with sirar by stc's SOC/SIEM and monitoring offerings., 260839003675, open, مفتوح, sirar by stc, SIEM / Security Event Monitoring & Management, تقنية المعلومات, تجديد رخص ودعم إدارة احداث الامن السيبراني ومراقبتها, منافسة عامة), List(جامعة أم القرى, 0.6, High, https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=8 HFpJXmpHBqF*@@**Ss7b16DQ==, Systems Integration, Moderate, 2026-09-17, Tender scope is physical transfer and relocation of main and secondary data center components to a new data center for Umm Al-Qura University. This is a systems integration/migration engagement rather than a sovereign cloud platform deployment or STC-owned colocation facility, so it maps to solutions by stc as the integrator handling end-to-end DC relocation., 260939001960, open, مفتوح, solutions by stc, Data Center Migration & Relocation Services, تقنية المعلومات, نقل وترحيل مكونات مركز البيانات الرئيسي والفرعي لمركز البيانات الجديد, منافسة عامة), List(المديرية العامة للأحوال المدنية, 0.55, High, https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=xgvkn8OxvqA8UA0e50iiBw==, Connectivity, Moderate, 2026-09-17, Tender requests connecting a government building (Civil Registry) to the Ministry of Interior's private fiber optic network (MiFON) to ensure secure identity-service continuity. This is a direct enterprise fiber connectivity link between a government building and an existing dedicated network, not a wholesale dark-fiber lease or carrier infrastructure sharing arrangement, so it maps to stc Business fiber link/FTTB connectivity rather than stc Wholesale Dark Fiber., 260939003967, open, مفتوح, stc Business, Business Fiber Link / FTTB Connectivity, تجارة الأدوات والآلات والأجهزة, ربط مبنى وكالة الأحوال المدنية بشبكة الألياف البصرية الخاصة بوزارة الداخلية (MiFON), منافسة عامة), List(الهيئة السعودية للبيانات والذكاء الاصطناعي, 0.6, High, https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=hmbY6frQ7h4 5Nad9fnxJw==, Systems Integration, Moderate, 2026-09-17, The tender seeks support services for systems, resources and tools operating SDAIA's 'Baseer' platform, i.e., ongoing operational support and maintenance of an existing government data/AI platform. This falls under managed platform operations and IT support rather than a named STC connectivity or cloud product, aligning with solutions by stc's system integration and managed operations scope., 260939005301, ended, منتهي, solutions by stc, Managed Platform Operations & Support, تقنية المعلومات, خدمات دعم أنظمة وموارد وأدوات تشغيل منصة بصير, منافسة عامة), List(هيئة التأمين, 0.62, High, https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=RX1gm0Y2*@@**eWL2 AT1vVvAQ==, Systems Integration, Moderate, 2026-09-17, Tender seeks a document and records management (ECM/archiving) system for a government regulatory body, which is a system integration/software implementation deliverable rather than a named stc connectivity or cybersecurity product. This maps to solutions by stc as the integrator responsible for deploying and integrating such enterprise so

In [0]:
# This cell checks how many top-level records Spark loaded from the STC JSON file.
# Since the STC data may contain nested tender records, this is an initial loading check rather than the final tender count.

print("Top-level rows:", stc_raw.count())

Top-level rows: 1


In [0]:
# The STC Bronze JSON stores tenders inside a nested data array.
# This cell explodes that array so each tender becomes one row, extracts the relevant tender fields into normal Spark columns.
# preserves the source and ingestion metadata, and then checks the number of extracted tenders.

from pyspark.sql.functions import explode, col

stc_flat = (
    stc_raw
    .select(
        explode("data").alias("tender"),
        col("source"),
        col("ingestion_timestamp")
    )
)

stc_flat = stc_flat.select(
    col("tender.agency_name").alias("agency_name"),
    col("tender.confidence").alias("confidence"),
    col("tender.criticality_level").alias("criticality_level"),
    col("tender.details_url").alias("tender_url"),
    col("tender.portfolio_domain").alias("portfolio_domain"),
    col("tender.portfolio_fit_level").alias("portfolio_fit_level"),
    col("tender.publish_date").alias("publish_date"),
    col("tender.reason").alias("reason"),
    col("tender.reference_number").alias("reference_number"),
    col("tender.status").alias("status"),
    col("tender.status_ar").alias("status_ar"),
    col("tender.stc_portfolio_company").alias("stc_portfolio_company"),
    col("tender.stc_product").alias("stc_product"),
    col("tender.tender_activity_name").alias("tender_activity_name"),
    col("tender.tender_name").alias("tender_name"),
    col("tender.tender_type_name").alias("tender_type_name"),
    col("source"),
    col("ingestion_timestamp")
)

print("Number of STC tenders:", stc_flat.count())

display(stc_flat)

Number of STC tenders: 124


agency_name,confidence,criticality_level,tender_url,portfolio_domain,portfolio_fit_level,publish_date,reason,reference_number,status,status_ar,stc_portfolio_company,stc_product,tender_activity_name,tender_name,tender_type_name,source,ingestion_timestamp
20251231SAUDI IRRIGATION ORGANIZATION (S,0.98,High,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=Z2pJczVTIefeWrCAJ43iNQ==,Systems Integration,Strong,2026-09-19,"The tender explicitly procures Oracle database software licences. Under the approved third-party enterprise software procurement rule, this maps to solutions by stc; no sovereign cloud hosting or stc connectivity service is specified.",2000002893,open,مفتوح,solutions by stc,Oracle Database Software Licensing & Enablement,تقنية المعلومات,توريد رخص قواعد بيانات أوراكل,منافسة عامة,STC,2026-09-23T15:04:18
صندوق التنمية العقارية,0.85,High,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=5BQuVrbCXegdQ91MJ7qMSA==,Cybersecurity,Strong,2026-09-17,"Tender is for renewal of licenses and support for cybersecurity event management and monitoring (SIEM-type capability), directly aligning with sirar by stc's SOC/SIEM and monitoring offerings.",260839003675,open,مفتوح,sirar by stc,SIEM / Security Event Monitoring & Management,تقنية المعلومات,تجديد رخص ودعم إدارة احداث الامن السيبراني ومراقبتها,منافسة عامة,STC,2026-09-23T15:04:18
جامعة أم القرى,0.6,High,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=8 HFpJXmpHBqF*@@**Ss7b16DQ==,Systems Integration,Moderate,2026-09-17,"Tender scope is physical transfer and relocation of main and secondary data center components to a new data center for Umm Al-Qura University. This is a systems integration/migration engagement rather than a sovereign cloud platform deployment or STC-owned colocation facility, so it maps to solutions by stc as the integrator handling end-to-end DC relocation.",260939001960,open,مفتوح,solutions by stc,Data Center Migration & Relocation Services,تقنية المعلومات,نقل وترحيل مكونات مركز البيانات الرئيسي والفرعي لمركز البيانات الجديد,منافسة عامة,STC,2026-09-23T15:04:18
المديرية العامة للأحوال المدنية,0.55,High,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=xgvkn8OxvqA8UA0e50iiBw==,Connectivity,Moderate,2026-09-17,"Tender requests connecting a government building (Civil Registry) to the Ministry of Interior's private fiber optic network (MiFON) to ensure secure identity-service continuity. This is a direct enterprise fiber connectivity link between a government building and an existing dedicated network, not a wholesale dark-fiber lease or carrier infrastructure sharing arrangement, so it maps to stc Business fiber link/FTTB connectivity rather than stc Wholesale Dark Fiber.",260939003967,open,مفتوح,stc Business,Business Fiber Link / FTTB Connectivity,تجارة الأدوات والآلات والأجهزة,ربط مبنى وكالة الأحوال المدنية بشبكة الألياف البصرية الخاصة بوزارة الداخلية (MiFON),منافسة عامة,STC,2026-09-23T15:04:18
الهيئة السعودية للبيانات والذكاء الاصطناعي,0.6,High,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=hmbY6frQ7h4 5Nad9fnxJw==,Systems Integration,Moderate,2026-09-17,"The tender seeks support services for systems, resources and tools operating SDAIA's 'Baseer' platform, i.e., ongoing operational support and maintenance of an existing government data/AI platform. This falls under managed platform operations and IT support rather than a named STC connectivity or cloud product, aligning with solutions by stc's system integration and managed operations scope.",260939005301,ended,منتهي,solutions by stc,Managed Platform Operations & Support,تقنية المعلومات,خدمات دعم أنظمة وموارد وأدوات تشغيل منصة بصير,منافسة عامة,STC,2026-09-23T15:04:18
هيئة التأمين,0.62,High,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=RX1gm0Y2*@@**eWL2 AT1vVvAQ==,Systems Integration,Moderate,2026-09-17,"Tender seeks a document and records management (ECM/archiving) system for a government regulatory body, which is a system integration/softw

In [0]:
stc_flat.select(
    "publish_date",
    "status",
    "status_ar",
    "tender_type_name",
    "ingestion_timestamp"
).show(10, truncate=False)

+------------+------+---------+----------------+-------------------+
|publish_date|status|status_ar|tender_type_name|ingestion_timestamp|
+------------+------+---------+----------------+-------------------+
|2026-09-19  |open  |مفتوح    |منافسة عامة     |2026-09-23T15:04:18|
|2026-09-17  |open  |مفتوح    |منافسة عامة     |2026-09-23T15:04:18|
|2026-09-17  |open  |مفتوح    |منافسة عامة     |2026-09-23T15:04:18|
|2026-09-17  |open  |مفتوح    |منافسة عامة     |2026-09-23T15:04:18|
|2026-09-17  |ended |منتهي    |منافسة عامة     |2026-09-23T15:04:18|
|2026-09-17  |open  |مفتوح    |منافسة عامة     |2026-09-23T15:04:18|
|2026-09-17  |open  |مفتوح    |منافسة عامة     |2026-09-23T15:04:18|
|2026-09-17  |open  |مفتوح    |منافسة عامة     |2026-09-23T15:04:18|
|2026-09-17  |open  |مفتوح    |منافسة عامة     |2026-09-23T15:04:18|
|2026-09-17  |open  |مفتوح    |منافسة عامة     |2026-09-23T15:04:18|
+------------+------+---------+----------------+-------------------+
only showing top 10 rows


In [0]:
# Data quality checks for STC tenders

from pyspark.sql.functions import sum as spark_sum

important_stc_columns = [
    "reference_number",
    "tender_name",
    "agency_name",
    "tender_type_name",
    "status",
    "publish_date",
    "tender_url",
    "ingestion_timestamp"
]

stc_flat.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in important_stc_columns
]).show()

+----------------+-----------+-----------+----------------+------+------------+----------+-------------------+
|reference_number|tender_name|agency_name|tender_type_name|status|publish_date|tender_url|ingestion_timestamp|
+----------------+-----------+-----------+----------------+------+------------+----------+-------------------+
|               0|          0|          0|               0|     0|           0|         0|                  0|
+----------------+-----------+-----------+----------------+------+------------+----------+-------------------+



In [0]:
print("Total:", stc_flat.count())

print(
    "Unique reference numbers:",
    stc_flat.select("reference_number").distinct().count()
)

print(
    "Null reference numbers:",
    stc_flat.filter(col("reference_number").isNull()).count()
)

Total: 124
Unique reference numbers: 124
Null reference numbers: 0


In [0]:
# STC transformation and standardization cell
# We clean important fields, preserve the original STC status and tender type, standardize values such as status and dates, use the STC activity as a category,
# generate a unique tender key, and assign typed NULL values for information STC doesn't provide. This makes STC compatible with the other tender sources.

from pyspark.sql.functions import upper
from pyspark.sql.functions import (
    col,
    trim,
    to_date,
    to_timestamp,
    lit,
    sha2,
    concat_ws,
    array
)

stc_silver = (
    stc_flat

    # Clean important text fields
    .withColumn("source_tender_id", trim(col("reference_number")))
    .withColumn("reference_number", trim(col("reference_number")))
    .withColumn("tender_name", trim(col("tender_name")))
    .withColumn("agency_name", trim(col("agency_name")))

    # Preserve original STC values
    .withColumn(
        "source_tender_type",
        trim(col("tender_type_name"))
    )
    .withColumn(
        "source_status",
        trim(col("status"))
    )

    # Standardized values
    # STC status is already simple: open -> OPEN
    .withColumn(
        "status",
        trim(col("status")).cast("string")
    )
    .withColumn(
        "status",
        upper(col("status"))
    )

    # For now preserve tender type as-is.
    # We can standardize Arabic tender types across ALL sources later.
    .withColumn(
        "tender_type",
        trim(col("tender_type_name"))
    )

    # Dates
    .withColumn(
        "publish_date",
        to_date(col("publish_date"), "yyyy-MM-dd")
    )
    .withColumn(
        "ingestion_timestamp",
        to_timestamp(col("ingestion_timestamp"))
    )

    # Standard source
    .withColumn("source", lit("stc"))

    # Unique Furas tender key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # Fields STC doesn't provide
    .withColumn("tender_name_en", lit(None).cast("string"))
    .withColumn("description", lit(None).cast("string"))
    .withColumn("closing_date", lit(None).cast("date"))
    .withColumn("opening_date", lit(None).cast("date"))

    # STC activity can become a category
    .withColumn(
        "categories",
        array(trim(col("tender_activity_name")))
    )

    .withColumn("regions", lit(None).cast("array<string>"))

    # Financial fields unavailable
    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn("currency", lit(None).cast("string"))

    # We are not assigning the Furas tech flag yet
    .withColumn("is_tech", lit(None).cast("boolean"))

    # Exact 24-column Silver order
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# print and checking the schema for stc

stc_silver.printSchema()

print("Rows:", stc_silver.count())
print("Columns:", len(stc_silver.columns))

print(
    "Unique tender keys:",
    stc_silver.select("tender_key").distinct().count()
)

root
 |-- tender_key: string (nullable = true)
 |-- source: string (nullable = false)
 |-- source_tender_id: string (nullable = true)
 |-- reference_number: string (nullable = true)
 |-- tender_name: string (nullable = true)
 |-- tender_name_en: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- source_tender_type: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- source_status: string (nullable = true)
 |-- status: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- closing_date: date (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- categories: array (nullable = false)
 |    |-- element: string (containsNull = true)
 |-- regions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- estimated_value_min: decimal(18,2) (nullable = true)
 |-- estimated_value_max: decimal(18,2) (nullable = true)
 |-- document_price: decimal(18,2) (nullable = tr

In [0]:
stc_silver.select(
    "source_tender_id",
    "tender_name",
    "source_tender_type",
    "tender_type",
    "source_status",
    "status",
    "publish_date",
    "categories"
).show(10, truncate=False)

+----------------+-------------------------------------------------------------------------------------------+------------------+-----------+-------------+------+------------+--------------------------------+
|source_tender_id|tender_name                                                                                |source_tender_type|tender_type|source_status|status|publish_date|categories                      |
+----------------+-------------------------------------------------------------------------------------------+------------------+-----------+-------------+------+------------+--------------------------------+
|2000002893      |توريد رخص قواعد بيانات أوراكل                                                              |منافسة عامة       |منافسة عامة|open         |OPEN  |2026-09-19  |[تقنية المعلومات]               |
|260839003675    |تجديد رخص ودعم إدارة احداث الامن السيبراني ومراقبتها                                       |منافسة عامة       |منافسة عامة|open         |OPEN  |20

In [0]:
# This cell loads the standardized STC dataset into its Silver Delta table. We use the same reusable merge function.
# where existing tenders are updated using tender_key and new tenders are inserted. This allows incremental processing instead of overwriting the entire source table.

silver_stc_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/stc"
)

merge_to_silver(
    stc_silver,
    silver_stc_path,
    "STC"
)

STC: Silver MERGE completed.


In [0]:
stc_silver_check = (
    spark.read
    .format("delta")
    .load(silver_stc_path)
)

print("Silver rows:", stc_silver_check.count())
print("Silver columns:", len(stc_silver_check.columns))
print(
    "Unique tender keys:",
    stc_silver_check.select("tender_key").distinct().count()
)

display(stc_silver_check)

Silver rows: 2506
Silver columns: 24
Unique tender keys: 2506


tender_key,source,source_tender_id,reference_number,tender_name,tender_name_en,description,agency_name,source_tender_type,tender_type,source_status,status,publish_date,closing_date,opening_date,categories,regions,estimated_value_min,estimated_value_max,document_price,currency,tender_url,is_tech,ingestion_timestamp
a527dfe4eb324ac6842e6befa7fed6e1057f8aabb8c1a8671e10c83185d65e71,stc,2000002790,2000002790,رخص الدعم الفني لـ50 مستخدم في نظام PAM,null,null,20251231SAUDI IRRIGATION ORGANIZATION (S,منافسة عامة,منافسة عامة,open,OPEN,2026-09-05,null,null,List(الرعاية الصحية والنقاهة),null,null,null,null,null,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=cMGYLYBKBSaN ya j*@@**DPFw==,null,2026-09-14T23:43:53.000Z
34baf81c58d51bdbd498a739de7605fc63357d5d762f2db9c9a01698a745e190,stc,260739005191,260739005191,تفعيل رخص خدمة الثقة الرقمية,null,null,جامعة حفر الباطن,شراء مباشر,شراء مباشر,open,OPEN,2026-09-03,null,null,List(تقنية المعلومات),null,null,null,null,null,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=Va4x9f0rJYnMC9*@@**RlqSsrQ==,null,2026-09-14T23:43:53.000Z
b552f6133febec665226483ca66fc210a18d8c08b4c23f0ea920f1c7f501554a,stc,260739007465,260739007465,صيانة وتشغيل البوابة الإلكترونية ونظام الدخول الموحد,null,null,جامعة الحدود الشمالية,منافسة عامة,منافسة عامة,open,OPEN,2026-09-03,null,null,List(تقنية المعلومات),null,null,null,null,null,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=PfjZkGvFxBoRUKpL8iN06g==,null,2026-09-14T23:43:53.000Z
9c8defea4f8da39d6b39758ff191b2cef614008c1ca43a889a277fcdad8b2a14,stc,260839002990,260839002990,مشروع متابعة الأسر الكافلة والأيتام المسندين لهم - 2026م,null,null,وزارة الموارد البشرية والتنمية الاجتماعية,منافسة عامة,منافسة عامة,open,OPEN,2026-09-03,null,null,List(تقنية المعلومات),null,null,null,null,null,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=FzU*@@**s*@@**8j2OkcL9n3YtZIhg==,null,2026-09-14T23:43:53.000Z
71616ab101e8a433c1e1307116e31f0fa85c32816f955026863b6711c05542f5,stc,260839008770,260839008770,تجديد الخدمات المدارة المستضافة في البيئة الافتراضية,null,null,وزارة الاقتصاد والتخطيط - الديوان العام,منافسة عامة,منافسة عامة,open,OPEN,2026-09-03,null,null,List(تقنية المعلومات),null,null,null,null,null,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=7q S1B2i9Rt2OCetDrsTxQ==,null,2026-09-14T23:43:53.000Z
0796eafcbae72236036c83fd1628366aa753f8d2ab54e6bc2954ac2954e1611e,stc,260839009203,260839009203,نظام إدارة الفحوصات المركزي للثغرات الأمنية,null,null,مجمع الملك سلمان العالمي للغة العربية,منافسة عامة,منافسة عامة,open,OPEN,2026-09-03,null,null,List(تقنية المعلومات),null,null,null,null,null,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=Qpvqv1V G9qM OAImMLhVw==,null,2026-09-14T23:43:53.000Z
e94d6900f1fb4429c137a05c35645fc85c4b751680c4317d835bf2f9438c7301,stc,260839009644,260839009644,الخدمات المدارة لتشغيل خدمات السوبر كمبيوتر (عزيز) وصيانة بيئة البنية التحتية,null,null,جامعة الملك عبدالعزيز,منافسة عامة,منافسة عامة,open,OPEN,2026-09-03,null,null,List(تقنية المعلومات),null,null,null,null,null,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=AkzSb9nlRvRKUOwtJTQKAw==,null,2026-09-14T23:43:53.000Z
9739fc9e5d671a5b0fddf5b2dc5e673945f3382e06a95a711971a533ab0bcaf1,stc,260839010395,260839010395,الخدمات المدارة لمكتب ادارة البيانات وتطوير منصة بيانات المركز,null,null,المركز الوطني للنخيل و التمور,منافسة عامة,منافسة عامة,open,OPEN,2026-09-03,null,null,List(تقنية المعلومات),null,null,null,null,null,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=yqBZBsK1MW*@@**3ktfya69d0g==,null,2026-09-14T23:43:53.000Z
f32bd907fd41f423d18e91a33d005821e46eeb771737b05642791ce635095fdc,stc,260839010487,260839010487,توريد معدات تدعم نظام الاتصال المرئي الأمن,null,null,مدينة الملك عبدالعزيز للعلوم والتقنية,شراء مباشر,شراء مباشر,open,OPEN,2026-09-03,null,null,List(تجارة الأدوات والآلات والأجهزة),null,null,null,null,null,https://tenders.etimad.sa/Tender/DetailsForVisitor?STenderId=n5MXz42dR7YUg*@@**BQtJCQwQ==,null,2026-09-14T23

# Etimad Bronze → Silver

In [0]:
# checking for etimad data path

etimad_path = bronze_path + "ETIMAD/"

display(dbutils.fs.ls(etimad_path))

path,name,size,modificationTime
abfss://bronze@tenderdatalake11.dfs.core.windows.net/ETIMAD/2026-09-16/,2026-09-16/,0,1789902679000
abfss://bronze@tenderdatalake11.dfs.core.windows.net/ETIMAD/2026-09-23/,2026-09-23/,0,1790190460000
abfss://bronze@tenderdatalake11.dfs.core.windows.net/ETIMAD/2026-09-24/,2026-09-24/,0,1790287241000
abfss://bronze@tenderdatalake11.dfs.core.windows.net/ETIMAD/2026-09-25/,2026-09-25/,0,1790378841000
abfss://bronze@tenderdatalake11.dfs.core.windows.net/ETIMAD/2026-09-27/,2026-09-27/,0,1790496895000
abfss://bronze@tenderdatalake11.dfs.core.windows.net/ETIMAD/2026-09-28/,2026-09-28/,0,1790618716000


In [0]:

# Get all date folders under ETIMAD
etimad_folders = dbutils.fs.ls(etimad_path)

# Keep folders only and choose the latest one
latest_etimad_folder = sorted(
    [f.path for f in etimad_folders if f.isDir()],
    reverse=True
)[0]

print("Latest ETIMAD folder:", latest_etimad_folder)

display(dbutils.fs.ls(latest_etimad_folder))

Latest ETIMAD folder: abfss://bronze@tenderdatalake11.dfs.core.windows.net/ETIMAD/2026-09-28/


path,name,size,modificationTime
abfss://bronze@tenderdatalake11.dfs.core.windows.net/ETIMAD/2026-09-28/raw_18-05-15.json,raw_18-05-15.json,260890,1790618716000


In [0]:
# # Get JSON files from the latest ETIMAD folder
# etimad_files = [
#     f for f in dbutils.fs.ls(latest_etimad_folder)
#     if not f.isDir() and f.name.endswith(".json")
# ]

# # Select the latest JSON file
# latest_etimad_file = max(
#     etimad_files,
#     key=lambda f: f.modificationTime
# )

# etimad_file = latest_etimad_file.path

# print("Latest ETIMAD file:", etimad_file)

# etimad_raw = (
#     spark.read
#     .option("multiline", "true")
#     .json(etimad_file)
# )

# etimad_raw.printSchema()

# print("Top-level rows:", etimad_raw.count())


# new cell here:
# ============================================================
# STEP: Load the latest ETIMAD Bronze file
# Purpose:
# Find the newest JSON file in the latest ETIMAD ingestion
# folder and load it into Spark for transformation.
# ============================================================

# Get JSON files from the latest ETIMAD folder
etimad_files = [
    f for f in dbutils.fs.ls(latest_etimad_folder)
    if not f.isDir() and f.name.endswith(".json")
]

# Stop if no ETIMAD JSON files are available
if not etimad_files:
    raise ValueError(
        f"No JSON files found in ETIMAD folder: {latest_etimad_folder}"
    )

# Select the most recently modified JSON file
latest_etimad_file = max(
    etimad_files,
    key=lambda f: f.modificationTime
)

etimad_file = latest_etimad_file.path

print("Latest ETIMAD file:", etimad_file)

# Read the raw ETIMAD JSON data
etimad_raw = (
    spark.read
    .option("multiline", "true")
    .json(etimad_file)
)

# Inspect the raw schema
etimad_raw.printSchema()

# Check the number of top-level records loaded
print("Top-level rows:", etimad_raw.count())

Latest ETIMAD file: abfss://bronze@tenderdatalake11.dfs.core.windows.net/ETIMAD/2026-09-28/raw_18-05-15.json
root
 |-- agencyCode: string (nullable = true)
 |-- agencyName: string (nullable = true)
 |-- branchId: long (nullable = true)
 |-- branchName: string (nullable = true)
 |-- buyingCost: double (nullable = true)
 |-- condetionalBookletPrice: double (nullable = true)
 |-- createdAt: string (nullable = true)
 |-- currentDate: string (nullable = true)
 |-- currentDateTime: string (nullable = true)
 |-- currentTime: string (nullable = true)
 |-- financialFees: double (nullable = true)
 |-- hasInvitations: boolean (nullable = true)
 |-- insideKSA: boolean (nullable = true)
 |-- invitationCost: double (nullable = true)
 |-- isUGRP: boolean (nullable = true)
 |-- lastEnqueriesDate: string (nullable = true)
 |-- lastEnqueriesDateHijri: string (nullable = true)
 |-- lastOfferPresentationDate: string (nullable = true)
 |-- lastOfferPresentationDateHijri: string (nullable = true)
 |-- multi

In [0]:
display(etimad_raw)

agencyCode,agencyName,branchId,branchName,buyingCost,condetionalBookletPrice,createdAt,currentDate,currentDateTime,currentTime,financialFees,hasInvitations,insideKSA,invitationCost,isUGRP,lastEnqueriesDate,lastEnqueriesDateHijri,lastOfferPresentationDate,lastOfferPresentationDateHijri,multipleSearch,offersOpeningDate,offersOpeningDateHijri,referenceNumber,remainingDays,remainingHours,remainingMins,submitionDate,technicalOrganizationId,tenderActivityId,tenderActivityName,tenderActivityNameList,tenderId,tenderIdString,tenderName,tenderNumber,tenderStatusId,tenderStatusIdString,tenderStatusName,tenderTypeId,tenderTypeName,ugrpRFXResponseURL,ugrpRfxUrl
null,مركز الإقامة المميزة,0,مركز الإقامة المميزة,500.0,1000.0,0001-01-01T00:00:00,2026-09-28T00:00:00+03:00,2026-09-28T20:19:54.9189904+03:00,00:00:00,500.0,false,null,200.0,false,2026-10-08T17:33:24.0598034,1448-04-27,2026-10-29T09:59:00,1448-05-18,null,2026-10-29T10:00:00,1448-05-18,260939007843,30,13,39,2026-09-28T17:33:24.0599197,null,902,تقنية المعلومات,null,1105933,M4GUIIyI7WdyXr1jNpfpvA==,"تطبيق أدلة مؤشر البيانات الوطني ""نضيء"" وحصر ممارسات البيانات الشخصية وتطبيق رخص حماية البيانات الشخصية، وإدارة البيانات المرجعية والرئيسية",230660,4,6qPa1DWumsGG0KjPQhbCXA==,null,1,منافسة عامة,null,null
null,ادارة الشؤون الفنية المركزية بمنطقة الباحة,0,ادارة الشؤون الفنية المركزية بمنطقة الباحة,0.0,0.0,0001-01-01T00:00:00,2026-09-28T00:00:00+03:00,2026-09-28T20:19:54.9190139+03:00,00:00:00,200.0,true,null,200.0,false,2026-09-30T17:25:07.3566901,1448-04-19,2026-10-04T09:59:00,1448-04-23,null,null,,260939009559,5,13,39,2026-09-28T17:25:07.3581185,null,1610,انشطة الاستشارات الهندسية - تصميم,null,1107018,t*@@**nVMj0LCjYq0DX*@@**ksV*@@**Bw==,جدول كميات فحص كشف تسريب مياه لبعض مباني الدفاع المدني بمنطقة الباحة للعام 2027,null,4,6qPa1DWumsGG0KjPQhbCXA==,null,2,شراء مباشر,null,null
null,المديرية العامة لحرس الحدود,0,قيادة حرس الحدود بالمنطقة الشرقية,0.0,0.0,0001-01-01T00:00:00,2026-09-28T00:00:00+03:00,2026-09-28T20:19:54.9190156+03:00,00:00:00,200.0,true,null,200.0,false,2026-10-08T17:03:48.5467305,1448-04-27,2026-10-12T09:59:00,1448-05-01,null,null,,260939009951,13,13,39,2026-09-28T17:03:48.546824,null,202,مقاولات عامة للمباني (الانشاء، الإصلاح، الهدم، الترميم),null,1107296,NwrhGSKalk7F9UZi3zCAkA==,اعمال صيانة لمدرسة اسكان حرس الحدود بالعزيزية بقطاع الخبر بالمنطقة الشرقية,null,4,6qPa1DWumsGG0KjPQhbCXA==,null,2,شراء مباشر,null,null
null,الشؤون الصحیة بوزارة الحرس الوطني,0,الإدارة العامة للعقود - عقود الخدمات,0.0,0.0,0001-01-01T00:00:00,2026-09-28T00:00:00+03:00,2026-09-28T20:19:54.9190173+03:00,00:00:00,200.0,true,null,200.0,false,2026-10-03T16:46:49.0039809,1448-04-22,2026-10-06T09:59:00,1448-04-25,null,null,,260939006583,7,13,39,2026-09-28T16:46:49.0040914,null,902,تقنية المعلومات,null,1105149,jk4 RsoCceC2XjVQI*@@**ptg==,تقديم حل متكامل لتطبيق العلامة المائية الرقمية بهدف حماية البيانات وتعزيز مستوى الأمن السيبراني على كافة الأجهزة المحددة ويكون مركزيًا في مدينة الملك عبدالعزيز الطبية بالرياض و يخدم جميع مناطق المملكة التابعة للشؤون الصحية بوزارة الحرس الوطني,2409752,4,6qPa1DWumsGG0KjPQhbCXA==,null,2,شراء مباشر,null,null
null,وزارة النقل والخدمات اللوجستية - الديوان العام,0,ادارة المنافسات-فرع عسير,0.0,0.0,0001-01-01T00:00:00,2026-09-28T00:00:00+03:00,2026-09-28T20:19:54.919022+03:00,00:00:00,0.0,false,null,200.0,false,2026-09-29T16:43:40.8789804,1448-04-18,2026-10-01T09:59:00,1448-04-20,null,null,,260939009736,2,13,39,2026-09-28T16:43:40.8790726,null,111,تجارة قطع الغيار الجديدة,null,1107152,fOJEwSK08n9bOkwVGNskaA==,تامين وتوريد قطع غيار سيارات للفرع مع التركيب,null,4,6qPa1DWumsGG0KjPQhbCXA==,null,2,شراء مباشر,null,null
null,هيئة الإذاعة والتلفزيون,0,الادارة المالية- المشتريات,0.0,0.0,0001-01-01T00:00:00,2026-09-28T00:00:00+03:00,2026-09-28T20:19:54.9190241+03:00,00:00:00,0.0,false,null,200.0,false,2026-09-30T16:23:19.7012545,1448-04-19,2026-10-02T09:59:00,1448-04-21,null,null,,260939009913,3,13,39,2026-09-28T16:23:19.7013236,null,108,تجارة الأدوات والآلات والأجهزة,null,1107277,9PuOVUBCkVaRIV

In [0]:
# checking the real etimad colmns

print("Number of Etimad columns:", len(etimad_raw.columns))

for c in etimad_raw.columns:
    print(c)

Number of Etimad columns: 42
agencyCode
agencyName
branchId
branchName
buyingCost
condetionalBookletPrice
createdAt
currentDate
currentDateTime
currentTime
financialFees
hasInvitations
insideKSA
invitationCost
isUGRP
lastEnqueriesDate
lastEnqueriesDateHijri
lastOfferPresentationDate
lastOfferPresentationDateHijri
multipleSearch
offersOpeningDate
offersOpeningDateHijri
referenceNumber
remainingDays
remainingHours
remainingMins
submitionDate
technicalOrganizationId
tenderActivityId
tenderActivityName
tenderActivityNameList
tenderId
tenderIdString
tenderName
tenderNumber
tenderStatusId
tenderStatusIdString
tenderStatusName
tenderTypeId
tenderTypeName
ugrpRFXResponseURL
ugrpRfxUrl


In [0]:
print("Number of Etimad tenders:", etimad_raw.count())

Number of Etimad tenders: 138


In [0]:
# This cell inspects the important raw ETIMAD fields before transformation.
# It helps us verify identifiers, tender information, dates, activity, price, and URL values so that we can map them correctly into the common Silver schema.

etimad_raw.select(
    "tenderId",
    "tenderIdString",
    "referenceNumber",
    "tenderNumber",
    "tenderName",
    "agencyName",
    "tenderTypeName",
    "tenderStatusName",
    "submitionDate",
    "lastOfferPresentationDate",
    "offersOpeningDate",
    "tenderActivityName",
    "condetionalBookletPrice",
    "ugrpRfxUrl"
).show(10, truncate=False)

+--------+------------------------------------+---------------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+--------------+----------------+---------------------------+-------------------------+-------------------+----------------------------------------------------------------+-----------------------+----------+
|tenderId|tenderIdString                      |referenceNumber|tenderNumber|tenderName                                                                                                                                                                                                                                        |agencyName                                    |tenderTypeName|tenderStatusName|submitionDate              |l

In [0]:
# ETIMAD provides multiple identifier fields, so this cell compares their uniqueness and checks for missing values. This helps us choose a reliable source 
# tender ID for generating the Furas tender key while keeping the tender reference number separately.

from pyspark.sql.functions import col

print("Total Etimad tenders:", etimad_raw.count())

print(
    "Unique tenderId:",
    etimad_raw.select("tenderId").distinct().count()
)

print(
    "Unique tenderIdString:",
    etimad_raw.select("tenderIdString").distinct().count()
)

print(
    "Unique referenceNumber:",
    etimad_raw.select("referenceNumber").distinct().count()
)

print(
    "Null tenderId:",
    etimad_raw.filter(col("tenderId").isNull()).count()
)

print(
    "Null referenceNumber:",
    etimad_raw.filter(col("referenceNumber").isNull()).count()
)

Total Etimad tenders: 138
Unique tenderId: 138
Unique tenderIdString: 138
Unique referenceNumber: 138
Null tenderId: 0
Null referenceNumber: 0


In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    to_date,
    to_timestamp,
    lit,
    sha2,
    concat_ws,
    array,
    upper
)

etimad_silver = (
    etimad_raw

    # IDs
    .withColumn(
        "source_tender_id",
        col("tenderId").cast("string")
    )
    .withColumn(
        "reference_number",
        trim(col("referenceNumber"))
    )

    # Main tender information
    .withColumn(
        "tender_name",
        trim(col("tenderName"))
    )
    .withColumn(
        "agency_name",
        trim(col("agencyName"))
    )

    # Preserve original Etimad values
    .withColumn(
        "source_tender_type",
        trim(col("tenderTypeName"))
    )
    .withColumn(
        "source_status",
        trim(col("tenderStatusName"))
    )

    # For now, preserve type/status as provided by Etimad.
    # Cross-source standardization comes later.
    .withColumn(
        "tender_type",
        trim(col("tenderTypeName"))
    )
    .withColumn(
        "status",
        trim(col("tenderStatusName"))
    )

    # Dates
    .withColumn(
        "publish_date",
        to_date(to_timestamp(col("submitionDate")))
    )
    .withColumn(
        "closing_date",
        to_date(to_timestamp(col("lastOfferPresentationDate")))
    )
    .withColumn(
        "opening_date",
        to_date(to_timestamp(col("offersOpeningDate")))
    )

    # Category
    .withColumn(
        "categories",
        array(trim(col("tenderActivityName")))
    )

    # Price
    .withColumn(
        "document_price",
        col("condetionalBookletPrice").cast("decimal(18,2)")
    )

    # Source
    .withColumn("source", lit("etimad"))

    # Deterministic Furas key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # Fields not available / not mapped from this Bronze dataset
    .withColumn("tender_name_en", lit(None).cast("string"))
    .withColumn("description", lit(None).cast("string"))
    .withColumn("regions", lit(None).cast("array<string>"))
    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn("currency", lit("SAR"))
    .withColumn("is_tech", lit(None).cast("boolean"))

    # URL is not available in the sample field we inspected
    .withColumn("tender_url", lit(None).cast("string"))

    # Bronze file doesn't show a separate ingestion timestamp,
    # so keep it NULL rather than inventing one.
    .withColumn(
        "ingestion_timestamp",
        lit(None).cast("timestamp")
    )

    # Exact common Silver schema
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# new cell here, why? becuase etimad may have a url:

# ============================================================
# STEP: Transform ETIMAD data to the common Silver schema
# Purpose:
# Clean and standardize ETIMAD tender data and map it to
# the 24-column Furas Silver structure.
# ============================================================

# from pyspark.sql.functions import (
#     col,
#     trim,
#     to_date,
#     to_timestamp,
#     lit,
#     sha2,
#     concat_ws,
#     array
# )

# etimad_silver = (
#     etimad_raw

#     # IDs
#     .withColumn(
#         "source_tender_id",
#         col("tenderId").cast("string")
#     )
#     .withColumn(
#         "reference_number",
#         trim(col("referenceNumber"))
#     )

#     # Main tender information
#     .withColumn(
#         "tender_name",
#         trim(col("tenderName"))
#     )
#     .withColumn(
#         "agency_name",
#         trim(col("agencyName"))
#     )

#     # Preserve original ETIMAD values
#     .withColumn(
#         "source_tender_type",
#         trim(col("tenderTypeName"))
#     )
#     .withColumn(
#         "source_status",
#         trim(col("tenderStatusName"))
#     )

#     # Preserve cleaned type/status for later cross-source standardization
#     .withColumn(
#         "tender_type",
#         trim(col("tenderTypeName"))
#     )
#     .withColumn(
#         "status",
#         trim(col("tenderStatusName"))
#     )

#     # Dates
#     .withColumn(
#         "publish_date",
#         to_date(to_timestamp(col("submitionDate")))
#     )
#     .withColumn(
#         "closing_date",
#         to_date(to_timestamp(col("lastOfferPresentationDate")))
#     )
#     .withColumn(
#         "opening_date",
#         to_date(to_timestamp(col("offersOpeningDate")))
#     )

#     # Category
#     .withColumn(
#         "categories",
#         array(trim(col("tenderActivityName")))
#     )

#     # Financial information
#     .withColumn(
#         "document_price",
#         col("condetionalBookletPrice").cast("decimal(18,2)")
#     )
#     .withColumn(
#         "estimated_value_min",
#         lit(None).cast("decimal(18,2)")
#     )
#     .withColumn(
#         "estimated_value_max",
#         lit(None).cast("decimal(18,2)")
#     )
#     .withColumn("currency", lit("SAR"))

#     # Standard source
#     .withColumn("source", lit("etimad"))

#     # Generate deterministic Furas tender key
#     .withColumn(
#         "tender_key",
#         sha2(
#             concat_ws("|", col("source"), col("source_tender_id")),
#             256
#         )
#     )

#     # Fields not available from this ETIMAD dataset
#     .withColumn("tender_name_en", lit(None).cast("string"))
#     .withColumn("description", lit(None).cast("string"))
#     .withColumn("regions", lit(None).cast("array<string>"))
#     .withColumn("is_tech", lit(None).cast("boolean"))

#     # Map ETIMAD tender URL
#     .withColumn(
#         "tender_url",
#         trim(col("ugrpRfxUrl"))
#     )

#     # No ingestion timestamp available in this Bronze dataset
#     .withColumn(
#         "ingestion_timestamp",
#         lit(None).cast("timestamp")
#     )

#     # Exact common Silver schema
#     .select(
#         "tender_key",
#         "source",
#         "source_tender_id",
#         "reference_number",
#         "tender_name",
#         "tender_name_en",
#         "description",
#         "agency_name",
#         "source_tender_type",
#         "tender_type",
#         "source_status",
#         "status",
#         "publish_date",
#         "closing_date",
#         "opening_date",
#         "categories",
#         "regions",
#         "estimated_value_min",
#         "estimated_value_max",
#         "document_price",
#         "currency",
#         "tender_url",
#         "is_tech",
#         "ingestion_timestamp"
#     )
# )

In [0]:
etimad_silver.printSchema()

print("Rows:", etimad_silver.count())
print("Columns:", len(etimad_silver.columns))

print(
    "Unique tender keys:",
    etimad_silver.select("tender_key").distinct().count()
)

root
 |-- tender_key: string (nullable = true)
 |-- source: string (nullable = false)
 |-- source_tender_id: string (nullable = true)
 |-- reference_number: string (nullable = true)
 |-- tender_name: string (nullable = true)
 |-- tender_name_en: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- source_tender_type: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- source_status: string (nullable = true)
 |-- status: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- closing_date: date (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- categories: array (nullable = false)
 |    |-- element: string (containsNull = true)
 |-- regions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- estimated_value_min: decimal(18,2) (nullable = true)
 |-- estimated_value_max: decimal(18,2) (nullable = true)
 |-- document_price: decimal(18,2) (nullable = tr

In [0]:
etimad_silver.select(
    "source_tender_id",
    "reference_number",
    "tender_name",
    "agency_name",
    "source_tender_type",
    "source_status",
    "publish_date",
    "closing_date",
    "opening_date",
    "categories",
    "document_price",
    "currency"
).show(10, truncate=False)

# # if we have a url then:
# etimad_silver.select(
#     "source_tender_id",
#     "reference_number",
#     "tender_name",
#     "agency_name",
#     "source_tender_type",
#     "source_status",
#     "publish_date",
#     "closing_date",
#     "opening_date",
#     "categories",
#     "document_price",
#     "currency",
#     "tender_url"
# ).show(10, truncate=False)

+----------------+----------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+------------------+-------------+------------+------------+------------+------------------------------------------------------------------+--------------+--------+
|source_tender_id|reference_number|tender_name                                                                                                                                                                                                                                       |agency_name                                   |source_tender_type|source_status|publish_date|closing_date|opening_date|categories                                                        |document_price|currency|
+----------------+----

In [0]:
# Checking Data Quality for dates:

print(
    "Null publish dates:",
    etimad_silver.filter(col("publish_date").isNull()).count()
)

print(
    "Null closing dates:",
    etimad_silver.filter(col("closing_date").isNull()).count()
)

print(
    "Null opening dates:",
    etimad_silver.filter(col("opening_date").isNull()).count()
)

Null publish dates: 0
Null closing dates: 2
Null opening dates: 88


In [0]:
# This cell loads the standardized ETIMAD data into its source-specific Silver Delta table.

silver_etimad_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/etimad"
)

merge_to_silver(
    etimad_silver,
    silver_etimad_path,
    "Etimad"
)

Etimad: Silver MERGE completed.


In [0]:
# This is the post-write validation for ETIMAD. It confirms that the Delta MERGE actually produced a readable Silver table with the expected structure and unique keys.

etimad_silver_check = (
    spark.read
    .format("delta")
    .load(silver_etimad_path)
)

print("Silver rows:", etimad_silver_check.count())
print("Silver columns:", len(etimad_silver_check.columns))
print(
    "Unique tender keys:",
    etimad_silver_check.select("tender_key").distinct().count()
)

Silver rows: 8257
Silver columns: 24
Unique tender keys: 8257


# Source NO.3 : CST Bronze → Silver

In [0]:
# Checking the cst folder in the bronze

cst_path = bronze_path + "cst/"

display(dbutils.fs.ls(cst_path))

path,name,size,modificationTime
abfss://bronze@tenderdatalake11.dfs.core.windows.net/cst/2026-09-16/,2026-09-16/,0,1789902679000


In [0]:

# Get all date folders under CST
cst_folders = dbutils.fs.ls(cst_path)

# Keep folders only and choose the latest one
latest_cst_folder = sorted(
    [f.path for f in cst_folders if f.isDir()],
    reverse=True
)[0]

print("Latest CST folder:", latest_cst_folder)

display(dbutils.fs.ls(latest_cst_folder))

Latest CST folder: abfss://bronze@tenderdatalake11.dfs.core.windows.net/cst/2026-09-16/


path,name,size,modificationTime
abfss://bronze@tenderdatalake11.dfs.core.windows.net/cst/2026-09-16/cst_2026-09-16csv,cst_2026-09-16csv,6599,1789902679000


In [0]:

# Get CSV files from the latest CST folder
# ============================================================
# STEP: Load the latest CST Bronze file
# Purpose:
# Find the newest CSV file in the latest CST ingestion folder
# and load it into Spark for transformation.
# ============================================================

cst_files = [
    f for f in dbutils.fs.ls(latest_cst_folder)
    if not f.isDir() and (
        f.name.endswith(".csv") or
        f.name.endswith("csv")
    )
]

print("CST files found:", [f.name for f in cst_files])

latest_cst_file = max(
    cst_files,
    key=lambda f: f.modificationTime
)

cst_file = latest_cst_file.path

print("Latest CST file:", cst_file)

cst_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiline", "true")
    .option("escape", '"')
    .csv(cst_file)
)

cst_raw.printSchema()

print("Number of CST records:", cst_raw.count())



CST files found: ['cst_2026-09-16csv']
Latest CST file: abfss://bronze@tenderdatalake11.dfs.core.windows.net/cst/2026-09-16/cst_2026-09-16csv
root
 |-- tender_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- deadline: string (nullable = true)
 |-- note: string (nullable = true)
 |-- type: string (nullable = true)
 |-- source: string (nullable = true)
 |-- extracted_at: timestamp (nullable = true)

Number of CST records: 33


In [0]:
# Print the original CST columns

print("Number of CST columns:", len(cst_raw.columns))

for c in cst_raw.columns:
    print(c)

Number of CST columns: 7
tender_id
title
deadline
note
type
source
extracted_at


In [0]:
display(cst_raw.limit(10))

tender_id,title,deadline,note,type,source,extracted_at
NEG-002083,Operational Cybersecurity Licenses,18/06/2026 11:00 PM,"To view Tender details and submit bids, please access the Vendor Portal",current,CST,2026-09-16T08:46:38.000Z
NEG-002082,Subscription to BIC Service – Procedures Management System – Completion of Hosting Tasks and Other Works,18/06/2026 11:00 PM,"To view Tender details and submit bids, please access the Vendor Portal",current,CST,2026-09-16T08:46:38.000Z
NEG-002080,Renewal of IBM Parent Company Licenses,18/06/2026 11:00 PM,"To view Tender details and submit bids, please access the Vendor Portal",current,CST,2026-09-16T08:46:38.000Z
NEG-002085,Media Awareness Initiatives Competition 2026,22/06/2026 11:00 PM,"To view Tender details and submit bids, please access the Vendor Portal",current,CST,2026-09-16T08:46:38.000Z
NEG-002086,Global Event for Telecommunications and Space Sustainability 2026 – NTN Scientific Content,23/06/2026 1:53 PM,"To view Tender details and submit bids, please access the Vendor Portal",current,CST,2026-09-16T08:46:38.000Z
NEG-002095,Provision of Interior and Exterior Glass Cleaning Services for the Authority's Internal and External Facades,24/06/2026 2:35 PM,"To view Tender details and submit bids, please access the Vendor Portal",current,CST,2026-09-16T08:46:38.000Z
NEG-002101,Insurance Against Emergency Damage to Non-Qualified Stations,24/06/2026 3:18 PM,"To view Tender details and submit bids, please access the Vendor Portal",current,CST,2026-09-16T08:46:38.000Z
NEG-002096,Subscription to IPinfo Platform – 2026,25/06/2026 11:00 PM,"To view Tender details and submit bids, please access the Vendor Portal",current,CST,2026-09-16T08:46:38.000Z
NEG-002093,Qualys Internet Scanning Program Licenses for the Authority's Systems,25/06/2026 11:00 PM,"To view Tender details and submit bids, please access the Vendor Portal",current,CST,2026-09-16T08:46:38.000Z
NEG-002092,Contract with King Saud University for the Provision of Consultancy Services,25/06/2026 11:00 PM,"To view Tender details and submit bids, please access the Vendor Portal",current,CST,2026-09-16T08:46:38.000Z


In [0]:
cst_raw.select(
    "tender_id",
    "title",
    "deadline",
    "note",
    "type",
    "source",
    "extracted_at"
).show(15, truncate=False)

+----------+--------------------------------------------------------------------------------------------------------------------+-------------------+-----------------------------------------------------------------------+-------+------+-------------------+
|tender_id |title                                                                                                               |deadline           |note                                                                   |type   |source|extracted_at       |
+----------+--------------------------------------------------------------------------------------------------------------------+-------------------+-----------------------------------------------------------------------+-------+------+-------------------+
|NEG-002083|Operational Cybersecurity Licenses                                                                                  |18/06/2026 11:00 PM|To view Tender details and submit bids, please access the Vendor Portal|current|

In [0]:
# This is a data-quality check before transforming CST into the common Silver schema

from pyspark.sql.functions import col, sum as spark_sum

print("Total CST tenders:", cst_raw.count())

print(
    "Unique tender IDs:",
    cst_raw.select("tender_id").distinct().count()
)

print(
    "Null tender IDs:",
    cst_raw.filter(col("tender_id").isNull()).count()
)

cst_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "tender_id",
        "title",
        "deadline",
        "note",
        "type",
        "source",
        "extracted_at"
    ]
]).show()

Total CST tenders: 33
Unique tender IDs: 33
Null tender IDs: 0
+---------+-----+--------+----+----+------+------------+
|tender_id|title|deadline|note|type|source|extracted_at|
+---------+-----+--------+----+----+------+------------+
|        0|    0|       0|   0|   0|     0|           0|
+---------+-----+--------+----+----+------+------------+



In [0]:
# This cell transforms the raw CST data into our common 24-column Silver schema.
# We map CST's ID, title, note and status to standardized fields, handle multiple deadline formats, preserve the extraction timestamp, generate a unique tender 
# key, and add typed NULL values for fields CST doesn't provide.

from pyspark.sql.functions import expr

from pyspark.sql.functions import (
    col,
    trim,
    to_timestamp,
    to_date,
    lit,
    sha2,
    concat_ws,
    upper
)

cst_silver = (
    cst_raw

    # IDs
    .withColumn(
        "source_tender_id",
        trim(col("tender_id"))
    )
    .withColumn(
        "reference_number",
        trim(col("tender_id"))
    )

    # Main tender information
    .withColumn(
        "tender_name",
        trim(col("title"))
    )
    .withColumn(
        "description",
        trim(col("note"))
    )

    # CST 'type' describes current/archive status
    .withColumn(
        "source_status",
        trim(col("type"))
    )
    .withColumn(
        "status",
        upper(trim(col("type")))
    )

    # Parse deadline
  .withColumn(
    "closing_date",
    to_date(
        expr("""
            coalesce(
                try_to_timestamp(trim(deadline), 'dd/MM/yyyy h:mm a'),
                try_to_timestamp(trim(deadline), 'dd/MM/yyyy')
            )
        """)
    )
)

    # Source
    .withColumn(
        "source",
        lit("cst")
    )

    # Unique Furas key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # Existing extraction time becomes ingestion timestamp
    .withColumn(
        "ingestion_timestamp",
        col("extracted_at").cast("timestamp")
    )

    # Fields CST doesn't provide
    .withColumn("tender_name_en", lit(None).cast("string"))
    .withColumn("agency_name", lit("CST").cast("string"))

    .withColumn("source_tender_type", lit(None).cast("string"))
    .withColumn("tender_type", lit(None).cast("string"))

    .withColumn("publish_date", lit(None).cast("date"))
    .withColumn("opening_date", lit(None).cast("date"))

    .withColumn("categories", lit(None).cast("array<string>"))
    .withColumn("regions", lit(None).cast("array<string>"))

    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )

    .withColumn("currency", lit(None).cast("string"))
    .withColumn("tender_url", lit(None).cast("string"))
    .withColumn("is_tech", lit(None).cast("boolean"))

    # Exact common Silver order
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# checking the schema

cst_silver.printSchema()

print("Rows:", cst_silver.count())
print("Columns:", len(cst_silver.columns))

print(
    "Unique tender keys:",
    cst_silver.select("tender_key").distinct().count()
)

print(
    "Null closing dates:",
    cst_silver.filter(col("closing_date").isNull()).count()
)

root
 |-- tender_key: string (nullable = true)
 |-- source: string (nullable = false)
 |-- source_tender_id: string (nullable = true)
 |-- reference_number: string (nullable = true)
 |-- tender_name: string (nullable = true)
 |-- tender_name_en: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agency_name: string (nullable = false)
 |-- source_tender_type: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- source_status: string (nullable = true)
 |-- status: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- closing_date: date (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- regions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- estimated_value_min: decimal(18,2) (nullable = true)
 |-- estimated_value_max: decimal(18,2) (nullable = true)
 |-- document_price: decimal(18,2) (nullable = tr

In [0]:
# # This cell loads the standardized CST dataset into its Silver Delta table. The reusable merge function uses tender_key to update existing tenders and insert 
# new ones

silver_cst_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/cst"
)

merge_to_silver(
    cst_silver,
    silver_cst_path,
    "CST"
)

CST: Silver MERGE completed.


In [0]:
# After merging CST into Silver, this cell reads the stored Delta table back from ADLS and verifies the number of records, confirms that it has the expected 
# 24-column schema, and checks that the tender keys remain unique.

cst_silver_check = (
    spark.read
    .format("delta")
    .load(silver_cst_path)
)

print("Silver rows:", cst_silver_check.count())
print("Silver columns:", len(cst_silver_check.columns))
print(
    "Unique tender keys:",
    cst_silver_check.select("tender_key").distinct().count()
)

Silver rows: 33
Silver columns: 24
Unique tender keys: 33


## Soruce NO.4: Tanafos Bronze -> Silver

In [0]:
# ============================================================
# STEP: Load the latest Tanafos Bronze file

tanafos_path = bronze_path + "Tanafos/"

# Get all date folders under Tanafos
tanafos_folders = dbutils.fs.ls(tanafos_path)

# Choose latest date folder
latest_tanafos_folder = sorted(
    [f.path for f in tanafos_folders if f.isDir()],
    reverse=True
)[0]

print("Latest Tanafos folder:", latest_tanafos_folder)

# Get JSON files inside latest folder
tanafos_files = [
    f.path
    for f in dbutils.fs.ls(latest_tanafos_folder)
    if f.path.lower().endswith(".json")
]

# Choose latest JSON file
tanafos_file = sorted(tanafos_files, reverse=True)[0]

print("Latest Tanafos file:", tanafos_file)

# Read Bronze exactly as produced by the old ingestion method
tanafos_raw = (
    spark.read
    .option("multiline", "true")
    .json(tanafos_file)
)

tanafos_raw.printSchema()

print("Top-level rows:", tanafos_raw.count())


Latest Tanafos folder: abfss://bronze@tenderdatalake11.dfs.core.windows.net/Tanafos/2026-09-16/
Latest Tanafos file: abfss://bronze@tenderdatalake11.dfs.core.windows.net/Tanafos/2026-09-16/tanafos_bids_tech_only.json
root
 |-- bid_status_id: long (nullable = true)
 |-- bid_type_name: string (nullable = true)
 |-- classifications: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- creation_date: string (nullable = true)
 |-- entity_name: string (nullable = true)
 |-- id: long (nullable = true)
 |-- is_open: boolean (nullable = true)
 |-- is_tech: boolean (nullable = true)
 |-- matched_classifications: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- price: double (nullable = true)
 |-- ref_number: string (nullable = true)
 |-- regions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- submission_deadline: string (nullable = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)

Top-l

In [0]:
# checking tanafos columns:

print("Number of columns:", len(tanafos_raw.columns))

for c in tanafos_raw.columns:
    print(c)


Number of columns: 15
bid_status_id
bid_type_name
classifications
creation_date
entity_name
id
is_open
is_tech
matched_classifications
price
ref_number
regions
submission_deadline
title
url


In [0]:

display(tanafos_raw.limit(10))

bid_status_id,bid_type_name,classifications,creation_date,entity_name,id,is_open,is_tech,matched_classifications,price,ref_number,regions,submission_deadline,title,url
11,عامة,"List(الأمن والسلامة / أنشطة الأمن و السلامة, الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",2026-08-06T13:24:04.9314614,اللجنة الوطنية لرعاية السجناء والمفرج عنهم وأسرهم بمنطقة الرياض,20794,false,true,"List(الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",230.0,2608149357,List(الرياض),2026-08-16T23:59:59,توريد وتركيب وتشغيل نظام كاميرات مراقبة لمبنى لجنة رعاية السجناء بمنطقة الرياض,https://tanafos.sa/bids/view-bid-details/20794
11,عامة,"List(الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",2026-07-13T13:42:47.73072,مؤسسة الملك خالد الخيرية,20773,false,true,"List(الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",249.9985,2607149566,List(الرياض),2026-07-30T23:59:59,تطوير نظام إلكتروني ثنائي اللغة لإدارة وتسجيل دخول الزوار,https://tanafos.sa/bids/view-bid-details/20773
10,فورية,"List(الإعلام والنشر والتوزيع / خدمات النشر والطباعة والدعاية والاعلان والإنتاج الفني, الاتصالات وتقنية المعلومات / تقنية المعلومات)",2026-05-31T12:33:16.7555251,جمعية زهرة لسرطان الثدي,20737,true,true,List(الاتصالات وتقنية المعلومات / تقنية المعلومات),1150.0,2605153168,List(جميع مناطق المملكة),null,التسويق الإلكتروني للحملة الوطنية للتوعية بسرطان الثدي – أكتوبر 2026,https://tanafos.sa/bids/view-bid-details/20737
10,محدودة,"List(التجارة / تجارة الأدوات والآلات والأجهزة, الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",2026-03-28T13:45:05.1203938,جمعية العناية بمساجد الطرق,20711,false,true,"List(الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",1437.5,2603137964,List(الرياض),2026-04-06T23:59:59,توريد وتركيب المستلزمات التقنية للمقر الرئيسي لجمعية العناية بمساجد الطرق,https://tanafos.sa/bids/view-bid-details/20711
10,فورية,"List(الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات, التجارة / تجارة الأدوات والآلات والأجهزة)",2026-02-11T13:43:15.8606874,الجمعية الخيرية لرعاية الأيتام بمنطقة حائل (رفاق),20688,true,true,"List(الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",230.0,2602156238,List(حائل),null,توريد جهاز تبرع ذاتي,https://tanafos.sa/bids/view-bid-details/20688
10,عامة,"List(الإعلام والنشر والتوزيع / خدمات النشر والطباعة والدعاية والاعلان والإنتاج الفني, الاتصالات وتقنية المعلومات / تقنية المعلومات, الخدمات الإدارية وخدمات الدعم / تنظيم وإدارة المعارض والمؤتمرات)",2026-01-14T13:44:01.3703661,جمعية تنمية الأيتام بمنطقة المدينة المنورة,20653,false,true,List(الاتصالات وتقنية المعلومات / تقنية المعلومات),749.9955,2601148826,List(المدينة المنورة),2026-01-27T23:59:59,"تنفيذ الحملة التسويقية ""رسالة تلهم وأثر مستدام""",https://tanafos.sa/bids/view-bid-details/20653
10,عامة,"List(الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",2025-12-10T15:35:30.5957431,جمعية حماية المستهلك,20616,false,true,"List(الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",999.994,2512144518,List(الرياض),2025-12-22T23:59:59,تطوير واستضافة وتشغيل البوابة الإلكترونية وتطبيق صوت المستهلك,https://tanafos.sa/bids/view-bid-details/20616
10,فورية,"List(التجارة / تجارة الأدوات والآلات والأجهزة, الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",2025-10-30T11:10:14.2768421,الجمعية الخيرية لرعاية الايتام بمنطقة المدينة المنورة تكافل,20573,true,true,"List(الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",230.0,2510154676,List(المدينة المنورة),null,توريد أجهزة حاسب آلي وملحقاتها,https://tanafos.sa/bids/view-bid-details/20573
10,عامة,"List(التعليم والتدريب / أنشطة التعليم, التعليم والتدريب / أنشطة التدريب, الاتصالات وتقنية المعلومات / تقنية المعلومات)",2025-10-07T12:52:23.7737875,جمعية

In [0]:
# This cell displays a sample of the important raw Tanafos fields so we can understand their values and structure before mapping them to the common Silver schema.

tanafos_raw.select(
    "id",
    "ref_number",
    "title",
    "entity_name",
    "bid_type_name",
    "bid_status_id",
    "is_open",
    "creation_date",
    "submission_deadline",
    "classifications",
    "matched_classifications",
    "regions",
    "price",
    "is_tech",
    "url"
).show(10, truncate=False)

+-----+----------+------------------------------------------------------------------------------+---------------------------------------------------------------+-------------+-------------+-------+---------------------------+-------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------+--------------------+--------+-------+----------------------------------------------+
|id   |ref_number|title                                                                         |entity_name                                                    |bid_type_name|bid_status_id|is_open|creation_date              |submission_deadline|classifications                                                                                                                                 

In [0]:
# These cells inspect the important raw Tanafos fields and perform data-quality checks before transformation.
# We compare the total number of records with unique IDs and reference numbers, check for missing IDs, and count null values in important fields.
# This helps us verify that id is suitable for generating the unique tender key and understand which source fields may be incomplete.


tanafos_raw.select(
    "id",
    "ref_number",
    "title",
    "entity_name",
    "bid_type_name",
    "bid_status_id",
    "is_open",
    "creation_date",
    "submission_deadline",
    "classifications",
    "matched_classifications",
    "regions",
    "price",
    "is_tech",
    "url"
).show(10, truncate=False)



from pyspark.sql.functions import col, sum as spark_sum

print("Total Tanafos tenders:", tanafos_raw.count())

print(
    "Unique IDs:",
    tanafos_raw.select("id").distinct().count()
)

print(
    "Unique reference numbers:",
    tanafos_raw.select("ref_number").distinct().count()
)

print(
    "Null IDs:",
    tanafos_raw.filter(col("id").isNull()).count()
)

tanafos_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "id",
        "ref_number",
        "title",
        "entity_name",
        "bid_type_name",
        "is_open",
        "creation_date",
        "submission_deadline",
        "price",
        "is_tech",
        "url"
    ]
]).show()

+-----+----------+------------------------------------------------------------------------------+---------------------------------------------------------------+-------------+-------------+-------+---------------------------+-------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------+--------------------+--------+-------+----------------------------------------------+
|id   |ref_number|title                                                                         |entity_name                                                    |bid_type_name|bid_status_id|is_open|creation_date              |submission_deadline|classifications                                                                                                                                 

In [0]:
# This cell transforms Tanafos into our common 24-column Silver schema.
# It maps the source ID, reference number, title, agency, tender type, dates, categories, regions, price, technology flag, and URL. It also converts the 
# is_open field into a standardized OPEN or CLOSED status, generates the unique Furas tender key, and adds typed NULL values for fields that Tanafos does not 
# provide.

from pyspark.sql.functions import (
    col,
    trim,
    to_timestamp,
    to_date,
    lit,
    sha2,
    concat_ws,
    when
)

tanafos_silver = (
    tanafos_raw

    # IDs
    .withColumn(
        "source_tender_id",
        col("id").cast("string")
    )
    .withColumn(
        "reference_number",
        trim(col("ref_number"))
    )

    # Main information
    .withColumn(
        "tender_name",
        trim(col("title"))
    )
    .withColumn(
        "agency_name",
        trim(col("entity_name"))
    )

    # Tender type
    .withColumn(
        "source_tender_type",
        trim(col("bid_type_name"))
    )
    .withColumn(
        "tender_type",
        trim(col("bid_type_name"))
    )

    # Preserve source status information
    .withColumn(
        "source_status",
        col("bid_status_id").cast("string")
    )

    # Human-readable standardized status
    .withColumn(
        "status",
        when(col("is_open") == True, lit("OPEN"))
        .otherwise(lit("CLOSED"))
    )

    # Dates
    .withColumn(
        "publish_date",
        to_date(to_timestamp(col("creation_date")))
    )
    .withColumn(
        "closing_date",
        to_date(to_timestamp(col("submission_deadline")))
    )

    # Source
    .withColumn(
        "source",
        lit("tanafos")
    )

    # Furas unique key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # Existing arrays map directly
    .withColumn(
        "categories",
        col("classifications")
    )
    .withColumn(
        "regions",
        col("regions")
    )

    # Price
    .withColumn(
        "document_price",
        col("price").cast("decimal(18,2)")
    )

    # Existing tech classification
    .withColumn(
        "is_tech",
        col("is_tech").cast("boolean")
    )

    # URL
    .withColumn(
        "tender_url",
        trim(col("url"))
    )

    # Fields unavailable from this source
    .withColumn("tender_name_en", lit(None).cast("string"))
    .withColumn("description", lit(None).cast("string"))
    .withColumn("opening_date", lit(None).cast("date"))

    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )

    .withColumn("currency", lit("SAR"))

    # Bronze file doesn't contain ingestion timestamp
    .withColumn(
        "ingestion_timestamp",
        lit(None).cast("timestamp")
    )

    # Exact 24-column Silver schema
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# checking tanafos schema and also checking data quality: 

tanafos_silver.printSchema()

print("Rows:", tanafos_silver.count())
print("Columns:", len(tanafos_silver.columns))

print(
    "Unique tender keys:",
    tanafos_silver.select("tender_key").distinct().count()
)

print(
    "Null closing dates:",
    tanafos_silver.filter(col("closing_date").isNull()).count()
)

print(
    "Tech tenders:",
    tanafos_silver.filter(col("is_tech") == True).count()
)

root
 |-- tender_key: string (nullable = true)
 |-- source: string (nullable = false)
 |-- source_tender_id: string (nullable = true)
 |-- reference_number: string (nullable = true)
 |-- tender_name: string (nullable = true)
 |-- tender_name_en: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- source_tender_type: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- source_status: string (nullable = true)
 |-- status: string (nullable = false)
 |-- publish_date: date (nullable = true)
 |-- closing_date: date (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- regions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- estimated_value_min: decimal(18,2) (nullable = true)
 |-- estimated_value_max: decimal(18,2) (nullable = true)
 |-- document_price: decimal(18,2) (nullable = tr

In [0]:

tanafos_silver.select(
    "source_tender_id",
    "reference_number",
    "tender_name",
    "agency_name",
    "source_tender_type",
    "status",
    "publish_date",
    "closing_date",
    "categories",
    "regions",
    "document_price",
    "currency",
    "is_tech"
).show(10, truncate=False)

+----------------+----------------+------------------------------------------------------------------------------+---------------------------------------------------------------+------------------+------+------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+--------------+--------+-------+
|source_tender_id|reference_number|tender_name                                                                   |agency_name                                                    |source_tender_type|status|publish_date|closing_date|categories                                                                                                                                                                                      |regions             |document_price|currency|is_tech|
+----------------+----------------+-----------

In [0]:

# from delta.tables import DeltaTable

# tanafos_silver_path = (
#     "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/tanafos"
# )

# # If Silver already exists -> MERGE
# if DeltaTable.isDeltaTable(spark, tanafos_silver_path):

#     target = DeltaTable.forPath(spark, tanafos_silver_path)

#     (
#         target.alias("old")
#         .merge(
#             tanafos_silver.alias("new"),
#             "old.tender_key = new.tender_key"
#         )
#         .whenMatchedUpdateAll()
#         .whenNotMatchedInsertAll()
#         .execute()
#     )

#     print("Tanafos Silver MERGE completed successfully!")

# # First run only -> create Silver
# else:
#     (
#         tanafos_silver.write
#         .format("delta")
#         .mode("overwrite")
#         .save(tanafos_silver_path)
#     )

#     print("Tanafos Silver created successfully!")

# new cell here:

# ============================================================
# STEP: Merge Tanafos data into the Silver layer
# Purpose:
# Incrementally update existing Tanafos tenders and insert
# new tenders using the reusable Silver MERGE function.
# ============================================================

tanafos_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/tanafos"
)

merge_to_silver(
    tanafos_silver,
    tanafos_silver_path,
    "Tanafos"
)

Tanafos: Silver MERGE completed.


In [0]:
# Data checking after the merge to silver:

tanafos_check = (
    spark.read
    .format("delta")
    .load(tanafos_silver_path)
)

print("Rows:", tanafos_check.count())
print("Columns:", len(tanafos_check.columns))
print(
    "Unique tender keys:",
    tanafos_check.select("tender_key").distinct().count()
)


Rows: 30
Columns: 24
Unique tender keys: 30


## Source NO.5: TAQEEM Bronze -> Silver

In [0]:

taqeem_path = bronze_path + "TAQEEM/"

# Get all date folders
taqeem_folders = dbutils.fs.ls(taqeem_path)

# Choose latest folder
latest_taqeem_folder = sorted(
    [f.path for f in taqeem_folders if f.isDir()],
    reverse=True
)[0]

print("Latest TAQEEM folder:", latest_taqeem_folder)

# Get JSON files from latest folder
taqeem_files = [
    f for f in dbutils.fs.ls(latest_taqeem_folder)
    if not f.isDir() and f.name.lower().endswith(".json")
]

# Choose latest JSON file
latest_taqeem_file = max(
    taqeem_files,
    key=lambda f: f.modificationTime
)

taqeem_file = latest_taqeem_file.path

print("Latest TAQEEM file:", taqeem_file)

# Read latest file
taqeem_raw = (
    spark.read
    .option("multiline", "true")
    .json(taqeem_file)
)

taqeem_raw.printSchema()

print("Top-level rows:", taqeem_raw.count())
print("Number of columns:", len(taqeem_raw.columns))

for c in taqeem_raw.columns:
    print(c)



Latest TAQEEM folder: abfss://bronze@tenderdatalake11.dfs.core.windows.net/TAQEEM/2026-09-16/
Latest TAQEEM file: abfss://bronze@tenderdatalake11.dfs.core.windows.net/TAQEEM/2026-09-16/taqeem_2026-09-16_09-30-44_tech_only.json
root
 |-- detail_url: string (nullable = true)
 |-- is_tech: boolean (nullable = true)
 |-- matched_keywords: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- publish_date: string (nullable = true)
 |-- purpose_summary: string (nullable = true)
 |-- status: string (nullable = true)
 |-- tender_id: string (nullable = true)
 |-- tender_number: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- title: string (nullable = true)

Top-level rows: 6
Number of columns: 10
detail_url
is_tech
matched_keywords
publish_date
purpose_summary
status
tender_id
tender_number
tender_type
title


In [0]:
display(taqeem_raw.limit(10))

detail_url,is_tech,matched_keywords,publish_date,purpose_summary,status,tender_id,tender_number,tender_type,title
https://taqeem.gov.sa/en/tenders/164,true,List(بيانات),null,null,Cancelled,164,PRCOMP20260600028,Public tender,مشروع الامتثال لمعايير مكتب إدارة البيانات الوطنية (المرحلة الأولى)
https://taqeem.gov.sa/en/tenders/161,true,List(سيبراني),null,null,Closed,161,PRCOMP20260400024,Public tender,مشروع الخدمات المدارة للأمن السيبراني
https://taqeem.gov.sa/en/tenders/159,true,"List(رقمي, التحول الرقمي)",null,null,Closed,159,PRCOMP20260200014,Public tender,مشروع تصميم استراتيجية التحول الرقمي
https://taqeem.gov.sa/en/tenders/157,true,List(أنظمة),null,null,Closed,157,PRCOMP20260200015,Public tender,مشروع التطوير والتشغيل المستمر للأنظمة الإلكترونية
https://taqeem.gov.sa/en/tenders/152,true,"List(تقنية, تقني, تقنية المعلومات, أنظمة, الأمن السيبراني, سيبراني)",null,null,Closed,152,PRCOMP20260100009,Public tender,تدقيق أنظمة تقنية المعلومات وضوابط الأمن السيبراني
https://taqeem.gov.sa/en/tenders/144,true,List(منظومة),null,null,Closed,144,PRCOMP20251100020,Public tender,التحول الاستراتيجي في منظومة تقدير


In [0]:
# This cell displays a sample of the important TAQEEM fields before transformation. It helps us understand the identifiers, tender information, status, dates, 
# technology classification, and URL values so they can be mapped correctly to the common Silver schema.

taqeem_raw.select(
    "tender_id",
    "tender_number",
    "title",
    "tender_type",
    "status",
    "publish_date",
    "purpose_summary",
    "is_tech",
    "matched_keywords",
    "detail_url"
).show(10, truncate=False)

+---------+-----------------+-------------------------------------------------------------------+-------------+---------+------------+---------------+-------+---------------------------------------------------------------+------------------------------------+
|tender_id|tender_number    |title                                                              |tender_type  |status   |publish_date|purpose_summary|is_tech|matched_keywords                                               |detail_url                          |
+---------+-----------------+-------------------------------------------------------------------+-------------+---------+------------+---------------+-------+---------------------------------------------------------------+------------------------------------+
|164      |PRCOMP20260600028|مشروع الامتثال لمعايير مكتب إدارة البيانات الوطنية (المرحلة الأولى)|Public tender|Cancelled|NULL        |NULL           |true   |[بيانات]                                                      

In [0]:
# This is the raw data-quality validation
# It compares the total number of tenders with unique tender IDs and tender numbers, checks whether any tender IDs are missing, and counts null values in the 
# important source fields before transformation.

from pyspark.sql.functions import col, sum as spark_sum

print("Total TAQEEM tenders:", taqeem_raw.count())

print(
    "Unique tender IDs:",
    taqeem_raw.select("tender_id").distinct().count()
)

print(
    "Unique tender numbers:",
    taqeem_raw.select("tender_number").distinct().count()
)

print(
    "Null tender IDs:",
    taqeem_raw.filter(col("tender_id").isNull()).count()
)

taqeem_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "tender_id",
        "tender_number",
        "title",
        "tender_type",
        "status",
        "publish_date",
        "purpose_summary",
        "is_tech",
        "detail_url"
    ]
]).show()

Total TAQEEM tenders: 6
Unique tender IDs: 6
Unique tender numbers: 6
Null tender IDs: 0
+---------+-------------+-----+-----------+------+------------+---------------+-------+----------+
|tender_id|tender_number|title|tender_type|status|publish_date|purpose_summary|is_tech|detail_url|
+---------+-------------+-----+-----------+------+------------+---------------+-------+----------+
|        0|            0|    0|          0|     0|           6|              6|      0|         0|
+---------+-------------+-----+-----------+------+------------+---------------+-------+----------+



In [0]:
taqeem_raw.select(
    "tender_id",
    "tender_number",
    "title",
    "tender_type",
    "status",
    "is_tech",
    "detail_url"
).show(10, truncate=False)

+---------+-----------------+-------------------------------------------------------------------+-------------+---------+-------+------------------------------------+
|tender_id|tender_number    |title                                                              |tender_type  |status   |is_tech|detail_url                          |
+---------+-----------------+-------------------------------------------------------------------+-------------+---------+-------+------------------------------------+
|164      |PRCOMP20260600028|مشروع الامتثال لمعايير مكتب إدارة البيانات الوطنية (المرحلة الأولى)|Public tender|Cancelled|true   |https://taqeem.gov.sa/en/tenders/164|
|161      |PRCOMP20260400024|مشروع الخدمات المدارة للأمن السيبراني                              |Public tender|Closed   |true   |https://taqeem.gov.sa/en/tenders/161|
|159      |PRCOMP20260200014|مشروع تصميم استراتيجية التحول الرقمي                               |Public tender|Closed   |true   |https://taqeem.gov.sa/en/tenders/159

In [0]:
# This cell profiles the distinct status and tender-type values available in TAQEEM. We inspect the source terminology before deciding how these fields should 
# be standardized in the Silver layer.

taqeem_raw.select("status").distinct().show(truncate=False)
taqeem_raw.select("tender_type").distinct().show(truncate=False)

+---------+
|status   |
+---------+
|Closed   |
|Cancelled|
+---------+

+-------------+
|tender_type  |
+-------------+
|Public tender|
+-------------+



In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lit,
    sha2,
    concat_ws
)

taqeem_silver = (
    taqeem_raw

    # IDs
    .withColumn(
        "source_tender_id",
        trim(col("tender_id"))
    )
    .withColumn(
        "reference_number",
        trim(col("tender_number"))
    )

    # Main information
    .withColumn(
        "tender_name",
        trim(col("title"))
    )

    # Type
    .withColumn(
        "source_tender_type",
        trim(col("tender_type"))
    )
    .withColumn(
        "tender_type",
        trim(col("tender_type"))
    )

    # Status
    .withColumn(
        "source_status",
        trim(col("status"))
    )
    .withColumn(
        "status",
        upper(trim(col("status")))
    )

    # Source
    .withColumn(
        "source",
        lit("taqeem")
    )

    # Unique key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # URL and technology flag
    .withColumn(
        "tender_url",
        trim(col("detail_url"))
    )
    .withColumn(
        "is_tech",
        col("is_tech").cast("boolean")
    )

    # Fields unavailable in this Bronze source
    .withColumn("tender_name_en", lit(None).cast("string"))
    # .withColumn("description", lit(None).cast("string"))
    .withColumn(
    "description",
    trim(col("purpose_summary"))
    )
    .withColumn("agency_name", lit("TAQEEM"))

    .withColumn("publish_date", lit(None).cast("date"))
    .withColumn("closing_date", lit(None).cast("date"))
    .withColumn("opening_date", lit(None).cast("date"))

    .withColumn(
        "categories",
        lit(None).cast("array<string>")
    )
    .withColumn(
        "regions",
        lit(None).cast("array<string>")
    )

    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )

    .withColumn("currency", lit(None).cast("string"))
    .withColumn(
        "ingestion_timestamp",
        lit(None).cast("timestamp")
    )

    # Exact common Silver schema
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# This cell validates the transformed TAQEEM dataset before loading it into Silver.

taqeem_silver.printSchema()

print("Rows:", taqeem_silver.count())
print("Columns:", len(taqeem_silver.columns))

print(
    "Unique tender keys:",
    taqeem_silver.select("tender_key").distinct().count()
)

print(
    "Null tender keys:",
    taqeem_silver.filter(col("tender_key").isNull()).count()
)

print(
    "Tech tenders:",
    taqeem_silver.filter(col("is_tech") == True).count()
)

taqeem_silver.select(
    "source_tender_id",
    "reference_number",
    "tender_name",
    "source_tender_type",
    "source_status",
    "status",
    "is_tech"
).show(truncate=False)

root
 |-- tender_key: string (nullable = true)
 |-- source: string (nullable = false)
 |-- source_tender_id: string (nullable = true)
 |-- reference_number: string (nullable = true)
 |-- tender_name: string (nullable = true)
 |-- tender_name_en: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agency_name: string (nullable = false)
 |-- source_tender_type: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- source_status: string (nullable = true)
 |-- status: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- closing_date: date (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- regions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- estimated_value_min: decimal(18,2) (nullable = true)
 |-- estimated_value_max: decimal(18,2) (nullable = true)
 |-- document_price: decimal(18,2) (nullable = tr

In [0]:
taqeem_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/taqeem"
)

merge_to_silver(
    taqeem_silver,
    taqeem_silver_path,
    "TAQEEM"
)

TAQEEM: Silver MERGE completed.


In [0]:
# Data Quality checks:

taqeem_check = (
    spark.read
    .format("delta")
    .load(taqeem_silver_path)
)

print("Rows:", taqeem_check.count())
print("Columns:", len(taqeem_check.columns))
print(
    "Unique tender keys:",
    taqeem_check.select("tender_key").distinct().count()
)

Rows: 6
Columns: 24
Unique tender keys: 6


## Source NO.6: TAHAKOM Bronze -> Silver

In [0]:
tahakom_path = bronze_path + "TAHAKOM/"

# Get all date folders under TAHAKOM
tahakom_folders = dbutils.fs.ls(tahakom_path)

# Choose latest folder
latest_tahakom_folder = sorted(
    [f.path for f in tahakom_folders if f.isDir()],
    reverse=True
)[0]

print("Latest TAHAKOM folder:", latest_tahakom_folder)

# Get JSON files from latest folder
tahakom_files = [
    f for f in dbutils.fs.ls(latest_tahakom_folder)
    if not f.isDir() and f.name.lower().endswith(".json")
]

# Choose latest JSON file
latest_tahakom_file = max(
    tahakom_files,
    key=lambda f: f.modificationTime
)

tahakom_file = latest_tahakom_file.path

print("Latest TAHAKOM file:", tahakom_file)

# Read latest file
tahakom_raw = (
    spark.read
    .option("multiline", "true")
    .json(tahakom_file)
)

tahakom_raw.printSchema()

print("Top-level rows:", tahakom_raw.count())
print("Number of columns:", len(tahakom_raw.columns))

for c in tahakom_raw.columns:
    print(c)



Latest TAHAKOM folder: abfss://bronze@tenderdatalake11.dfs.core.windows.net/TAHAKOM/2026-09-17/
Latest TAHAKOM file: abfss://bronze@tenderdatalake11.dfs.core.windows.net/TAHAKOM/2026-09-17/tahakom_tenders_2026.json
root
 |-- Contact_us: string (nullable = true)
 |-- Proposals_deadline: string (nullable = true)
 |-- QA_publishing_date: string (nullable = true)
 |-- Questions_deadline: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Tender_details_arabic: string (nullable = true)
 |-- Tender_details_english: string (nullable = true)
 |-- Tender_issuing_date: string (nullable = true)
 |-- Tender_open_date: string (nullable = true)
 |-- Tender_type: string (nullable = true)
 |-- Title_arabic: string (nullable = true)
 |-- Title_english: string (nullable = true)
 |-- contact_us_2: string (nullable = true)
 |-- createdAt: string (nullable = true)
 |-- id: long (nullable = true)
 |-- locale: string (nullable = true)
 |-- localizations: array (nullable = true)
 |    |-- ele

In [0]:
display(tahakom_raw.limit(10))

Contact_us,Proposals_deadline,QA_publishing_date,Questions_deadline,Status,Tender_details_arabic,Tender_details_english,Tender_issuing_date,Tender_open_date,Tender_type,Title_arabic,Title_english,contact_us_2,createdAt,id,locale,localizations,publishedAt,strapi_stage,tender_number,updatedAt
saldahash@Tahakom.com,2026-04-20,2026-04-20,2026-04-13,Open,تقديم حملة التوعية بالمراجعة الداخلية ولمزيد من المعلومات نأمل التواصل عبر الايميل ادناه,"Submission of an Internal Audit Awareness Campaign. For More information, Please contact us through email below",2026-04-13,2026-04-20,RFQ,حملة التوعية بالمراجعة الداخلية,Internal Audit Awareness Campaign,null,2026-04-15T07:03:04.725Z,2271,en,List(),2026-04-15T07:03:05.940Z,null,102001115,2026-04-15T07:03:05.958Z
abdelkarim.c@Tahakom.com,2026-01-04,2026-01-04,2025-12-02,Open,تعلن الشركة السعودية للتقنيات الأمنية الشاملة (تحكّم) عن طرح منافسة لتوقيع اتفاقية إطارية لإنشاء مراكز عمليات تحكّم، على أن يتم تنفيذ المشروع إما بنظام الإنشاء فقط أو بنظام التصميم والتنفيذ، وذلك وفقًا لما هو سيوضح في طلب العروض المصغّر (Mini RFP). وفي حال اعتماد نظام التصميم والتنفيذ، يتعيّن على المناقص التحقق من صحة التصميم المبدئي وتطوير تصميم تفصيلي تكييفي يلائم موقع المشروع داخل المملكة، مع إعداد رسومات التنفيذ (IFC) ورسومات (Shop Drawings). وتؤكد الشركة أن جميع الأعمال يجب أن تُنفذ وفقًا للمواصفات والرسومات المعتمدة ضمن الاتفاقية الإطارية. فعلى الشركات المتخصصة الراغبة في المشاركة، الحصول على وثائق المنافسة وتقديم عروضها حسب االشروط والأحكام الواردة أو مزيد من المعلومات الرجاء التواصل عبر الايميل ادناه تفاصيل المنافسة: الاسم: م.فادي إسماعيل عبد الكريم. إيميل : fabdelkarim.c@tahakom.com,"Saudi Technology and Security Comprehensive Control Company (Tahakom) announces the launch of a tender for signing a framework agreement for the construction of Control Operation Centers. The project shall be executed either under a construction-only or a design and build model, as detailed in the Mini Request for Proposal (Mini RFP). In the case of the design and build model, the bidder is required to verify the preliminary design and develop a detailed adaptive design suitable for the designated project location within the Kingdom, in addition to preparing the Issued for Construction (IFC) and Shop Drawings. The Company emphasizes that all works must be carried out in accordance with the specifications and approved drawings under the Framework Agreement. Interested and qualified companies are invited to obtain the tender documents and submit their proposals in accordance with the terms and conditions stated. For further information, please contact us via the email below: Name :Eng. Fadi Ismail Abdelkarim, Email: fabdelkarim.c@tahakom.com",2025-12-02,2026-01-04,RFQ,اتفاقية إطارية لإنشاء مراكز تحكّم التابعة للشركة السعودية للتقنيات الأمنية الشاملة (تحكّم).,"Construction Framework Agreement, for Marakez Tahakom for the Saudi Technology and Security Comprehensive Control Company (TAHAKOM), Framework Agreement.",null,2025-12-04T09:31:54.832Z,2197,en,List(),2025-12-04T09:32:01.670Z,null,101000593,2025-12-10T09:29:12.762Z
makhan.c@tahakom.com,2026-01-22,2025-12-31,2026-01-14,Open,"تطلب شركة ""تحكم"" حلول أثاث متكاملة لمرافقها في جميع أنحاء المملكة العربية السعودية، وفقاً للمواصفات المذكورة في نطاق العمل.",TAHAKOM requires Integrated Furniture Solutions for their facilities across the Kingdom of Saudi Arabia as per the specifications mentioned in the SOW.,2025-12-31,2026-01-31,RFP,حلول الأثاث المتكاملة,Integrated Furniture Solutions,null,2025-12-31T09:38:31.423Z,2215,en,List(),2025-12-31T09:38:32.386Z,null,101000647,2025-12-31T09:38:32.394Z
aoaldajani@tahakom.com,2026-01-01,2026-01-01,2025-12-21,Open,تنظيم حملة داخلية في مقر تحكم لاستعراض إنجازات قطاع الذكاء الاصطناعي والمشاريع والأبحاث العلمية التي أنجزها فريق الذكاء الاصطناعي. لمزيد من المعلومات حول نطاق العمل الرجاء التواصل عن طريق البريد الالكتروني ادناه,"An internal campaign at Tahakom headquarters showcasing the AI Sector’s achievements, projects, published papers. For more informatio

In [0]:
# This cell displays the important raw TAHAKOM fields before transformation.

tahakom_raw.select(
    "id",
    "tender_number",
    "Title_arabic",
    "Title_english",
    "Tender_type",
    "Status",
    "Tender_issuing_date",
    "Tender_open_date",
    "Proposals_deadline",
    "Tender_details_arabic",
    "Tender_details_english"
).show(10, truncate=False)

+----+-------------+-------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------+-------------------+----------------+------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
# Data checking 

from pyspark.sql.functions import col, sum as spark_sum

print("Total TAHAKOM tenders:", tahakom_raw.count())

print(
    "Unique IDs:",
    tahakom_raw.select("id").distinct().count()
)

print(
    "Unique tender numbers:",
    tahakom_raw.select("tender_number").distinct().count()
)

print(
    "Null IDs:",
    tahakom_raw.filter(col("id").isNull()).count()
)

tahakom_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "id",
        "tender_number",
        "Title_arabic",
        "Title_english",
        "Tender_type",
        "Status",
        "Tender_issuing_date",
        "Tender_open_date",
        "Proposals_deadline"
    ]
]).show()

Total TAHAKOM tenders: 160
Unique IDs: 160
Unique tender numbers: 148
Null IDs: 0
+---+-------------+------------+-------------+-----------+------+-------------------+----------------+------------------+
| id|tender_number|Title_arabic|Title_english|Tender_type|Status|Tender_issuing_date|Tender_open_date|Proposals_deadline|
+---+-------------+------------+-------------+-----------+------+-------------------+----------------+------------------+
|  0|            2|           0|            0|          0|     0|                  0|               0|                 0|
+---+-------------+------------+-------------+-----------+------+-------------------+----------------+------------------+



In [0]:
# This cell transforms TAHAKOM into our common 24-column Silver schema. It maps the source ID and reference number, preserves the Arabic and English tender 
# titles, standardizes the tender type and status, converts the issuing, opening, and deadline dates, generates the unique Furas tender key, and adds typed 
# NULL values for fields TAHAKOM does not provide.

from pyspark.sql.functions import (
    col,
    trim,
    upper,
    to_date,
    lit,
    sha2,
    concat_ws
)

tahakom_silver = (
    tahakom_raw

    # IDs
    .withColumn(
        "source_tender_id",
        col("id").cast("string")
    )
    .withColumn(
        "reference_number",
        trim(col("tender_number"))
    )

    # Names and description
    .withColumn(
        "tender_name",
        trim(col("Title_arabic"))
    )
    .withColumn(
        "tender_name_en",
        trim(col("Title_english"))
    )
    .withColumn(
        "description",
        trim(col("Tender_details_arabic"))
    )

    # Agency
    .withColumn(
        "agency_name",
        lit("TAHAKOM")
    )

    # Tender type
    .withColumn(
        "source_tender_type",
        trim(col("Tender_type"))
    )
    .withColumn(
        "tender_type",
        trim(col("Tender_type"))
    )

    # Status
    .withColumn(
        "source_status",
        trim(col("Status"))
    )
    .withColumn(
        "status",
        upper(trim(col("Status")))
    )

    # Dates
    .withColumn(
        "publish_date",
        to_date(col("Tender_issuing_date"), "yyyy-MM-dd")
    )
    .withColumn(
        "closing_date",
        to_date(col("Proposals_deadline"), "yyyy-MM-dd")
    )
    .withColumn(
        "opening_date",
        to_date(col("Tender_open_date"), "yyyy-MM-dd")
    )

    # Source
    .withColumn(
        "source",
        lit("tahakom")
    )

    # Deterministic key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    # Fields unavailable in this source
    .withColumn("categories", lit(None).cast("array<string>"))
    .withColumn("regions", lit(None).cast("array<string>"))

    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )

    .withColumn("currency", lit(None).cast("string"))
    .withColumn("tender_url", lit(None).cast("string"))
    .withColumn("is_tech", lit(None).cast("boolean"))

    # createdAt is available, so preserve it as ingestion timestamp
    .withColumn(
        "ingestion_timestamp",
        col("createdAt").cast("timestamp")
    )

    # Exact 24-column schema
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# Checking tahakom_silver schema and the data quality

tahakom_silver.printSchema()

print("Rows:", tahakom_silver.count())
print("Columns:", len(tahakom_silver.columns))

print(
    "Unique tender keys:",
    tahakom_silver.select("tender_key").distinct().count()
)

print(
    "Null tender keys:",
    tahakom_silver.filter(col("tender_key").isNull()).count()
)

print(
    "Null publish dates:",
    tahakom_silver.filter(col("publish_date").isNull()).count()
)

print(
    "Null closing dates:",
    tahakom_silver.filter(col("closing_date").isNull()).count()
)

print(
    "Null opening dates:",
    tahakom_silver.filter(col("opening_date").isNull()).count()
)

root
 |-- tender_key: string (nullable = true)
 |-- source: string (nullable = false)
 |-- source_tender_id: string (nullable = true)
 |-- reference_number: string (nullable = true)
 |-- tender_name: string (nullable = true)
 |-- tender_name_en: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agency_name: string (nullable = false)
 |-- source_tender_type: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- source_status: string (nullable = true)
 |-- status: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- closing_date: date (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- regions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- estimated_value_min: decimal(18,2) (nullable = true)
 |-- estimated_value_max: decimal(18,2) (nullable = true)
 |-- document_price: decimal(18,2) (nullable = tr

In [0]:
tahakom_silver.select(
    "source_tender_id",
    "reference_number",
    "tender_name",
    "tender_name_en",
    "source_tender_type",
    "status",
    "publish_date",
    "closing_date",
    "opening_date"
).show(10, truncate=False)

+----------------+----------------+-------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+------+------------+------------+------------+
|source_tender_id|reference_number|tender_name                                                                                |tender_name_en                                                                                                                                           |source_tender_type|status|publish_date|closing_date|opening_date|
+----------------+----------------+-------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+------

In [0]:

tahakom_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/tahakom"
)

merge_to_silver(
    tahakom_silver,
    tahakom_silver_path,
    "TAHAKOM"
)


TAHAKOM: Silver MERGE completed.


In [0]:
# This is the post-write validation for TAHAKOM.

tahakom_check = (
    spark.read
    .format("delta")
    .load(tahakom_silver_path)
)

print("Rows:", tahakom_check.count())
print("Columns:", len(tahakom_check.columns))
print(
    "Unique tender keys:",
    tahakom_check.select("tender_key").distinct().count()
)

Rows: 160
Columns: 24
Unique tender keys: 160


## Source NO.7: FORSAH Bronze -> Silver

In [0]:
forsah_path = bronze_path + "FORSAH/"

# Get all date folders
forsah_folders = dbutils.fs.ls(forsah_path)

# Choose latest folder
latest_forsah_folder = sorted(
    [f.path for f in forsah_folders if f.isDir()],
    reverse=True
)[0]

print("Latest FORSAH folder:", latest_forsah_folder)

# Get JSON files from latest folder
forsah_files = [
    f for f in dbutils.fs.ls(latest_forsah_folder)
    if not f.isDir() and f.name.lower().endswith(".json")
]

latest_forsah_file = max(
    forsah_files,
    key=lambda f: f.modificationTime
)

forsah_file = latest_forsah_file.path

print("Latest FORSAH file:", forsah_file)

forsah_raw = (
    spark.read
    .option("multiline", "true")
    .json(forsah_file)
)

forsah_raw.printSchema()

print("Top-level rows:", forsah_raw.count())
print("Number of columns:", len(forsah_raw.columns))

for c in forsah_raw.columns:
    print(c)




Latest FORSAH folder: abfss://bronze@tenderdatalake11.dfs.core.windows.net/FORSAH/2026-09-28/
Latest FORSAH file: abfss://bronze@tenderdatalake11.dfs.core.windows.net/FORSAH/2026-09-28/forsah_2026-09-28_15-52-09.json
root
 |-- bids_count: long (nullable = true)
 |-- categories_ar: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories_en: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- cities: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- close_date: string (nullable = true)
 |-- due_date: string (nullable = true)
 |-- extended_due_date: string (nullable = true)
 |-- id: string (nullable = true)
 |-- is_tech: boolean (nullable = true)
 |-- matched_categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- publish_date: string (nullable = true)
 |-- status_key: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type_key: string (nullable = true)
 |--

In [0]:
display(forsah_raw.limit(10))

bids_count,categories_ar,categories_en,cities,close_date,due_date,extended_due_date,id,is_tech,matched_categories,publish_date,status_key,title,type_key,type_name_ar,type_name_en,url,value_max,value_min,value_range_en,view_count
0,List(تجارة مواد البناء والادوات الكهربائية والصحية),"List(Trade in building materials, electrical and sanitary appliances)",List(الرياض),null,2026-09-30 07:59:59,null,1f1bb4b1-e6f8-6296-95c1-05c726730511,true,"List(Trade in building materials, electrical and sanitary appliances)",2026-09-28,open,توريد مواد بناء و سخانات مياه,rfq,شراء مباشر,RFQ,https://forsah.sa/marketplace/1f1bb4b1-e6f8-6296-95c1-05c726730511,100000,25001,Small,29
0,List(الاستشارات الادارية),List(Management Consultation),List(جدة),null,2026-10-05 11:59:59,null,1f1bb14e-95dd-6278-88f8-0f01b4b9b26e,false,List(),2026-09-28,open,منافسة إسناد خدمات الاستشارات الاستراتيجية الشاملة,rfp,مناقصة,RFP,https://forsah.sa/marketplace/1f1bb14e-95dd-6278-88f8-0f01b4b9b26e,1000000000,250001,Big,256
5,List(أجهزة الحاسب الآلي),List(Computers),List(الظهران),null,2026-09-29 23:59:59,null,1f1bb20b-a552-65ec-8a41-671fd98944d6,false,List(),2026-09-28,open,اجهزة الكترونية,rfq,شراء مباشر,RFQ,https://forsah.sa/marketplace/1f1bb20b-a552-65ec-8a41-671fd98944d6,5000,0,Micro,138
3,List(خدمات الصيانة والتشغيل),List(Maintenance and operation services),List(جازان),null,2026-10-01 23:59:59,null,1f1bb02c-2aa6-6fae-9cba-f3cf80ac073e,false,List(),2026-09-28,open,صيانة مبنى الضيافة لمصنع جيزان,rfp,مناقصة,RFP,https://forsah.sa/marketplace/1f1bb02c-2aa6-6fae-9cba-f3cf80ac073e,1000000000,250001,Big,161
14,"List(خدمات الصيانة والتشغيل, خدمات الموارد البشرية(القوى العاملة))","List(Maintenance and operation services, Human resources services (manpower))",List(الرياض),null,2026-09-30 08:59:59,2026-10-04 08:59:59,1f1ba800-f411-62e2-bda0-bb5869aa318b,false,List(),2026-09-28,open,توفير خدمات النظافة والضيافة والعمالة ومستلزماتها,rfp,مناقصة,RFP,https://forsah.sa/marketplace/1f1ba800-f411-62e2-bda0-bb5869aa318b,100000,25001,Small,255
15,"List(توريد وتركيب الاجهزة الكهربائية, أجهزة الحاسب الآلي)","List(Supply and installation of electrical appliances, Computers)",List(الرياض),null,2026-09-30 23:59:59,null,1f1bb085-5b26-673e-aa53-ede838ca1523,true,List(Supply and installation of electrical appliances),2026-09-28,open,تأمين وتوريد شاشة لقاعة الاجتماعات,rfq,شراء مباشر,RFQ,https://forsah.sa/marketplace/1f1bb085-5b26-673e-aa53-ede838ca1523,25000,5001,Very Small,175
4,List(أجهزة شبكات الاتصالات وتقنية المعلومات),List(Information & Communication Network Devices),List(الرياض),null,2026-09-30 08:59:59,null,1f1bb106-f8e4-6c24-b616-85d9803adf03,true,List(Information & Communication Network Devices),2026-09-28,open,توريد راوتر 5G باشتراك سنوي – الرياض,rfq,شراء مباشر,RFQ,https://forsah.sa/marketplace/1f1bb106-f8e4-6c24-b616-85d9803adf03,5000,0,Micro,98
0,"List(أثاث وتجهيزات رياضية, الأثاث النوعي)","List(Sports furniture & equipment, Qualitative furniture)",List(جدة),null,2026-10-01 10:59:59,null,1f1ba719-408f-69a4-a27f-690d8741f541,false,List(),2026-09-27,open,معدات رياضية لكرة الطائرة,rfq,شراء مباشر,RFQ,https://forsah.sa/marketplace/1f1ba719-408f-69a4-a27f-690d8741f541,100000,25001,Small,85
13,List(الاستشارات الهندسية),List(Engineering Consultation),List(مكة المكرمة),null,2026-10-01 11:59:59,null,1f1ba5b8-ca88-6ef6-93c1-7105d37851e4,false,List(),2026-09-27,open,إعداد مخطط كنتوري لأرض الهدا – الطائف,rfq,شراء مباشر,RFQ,https://forsah.sa/marketplace/1f1ba5b8-ca88-6ef6-93c1-7105d37851e4,25000,5001,Very Small,183
1,List(تامين السيارات),List(cars insurance),List(مكة المكرمة),null,2026-09-28 14:59:59,2026-09-29 01:59:59,1f1ba609-8aa8-6894-baaf-43a464607cbc,false,List(),2026-09-27,open,تامين سيارات الغرفة ضد الغير 2026,rfq,شراء مباشر,RFQ,https://forsah.sa/marketplace/1f1ba609-8aa8-6894-baaf-43a464607cbc,25000,5001,Very Small,64


In [0]:
# This cell inspects the important raw FORSAH fields before transformation, including identifiers, tender type and status, dates, categories, cities, estimated 
# values, technology classification

forsah_raw.select(
    "id",
    "title",
    "type_key",
    "type_name_ar",
    "type_name_en",
    "status_key",
    "publish_date",
    "due_date",
    "extended_due_date",
    "close_date",
    "categories_ar",
    "cities",
    "value_min",
    "value_max",
    "is_tech",
    "url"
).show(10, truncate=False)

+------------------------------------+--------------------------------------------------+--------+------------+------------+----------+------------+-------------------+-------------------+----------+--------------------------------------------------------------+-------------+---------+----------+-------+------------------------------------------------------------------+
|id                                  |title                                             |type_key|type_name_ar|type_name_en|status_key|publish_date|due_date           |extended_due_date  |close_date|categories_ar                                                 |cities       |value_min|value_max |is_tech|url                                                               |
+------------------------------------+--------------------------------------------------+--------+------------+------------+----------+------------+-------------------+-------------------+----------+-------------------------------------------------------

In [0]:
# checks the quality and completeness of the raw FORSAH data before transforming it into Silver.

from pyspark.sql.functions import col, sum as spark_sum

print("Total FORSAH tenders:", forsah_raw.count())

print(
    "Unique IDs:",
    forsah_raw.select("id").distinct().count()
)

print(
    "Null IDs:",
    forsah_raw.filter(col("id").isNull()).count()
)

forsah_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "id",
        "title",
        "type_name_ar",
        "status_key",
        "publish_date",
        "due_date",
        "extended_due_date",
        "close_date",
        "value_min",
        "value_max",
        "is_tech",
        "url"
    ]
]).show()

Total FORSAH tenders: 13
Unique IDs: 13
Null IDs: 0
+---+-----+------------+----------+------------+--------+-----------------+----------+---------+---------+-------+---+
| id|title|type_name_ar|status_key|publish_date|due_date|extended_due_date|close_date|value_min|value_max|is_tech|url|
+---+-----+------------+----------+------------+--------+-----------------+----------+---------+---------+-------+---+
|  0|    0|           0|         0|           0|       0|               11|        12|        0|        0|      0|  0|
+---+-----+------------+----------+------------+--------+-----------------+----------+---------+---------+-------+---+



In [0]:
# This cell profiles the distinct status and tender-type values used by FORSAH. We inspect these source values before standardizing them into the common Silver schema.

forsah_raw.select("status_key").distinct().show(truncate=False)

forsah_raw.select(
    "type_key",
    "type_name_ar",
    "type_name_en"
).distinct().show(truncate=False)

+----------+
|status_key|
+----------+
|pending   |
|open      |
+----------+

+--------+------------+------------+
|type_key|type_name_ar|type_name_en|
+--------+------------+------------+
|rfq     |شراء مباشر  |RFQ         |
|rfp     |مناقصة      |RFP         |
+--------+------------+------------+



In [0]:
# checking for the date

forsah_raw.filter(
    col("extended_due_date").isNotNull()
).select(
    "id",
    "status_key",
    "publish_date",
    "due_date",
    "extended_due_date",
    "close_date"
).show(20, truncate=False)

+------------------------------------+----------+------------+-------------------+-------------------+----------+
|id                                  |status_key|publish_date|due_date           |extended_due_date  |close_date|
+------------------------------------+----------+------------+-------------------+-------------------+----------+
|1f1ba800-f411-62e2-bda0-bb5869aa318b|open      |2026-09-28  |2026-09-30 08:59:59|2026-10-04 08:59:59|NULL      |
|1f1ba609-8aa8-6894-baaf-43a464607cbc|open      |2026-09-27  |2026-09-28 14:59:59|2026-09-29 01:59:59|NULL      |
+------------------------------------+----------+------------+-------------------+-------------------+----------+



In [0]:
# This cell transforms the raw FORSAH data into our common 24-column Silver schema. It standardizes identifiers, tender type, status and dates, uses the 
# extended deadline when available, maps categories and regions, preserves estimated tender values and the technology flag, and generates a unique tender key.

from pyspark.sql.functions import (
    col,
    trim,
    upper,
    to_date,
    to_timestamp,
    coalesce,
    lit,
    sha2,
    concat_ws
)

forsah_silver = (
    forsah_raw

    # IDs
    .withColumn(
        "source_tender_id",
        trim(col("id"))
    )
    .withColumn(
        "reference_number",
        lit(None).cast("string")
    )

    # Main information
    .withColumn(
        "tender_name",
        trim(col("title"))
    )
    .withColumn(
        "tender_name_en",
        lit(None).cast("string")
    )
    .withColumn(
        "description",
        lit(None).cast("string")
    )
    .withColumn(
        "agency_name",
        lit(None).cast("string")
    )

    # Tender type
    .withColumn(
        "source_tender_type",
        trim(col("type_name_ar"))
    )
    .withColumn(
        "tender_type",
        trim(col("type_name_ar"))
    )

    # Status
    .withColumn(
        "source_status",
        trim(col("status_key"))
    )
    .withColumn(
        "status",
        upper(trim(col("status_key")))
    )

    # Dates
    .withColumn(
        "publish_date",
        to_date(col("publish_date"), "yyyy-MM-dd")
    )
    .withColumn(
        "closing_date",
        to_date(
            coalesce(
                to_timestamp(
                    col("extended_due_date"),
                    "yyyy-MM-dd HH:mm:ss"
                ),
                to_timestamp(
                    col("due_date"),
                    "yyyy-MM-dd HH:mm:ss"
                )
            )
        )
    )
    .withColumn(
        "opening_date",
        lit(None).cast("date")
    )

    # Categories and regions already match our schema
    .withColumn(
        "categories",
        col("categories_ar")
    )
    .withColumn(
        "regions",
        col("cities")
    )

    # Estimated tender value
    .withColumn(
        "estimated_value_min",
        col("value_min").cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        col("value_max").cast("decimal(18,2)")
    )

    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )

    .withColumn(
        "currency",
        lit("SAR")
    )

    # URL + tech classification
    .withColumn(
        "tender_url",
        trim(col("url"))
    )
    .withColumn(
        "is_tech",
        col("is_tech").cast("boolean")
    )

    # Source
    .withColumn(
        "source",
        lit("forsah")
    )

    # Unique key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    .withColumn(
        "ingestion_timestamp",
        lit(None).cast("timestamp")
    )

    # Exact common Silver schema
    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# Checking data quality for forsah and print the schema

forsah_silver.printSchema()

print("Rows:", forsah_silver.count())
print("Columns:", len(forsah_silver.columns))

print(
    "Unique tender keys:",
    forsah_silver.select("tender_key").distinct().count()
)

print(
    "Null tender keys:",
    forsah_silver.filter(col("tender_key").isNull()).count()
)

print(
    "Null publish dates:",
    forsah_silver.filter(col("publish_date").isNull()).count()
)

print(
    "Null closing dates:",
    forsah_silver.filter(col("closing_date").isNull()).count()
)

print(
    "Tech tenders:",
    forsah_silver.filter(col("is_tech") == True).count()
)

root
 |-- tender_key: string (nullable = true)
 |-- source: string (nullable = false)
 |-- source_tender_id: string (nullable = true)
 |-- reference_number: string (nullable = true)
 |-- tender_name: string (nullable = true)
 |-- tender_name_en: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- source_tender_type: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- source_status: string (nullable = true)
 |-- status: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- closing_date: date (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- regions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- estimated_value_min: decimal(18,2) (nullable = true)
 |-- estimated_value_max: decimal(18,2) (nullable = true)
 |-- document_price: decimal(18,2) (nullable = tru

In [0]:
forsah_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/forsah"
)

merge_to_silver(
    forsah_silver,
    forsah_silver_path,
    "FORSAH"
)


FORSAH: Silver MERGE completed.


In [0]:
# Checking the data

forsah_check = (
    spark.read
    .format("delta")
    .load(forsah_silver_path)
)

print("Rows:", forsah_check.count())
print("Columns:", len(forsah_check.columns))
print(
    "Unique tender keys:",
    forsah_check.select("tender_key").distinct().count()
)

Rows: 1001
Columns: 24
Unique tender keys: 1001


## Source NO.8: GEO Bronze -> Silver

In [0]:

# GEO Bronze -> Silver

geo_path = bronze_path + "GEO/"

# Get all date folders
geo_folders = dbutils.fs.ls(geo_path)

# Choose latest folder automatically
latest_geo_folder = sorted(
    [f.path for f in geo_folders if f.isDir()],
    reverse=True
)[0]

print("Latest GEO folder:", latest_geo_folder)

# Get CSV files from latest folder
geo_files = [
    f for f in dbutils.fs.ls(latest_geo_folder)
    if not f.isDir() and f.name.lower().endswith(".csv")
]

latest_geo_file = max(
    geo_files,
    key=lambda f: f.modificationTime
)

geo_file = latest_geo_file.path

print("Latest GEO file:", geo_file)

# Read GEO file
geo_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiline", "true")
    .option("escape", '"')
    .csv(geo_file)
)

geo_raw.printSchema()

print("Rows:", geo_raw.count())
print("Number of columns:", len(geo_raw.columns))




Latest GEO folder: abfss://bronze@tenderdatalake11.dfs.core.windows.net/GEO/2026-09-20/
Latest GEO file: abfss://bronze@tenderdatalake11.dfs.core.windows.net/GEO/2026-09-20/geo_tenders.csv
root
 |-- source: string (nullable = true)
 |-- content: string (nullable = true)

Rows: 1
Number of columns: 2


In [0]:
geo_raw.show(10, truncate=False)

+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
geo_raw.select("source").show(truncate=False)

content = geo_raw.select("content").first()["content"]

print("Content length:", len(content))
print()
print(content[:5000])

+------+
|source|
+------+
|GEO SA|
+------+

Content length: 13631

عرض المنافسات
            
            
            عرض المنافسات إطلاق تجريبي English عن الجيومكانية مجلس الإدارة تنظيم الجيومكانية نشأة الجيومكانية التوجهات الاستراتيجة الأعمال و الأنشطة الجيوديسيا المسح الأرضي المسح البحري المسح الجوي الاسماء الجغرافية الاستشعار عن بعد اللجان الوطنية اللجنة الوطنية للأسماء الجغرافية اللجنة الاستشارية للبحوث العلمية البحرية الرؤية والرسالة الشراكات البيانات المفتوحة التنظيمات واللوائح التنظيمات الأدلة الوثائق التنظيمية الحوكمة والسياسات سياسات البيانات الجيومكانية السياسات المعايير والمواصفات معايير البيانات الجيومكانية الخدمات و المنتجات المنصة الجيومكانية الوطنية بوابة الخدمات الجيومكانية الرقمية مركز المعرفة المعلومات الجيومكانية منصة المجتمع الجيومكاني المكتبة الرقمية الجيومكانية منصة الأفكار الابتكارية الاستثمار الأنشطة الاقتصادية الجيومكانية مركز الاعمال مركز الاعمال انضم لنا التوظيف المركز الإعلامي الأخبار الأسئلة الشائعة الهيئة العامة للمساحة والمعلومات الجيومكانية عن الهيئة

In [0]:
# This cell counts repeated Arabic labels in the raw GEO webpage content.

print("رقم المنافسة count:", content.count("رقم المنافسة"))
print("الرقم المرجعي count:", content.count("الرقم المرجعي"))
print("رقم المنافسة: count:", content.count("رقم المنافسة:"))

رقم المنافسة count: 8
الرقم المرجعي count: 3
رقم المنافسة: count: 8


In [0]:
# This cell parses the raw GEO webpage text into individual tender sections. A regular expression identifies the beginning of each tender based on the webpage 
# structure, then non-tender header and footer content is removed. Finally, we count and preview the extracted tender sections to verify that the parsing 
# worked correctly

import re

content = geo_raw.select("content").first()["content"]

# The tender section starts with 01, 02, 03 ... followed by the title,
# and then "1 الجهة الحكومية"
pattern = r'(?=\b(?:0[1-9]|1[0-9])\s+.+?\s+1\s+[​\s]*الجهة الحكومية)'

sections = re.split(pattern, content)

# Remove page header / footer pieces that don't contain a tender ID
tender_sections = [
    s.strip()
    for s in sections
    if (
        "رقم المنافسة" in s
        or "الرقم المرجعي" in s
    )
]

print("Tender sections found:", len(tender_sections))

for i, section in enumerate(tender_sections, 1):
    print("\n----------------")
    print("Tender", i)
    print(section[:400])

Tender sections found: 10

----------------
Tender 1
عرض المنافسات
            
            
            عرض المنافسات إطلاق تجريبي English عن الجيومكانية مجلس الإدارة تنظيم الجيومكانية نشأة الجيومكانية التوجهات الاستراتيجة الأعمال و الأنشطة الجيوديسيا المسح الأرضي المسح البحري المسح الجوي الاسماء الجغرافية الاستشعار عن بعد اللجان الوطنية اللجنة الوطنية للأسماء الجغرافية اللجنة الاستشارية للبحوث العلمية البحرية الرؤية والرسالة الشراكات البيانات المف

----------------
Tender 2
02 تشغيل وصيانة أنظمة ورخص الموارد المؤسسية 1 الجهة الحكومية الهيئة العامة للمساحة والمعلومات الجيومكانية 2 المعلومات الأساسية تشغيل وصيانة أنظمة ورخص الموارد المؤسسية رقم المنافسة: 47/412/17 3 قيمة المنافسة 1000.00 4 الغاية من المنافسة يهدف هذا المشروع إلى تشغيل وصيانة أنظمة الموارد المؤسسية بما يضمن
استمرارية الأعمال ورفع كفاءة التشغيل وتعز يز موثوقية الأنظمة وتحقيق
متطلبات الهيئة 5 اخر موعد لإس

----------------
Tender 3
03 تجديد الرخص والبرامج لمشاريع المسح البحري 1 الجهة الحكومية الهيئة العامة للمساحة والمعلو

In [0]:
# This cell tests the extraction of a unique identifier from each GEO tender section.

for i, section in enumerate(tender_sections, 1):

    competition = re.search(
        r'رقم\s+المنافسة\s*:\s*([^\s]+)',
        section
    )

    reference = re.search(
        r'الرقم\s+المرجعي\s*:\s*([^\s]+)',
        section
    )

    tender_id = (
        competition.group(1)
        if competition
        else reference.group(1)
        if reference
        else None
    )

    print(i, "->", tender_id)

1 -> 48/414/1
2 -> 47/412/17
3 -> 47/409/14
4 -> 47/406/11
5 -> 47/403/8
6 -> 47/398/3
7 -> 241039021914
8 -> 241039017961
9 -> 241039017854
10 -> 47/396/1


In [0]:
# This cell searches the complete GEO webpage content for all competition numbers and reference numbers. It counts and displays the extracted identifiers so we 
# can verify that our regular expressions are correctly detecting the tender IDs before building the structured records.

import re

competition_ids = re.findall(
    r'رقم\s+المنافسة\s*:\s*([^\s]+)',
    content
)

reference_ids = re.findall(
    r'الرقم\s+المرجعي\s*:\s*([^\s]+)',
    content
)

print("Competition IDs:", len(competition_ids))
for x in competition_ids:
    print(x)

print("\nReference IDs:", len(reference_ids))
for x in reference_ids:
    print(x)

print("\nTotal IDs:", len(competition_ids) + len(reference_ids))

Competition IDs: 8
48/414/1
47/412/17
47/409/14
47/406/11
47/403/8
47/397/2
47/398/3
47/396/1

Reference IDs: 3
241039021914
241039017961
241039017854

Total IDs: 11


In [0]:
# Data checking

all_ids = competition_ids + reference_ids

print("IDs missing from tender_sections:")

for tender_id in all_ids:
    found = any(tender_id in section for section in tender_sections)

    if not found:
        print("MISSING:", tender_id)

IDs missing from tender_sections:


In [0]:
# ============================================================
# STEP: Create structured GEO tender records
# Purpose:
# Convert each parsed GEO tender section into a structured
# record with one tender ID and its complete source text.
# ============================================================

import re

# Match either type of tender identifier
id_pattern = (
    r'(?:رقم\s+المنافسة|الرقم\s+المرجعي)\s*:\s*([^\s]+)'
)

matches = list(re.finditer(id_pattern, content))

print("IDs found:", len(matches))

geo_records = []

for i, match in enumerate(matches):
    tender_id = match.group(1)

    # Start a little before the ID so we retain the title/agency
    if i == 0:
        start = 0
    else:
        previous_end = matches[i - 1].end()
        start = previous_end

    end = matches[i + 1].start() if i + 1 < len(matches) else len(content)

    geo_records.append({
        "tender_id": tender_id,
        "text": content[start:end].strip()
    })

print("Records created:", len(geo_records))

for i, r in enumerate(geo_records, 1):
    print(i, "->", r["tender_id"])



IDs found: 11
Records created: 11
1 -> 48/414/1
2 -> 47/412/17
3 -> 47/409/14
4 -> 47/406/11
5 -> 47/403/8
6 -> 47/397/2
7 -> 47/398/3
8 -> 241039021914
9 -> 241039017961
10 -> 241039017854
11 -> 47/396/1


In [0]:
for i, r in enumerate(geo_records, 1):
    print("\n==============================")
    print("RECORD:", i)
    print("ID:", r["tender_id"])
    print(r["text"][:500])


RECORD: 1
ID: 48/414/1
عرض المنافسات
            
            
            عرض المنافسات إطلاق تجريبي English عن الجيومكانية مجلس الإدارة تنظيم الجيومكانية نشأة الجيومكانية التوجهات الاستراتيجة الأعمال و الأنشطة الجيوديسيا المسح الأرضي المسح البحري المسح الجوي الاسماء الجغرافية الاستشعار عن بعد اللجان الوطنية اللجنة الوطنية للأسماء الجغرافية اللجنة الاستشارية للبحوث العلمية البحرية الرؤية والرسالة الشراكات البيانات المفتوحة التنظيمات واللوائح التنظيمات الأدلة الوثائق التنظيمية الحوكمة والسياسات سياسات البيانات الجيومك

RECORD: 2
ID: 47/412/17
3 قيمة المنافسة 2000.00 4 الغاية من المنافسة تقديم خدمات التشغيل والصيانة والنظافة للحفاظ على ممتلكات الهيئة وتوفير
بيئة عمل آمنة وصحية لمنسوبي الهيئة 5 اخر موعد لإستلام استفسارات الموردين و إضافة الملحقات التاريخ: 27/01/1448 الموافق: 12/07/2026 6 آخر موعد لإستلام العروض التاريخ: 11/02/1448 الموافق: 25/07/2026 7 تاريخ و وقت فتح العروض التاريخ: 12/02/1448 الموافق: 26/07/2026 8 فترة التوقف 5 9 التاريخ المتوقع للترسية التاريخ: 04/03/1448 الموافق: 17

In [0]:
# ============================================================
# STEP: Parse structured fields from GEO tender sections
# Purpose:
# Extract the tender ID, name, description, document price,
# closing date, and opening date from each GEO tender section.
# # ============================================================

import re
from pyspark.sql import Row

# Normalize whitespace first
clean_content = re.sub(r'\s+', ' ', content).strip()

# Find all 11 tender IDs
id_pattern = r'(?:رقم\s+المنافسة|الرقم\s+المرجعي)\s*:\s*([^\s]+)'
matches = list(re.finditer(id_pattern, clean_content))

parsed_records = []

for i, match in enumerate(matches):

    tender_id = match.group(1)

    # Use midpoint between IDs to isolate each tender.
    # Enough text before the ID is needed because title/agency come before it.
    previous_pos = matches[i - 1].end() if i > 0 else 0
    next_pos = matches[i + 1].start() if i + 1 < len(matches) else len(clean_content)

    block = clean_content[previous_pos:next_pos]

    # -------------------------
    # Tender name
    # -------------------------
    # Usually directly before رقم المنافسة / الرقم المرجعي
    title_match = re.search(
        r'2\s+(?:المعلومات الأساسية|​المعلومات الأساسية)\s+(.+?)'
        r'(?:رقم\s+المنافسة|الرقم\s+المنافسة|الرقم\s+المرجعي)',
        block
    )

    title = title_match.group(1).strip() if title_match else None

    # Remove labels occasionally included in title
    if title:
        title = re.sub(
            r'^(?:اسم|إسم|إﺳﻢ)\s+المنافسة\s*:\s*',
            '',
            title
        ).strip()

    # -------------------------
    # Price
    # -------------------------
    price_match = re.search(
        r'قيمة\s+المنافسة\s+([\d,.]+)',
        block
    )

    price = (
        float(price_match.group(1).replace(",", ""))
        if price_match
        else None
    )

    # -------------------------
    # Description / purpose
    # -------------------------
    purpose_match = re.search(
        r'(?:الغاية|الغرض)\s+من\s+المنافسة\s+(.+?)'
        r'(?=\s+[45]\s+(?:اخر|آخر))',
        block
    )

    description = (
        purpose_match.group(1).strip()
        if purpose_match
        else None
    )

    # -------------------------
    # Closing date
    # -------------------------
    closing_match = re.search(
        r'(?:آخر|اخر)\s+موعد\s+(?:لإستلام|لتقديم)\s+العروض'
        r'.*?الموافق\s*:?\s*(\d{2}/\d{2}/\d{4})',
        block
    )

    closing_date = (
        closing_match.group(1)
        if closing_match
        else None
    )

    # -------------------------
    # Opening date
    # -------------------------
    opening_match = re.search(
        r'تاريخ(?:\s+و\s+وقت)?\s+فتح\s+العروض'
        r'.*?الموافق\s*:?\s*(\d{2}/\d{2}/\d{4})',
        block
    )

    opening_date = (
        opening_match.group(1)
        if opening_match
        else None
    )

    parsed_records.append(
        Row(
            source_tender_id=tender_id,
            tender_name=title,
            description=description,
            document_price=price,
            closing_date_raw=closing_date,
            opening_date_raw=opening_date
        )
    )

geo_parsed = spark.createDataFrame(parsed_records)

geo_parsed.show(20, truncate=False)



+----------------+-------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+----------------+----------------+
|source_tender_id|tender_name                                                                                      |description                                                                                                                                                                                                                                                                                                           |document_price|closing_date_raw|opening_date_raw|
+----------------+----------------------------

In [0]:
print("Rows:", geo_parsed.count())

geo_parsed.select(
    "source_tender_id",
    "tender_name",
    "document_price",
    "closing_date_raw",
    "opening_date_raw"
).show(20, truncate=False)

Rows: 11
+----------------+-------------------------------------------------------------------------------------------------+--------------+----------------+----------------+
|source_tender_id|tender_name                                                                                      |document_price|closing_date_raw|opening_date_raw|
+----------------+-------------------------------------------------------------------------------------------------+--------------+----------------+----------------+
|48/414/1        |النظافة و التشغيل والصيانة لمباني الهيئة العامة للمساحة والمعلومات الجيومكانية                   |2000.0        |25/07/2026      |26/07/2026      |
|47/412/17       |تشغيل وصيانة أنظمة ورخص الموارد المؤسسية                                                         |2000.0        |25/07/2026      |26/07/2026      |
|47/409/14       |تجديد الرخص والبرامج لمشاريع المسح البحري                                                        |1000.0        |05/04/2026      |05/04/2026   

In [0]:
# This is a data-quality validation cell.


from pyspark.sql.functions import col

for c in [
    "source_tender_id",
    "tender_name",
    "description",
    "document_price",
    "closing_date_raw",
    "opening_date_raw"
]:
    print(
        c,
        geo_parsed.filter(col(c).isNull()).count()
    )

source_tender_id 0
tender_name 0
description 0
document_price 2
closing_date_raw 1
opening_date_raw 1


In [0]:
# This cell identifies the specific GEO tenders that have missing document prices, closing dates, or opening dates after parsing. It helps us investigate 
# whether the values are actually missing from the source or whether the regex extraction needs adjustment.

geo_parsed.filter(
    col("document_price").isNull() |
    col("closing_date_raw").isNull() |
    col("opening_date_raw").isNull()
).select(
    "source_tender_id",
    "tender_name",
    "document_price",
    "closing_date_raw",
    "opening_date_raw"
).show(truncate=False)

+----------------+-------------------------------------------------------------------------------------------------+--------------+----------------+----------------+
|source_tender_id|tender_name                                                                                      |document_price|closing_date_raw|opening_date_raw|
+----------------+-------------------------------------------------------------------------------------------------+--------------+----------------+----------------+
|241039017961    |توريد وتفعيل خدمات بيانات الانترنت المتنقلة للشبكة الوطنية لمحطات الرصد المستمر (المرحلة الثالثة)|NULL          |16/11/2024      |17/11/2024      |
|241039017854    |توريد وتفعيل خدمات بيانات الانترنت المتنقلة للشبكة الوطنية لمحطات الرصد المستمر (المرحلة الثانية)|NULL          |NULL            |NULL            |
+----------------+-------------------------------------------------------------------------------------------------+--------------+----------------+----------------+



In [0]:
# checking the data

missing_date_ids = [
    r["source_tender_id"]
    for r in geo_parsed.filter(
        col("closing_date_raw").isNull() |
        col("opening_date_raw").isNull()
    ).select("source_tender_id").collect()
]

print(missing_date_ids)

for tender_id in missing_date_ids:
    for r in geo_records:
        if r["tender_id"] == tender_id:
            print("\nID:", tender_id)
            print(r["text"])

['241039017854']

ID: 241039017854
3 الغرض من المنافسة توريد وتفعيل خدمات بيانات الانترنت المتنقلة لمحطات الرصد المستمر 4 آخر موعد لإستلام الإستفسارات التاريخ : 29/04/1446
                                        ​ الموافق​ : 01/11/2024 5 آخر موعد لتقديم العروض التاريخ : 10/05/1446 الموافق​ : 12/11/2024 6 تاريخ فتح العروض التاريخ : 10/05/1446 الموافق​ : 12/11/2024 7 تاريخ فحص العروض يحدد لاحقًا 8 عنوان تسليم عينات للمورد لا يوجد 9 مكان فتح العروض الكتروني عبر منصة اعتماد 10 مكان التنفيذ داخل المملكة 11 مجال التصنيف - 10 توريد وتفعيل خدمات بيانات الانترنت المتنقلة للشبكة الوطنية لمحطات الرصد المستمر (المرحلة الثانية) 1 ​الجهة الحكومية ​ الهيئة العامة للمساحة والمعلومات الجيومكانية 2 ​المعلومات الأساسية اسم المنافسة: توريد وتفعيل خدمات بيانات الانترنت المتنقلة للشبكة الوطنية لمحطات الرصد المستمر (المرحلة الثانية) الرقم المرجعي : 241039017854 3 الغرض من المنافسة توريد وتفعيل خدمات بيانات الانترنت المتنقلة لمحطات الرصد المستمر 4 آخر موعد لإستلام الإستفسارات التاريخ : 29/04/1446
            

In [0]:
# Remove invisible Unicode characters that appear in the webpage
clean_content = re.sub(r'[\u200b\u200c\u200d\u2060\ufeff]', '', content)
clean_content = re.sub(r'\s+', ' ', clean_content).strip()

# Rebuild matches after cleaning
id_pattern = r'(?:رقم\s+المنافسة|الرقم\s+المرجعي)\s*:\s*([^\s]+)'
matches = list(re.finditer(id_pattern, clean_content))

print("IDs after cleaning:", len(matches))

IDs after cleaning: 11


In [0]:
# ============================================================
# STEP: Apply known GEO source correction
# Purpose:
# Restore the closing and opening dates for tender 241039017854.
# These dates were available in the Bronze source content but
# were not captured by the standard GEO regex parser.
# ============================================================

from pyspark.sql.functions import when, lit, col

geo_parsed = (
    geo_parsed
    .withColumn(
        "closing_date_raw",
        when(
            col("source_tender_id") == "241039017854",
            lit("12/11/2024")
        ).otherwise(col("closing_date_raw"))
    )
    .withColumn(
        "opening_date_raw",
        when(
            col("source_tender_id") == "241039017854",
            lit("12/11/2024")
        ).otherwise(col("opening_date_raw"))
    )
)

In [0]:
for c in [
    "source_tender_id",
    "tender_name",
    "description",
    "document_price",
    "closing_date_raw",
    "opening_date_raw"
]:
    print(
        c,
        geo_parsed.filter(col(c).isNull()).count()
    )

source_tender_id 0
tender_name 0
description 0
document_price 2
closing_date_raw 0
opening_date_raw 0


In [0]:
# This cell verifies the known GEO date correction by displaying the affected tender after the update and confirming that its closing and opening dates were populated correctly.

geo_parsed.filter(
    col("source_tender_id") == "241039017854"
).show(truncate=False)

+----------------+-------------------------------------------------------------------------------------------------+----------------------------------------------------------------+--------------+----------------+----------------+
|source_tender_id|tender_name                                                                                      |description                                                     |document_price|closing_date_raw|opening_date_raw|
+----------------+-------------------------------------------------------------------------------------------------+----------------------------------------------------------------+--------------+----------------+----------------+
|241039017854    |توريد وتفعيل خدمات بيانات الانترنت المتنقلة للشبكة الوطنية لمحطات الرصد المستمر (المرحلة الثانية)|توريد وتفعيل خدمات بيانات الانترنت المتنقلة لمحطات الرصد المستمر|NULL          |12/11/2024      |12/11/2024      |
+----------------+----------------------------------------------------------

In [0]:
# This cell converts the parsed GEO tenders into our common 24-column Silver schema.

from pyspark.sql.functions import (
    col, lit, trim, to_date, sha2, concat_ws
)

geo_silver = (
    geo_parsed

    .withColumn(
        "source",
        lit("geo_resources")
    )

    # ID / reference number
    .withColumn(
        "reference_number",
        col("source_tender_id")
    )

    # No English title available
    .withColumn(
        "tender_name_en",
        lit(None).cast("string")
    )

    # Agency is the same for this source
    .withColumn(
        "agency_name",
        lit("الهيئة العامة للمساحة والمعلومات الجيومكانية")
    )

    # Tender type is not explicitly provided
    .withColumn(
        "source_tender_type",
        lit(None).cast("string")
    )
    .withColumn(
        "tender_type",
        lit(None).cast("string")
    )

    # Status is not explicitly provided
    .withColumn(
        "source_status",
        lit(None).cast("string")
    )
    .withColumn(
        "status",
        lit(None).cast("string")
    )

    # No publication date available
    .withColumn(
        "publish_date",
        lit(None).cast("date")
    )

    # Convert DD/MM/YYYY
    .withColumn(
        "closing_date",
        to_date(col("closing_date_raw"), "dd/MM/yyyy")
    )
    .withColumn(
        "opening_date",
        to_date(col("opening_date_raw"), "dd/MM/yyyy")
    )

    # No structured categories/regions extracted
    .withColumn(
        "categories",
        lit(None).cast("array<string>")
    )
    .withColumn(
        "regions",
        lit(None).cast("array<string>")
    )

    # No estimated contract value
    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )

    # Competition/document price
    .withColumn(
        "document_price",
        col("document_price").cast("decimal(18,2)")
    )

    .withColumn(
        "currency",
        lit("SAR")
    )

    # We don't have individual URLs
    .withColumn(
        "tender_url",
        lit(None).cast("string")
    )

    # Source doesn't explicitly classify records as tech
    .withColumn(
        "is_tech",
        lit(None).cast("boolean")
    )

    .withColumn(
        "ingestion_timestamp",
        lit(None).cast("timestamp")
    )

    # Deterministic key
    .withColumn(
        "tender_key",
        sha2(
            concat_ws(
                "|",
                col("source"),
                col("source_tender_id")
            ),
            256
        )
    )

    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# Data quality check

geo_silver.printSchema()

print("Rows:", geo_silver.count())
print("Columns:", len(geo_silver.columns))

print(
    "Unique tender keys:",
    geo_silver.select("tender_key").distinct().count()
)

print(
    "Null tender keys:",
    geo_silver.filter(col("tender_key").isNull()).count()
)

print(
    "Null closing dates:",
    geo_silver.filter(col("closing_date").isNull()).count()
)

print(
    "Null opening dates:",
    geo_silver.filter(col("opening_date").isNull()).count()
)

print(
    "Null document prices:",
    geo_silver.filter(col("document_price").isNull()).count()
)

root
 |-- tender_key: string (nullable = true)
 |-- source: string (nullable = false)
 |-- source_tender_id: string (nullable = true)
 |-- reference_number: string (nullable = true)
 |-- tender_name: string (nullable = true)
 |-- tender_name_en: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agency_name: string (nullable = false)
 |-- source_tender_type: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- source_status: string (nullable = true)
 |-- status: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- closing_date: date (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- regions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- estimated_value_min: decimal(18,2) (nullable = true)
 |-- estimated_value_max: decimal(18,2) (nullable = true)
 |-- document_price: decimal(18,2) (nullable = tr

In [0]:
geo_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/"
    "tenders/geo_resources"
)

merge_to_silver(
    geo_silver,
    geo_silver_path,
    "Geo Resources"
)


Geo Resources: Silver MERGE completed.


In [0]:
# Validate the Saved GEO Silver Table

geo_check = (
    spark.read
    .format("delta")
    .load(geo_silver_path)
)

print("Rows:", geo_check.count())
print("Columns:", len(geo_check.columns))
print(
    "Unique tender keys:",
    geo_check.select("tender_key").distinct().count()
)

geo_check.select(
    "source_tender_id",
    "tender_name",
    "closing_date",
    "opening_date",
    "document_price"
).show(20, truncate=False)

Rows: 11
Columns: 24
Unique tender keys: 11
+----------------+-------------------------------------------------------------------------------------------------+------------+------------+--------------+
|source_tender_id|tender_name                                                                                      |closing_date|opening_date|document_price|
+----------------+-------------------------------------------------------------------------------------------------+------------+------------+--------------+
|47/403/8        |إﺳﻢ اﻟﻤﻨﺎﻓﺴﺔ: ﺗﻮرﻳﺪ وﺗﺮﻛﻴﺐ وﻓﻚ اﻟﺘﻜﻴﻴﻒ ﻟﻤﺒﺎﻧﻲ اﻟﻬﻴﺌﺔ اﻟﻌﺎﻣﺔ ﻟﻠﻤﺴﺎﺣﺔ واﻟﻤﻌﻠﻮﻣﺎت اﻟﺠﻴﻮﻣﻜﺎﻧﻴﺔ       |2025-11-23  |2025-11-23  |1000.00       |
|241039017961    |توريد وتفعيل خدمات بيانات الانترنت المتنقلة للشبكة الوطنية لمحطات الرصد المستمر (المرحلة الثالثة)|2024-11-16  |2024-11-17  |NULL          |
|47/398/3        |صيانة الشبكة الوطنية لمحطات الرصد المستمر CORS-K                                                 |2025-08-12  |2025-08-12  |2000.00       |
|2410390

## Source NO. 10: SAR Bronze -> Silver

In [0]:

# SAR Bronze -> Silver
# ============================================================
# STEP: Load the latest SAR Bronze file
# Purpose:
# Find the latest SAR ingestion folder and newest CSV file,
# then load and inspect the raw data.
# ============================================================

sar_path = bronze_path + "SAR/"

# Get all date folders under SAR
sar_folders = dbutils.fs.ls(sar_path)

# Choose latest folder
latest_sar_folder = sorted(
    [f.path for f in sar_folders if f.isDir()],
    reverse=True
)[0]

print("Latest SAR folder:", latest_sar_folder)

# Get CSV files from latest folder
sar_files = [
    f for f in dbutils.fs.ls(latest_sar_folder)
    if not f.isDir() and f.name.lower().endswith(".csv")
]

# Choose latest CSV file
latest_sar_file = max(
    sar_files,
    key=lambda f: f.modificationTime
)

sar_file = latest_sar_file.path

print("Latest SAR file:", sar_file)

# Read latest SAR CSV
sar_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiline", "true")
    .option("escape", '"')
    .csv(sar_file)
)

sar_raw.printSchema()

print("Rows:", sar_raw.count())
print("Number of columns:", len(sar_raw.columns))

for c in sar_raw.columns:
    print(c)



Latest SAR folder: abfss://bronze@tenderdatalake11.dfs.core.windows.net/SAR/2026-09-20/
Latest SAR file: abfss://bronze@tenderdatalake11.dfs.core.windows.net/SAR/2026-09-20/sar_tenders.csv
root
 |-- tender_id: integer (nullable = true)
 |-- tender_number: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- published_date: string (nullable = true)
 |-- closing_date: string (nullable = true)
 |-- source: string (nullable = true)

Rows: 3
Number of columns: 7
tender_id
tender_number
title
description
published_date
closing_date
source


In [0]:
# This cell profiles the raw SAR data before transformation.

sar_raw.show(10, truncate=False)

print("Total SAR tenders:", sar_raw.count())

print(
    "Unique tender IDs:",
    sar_raw.select("tender_id").distinct().count()
)

print(
    "Unique tender numbers:",
    sar_raw.select("tender_number").distinct().count()
)

for c in [
    "tender_id",
    "tender_number",
    "title",
    "description",
    "published_date",
    "closing_date",
    "source"
]:
    print(
        c,
        sar_raw.filter(col(c).isNull()).count()
    )

+---------+-------------+-----------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+------------+------+
|tender_id|tender_number|title                                                                        |description                                                                                                                                                                                                                                           

In [0]:
sar_raw.select(
    "tender_id",
    "tender_number",
    "published_date",
    "closing_date"
).show(10, truncate=False)

+---------+-------------+--------------+------------+
|tender_id|tender_number|published_date|closing_date|
+---------+-------------+--------------+------------+
|3149     |790770       |17-Aug-2026   |04-Dec-2026 |
|3169     |795756       |31-Aug-2026   |27-Sep-2026 |
|3189     |799765       |10-Sep-2026   |18-Oct-2026 |
+---------+-------------+--------------+------------+



In [0]:
# This cell transforms the raw SAR tender data into our common Silver schema

from pyspark.sql.functions import (
    col, trim, upper, to_date, lit, sha2, concat_ws
)

sar_silver = (
    sar_raw

    .withColumn("source_tender_id", col("tender_id").cast("string"))
    .withColumn("reference_number", col("tender_number").cast("string"))

    .withColumn("tender_name", trim(col("title")))
    .withColumn("tender_name_en", lit(None).cast("string"))

    # description already exists
    .withColumn("description", trim(col("description")))

    # SAR = Saudi Arabia Railways
    .withColumn("agency_name", lit("Saudi Arabia Railways (SAR)"))

    .withColumn("source_tender_type", lit(None).cast("string"))
    .withColumn("tender_type", lit(None).cast("string"))

    .withColumn("source_status", lit(None).cast("string"))
    .withColumn("status", lit(None).cast("string"))

    # Dates: 17-Aug-2026
    .withColumn(
        "publish_date",
        to_date(col("published_date"), "dd-MMM-yyyy")
    )
    .withColumn(
        "closing_date",
        to_date(col("closing_date"), "dd-MMM-yyyy")
    )
    .withColumn(
        "opening_date",
        lit(None).cast("date")
    )

    .withColumn("categories", lit(None).cast("array<string>"))
    .withColumn("regions", lit(None).cast("array<string>"))

    .withColumn(
        "estimated_value_min",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_value_max",
        lit(None).cast("decimal(18,2)")
    )
    .withColumn(
        "document_price",
        lit(None).cast("decimal(18,2)")
    )

    .withColumn("currency", lit(None).cast("string"))
    .withColumn("tender_url", lit(None).cast("string"))
    .withColumn("is_tech", lit(None).cast("boolean"))
    .withColumn("ingestion_timestamp", lit(None).cast("timestamp"))

    # Normalize source name
    .withColumn("source", lit("sar"))

    .withColumn(
        "tender_key",
        sha2(
            concat_ws("|", col("source"), col("source_tender_id")),
            256
        )
    )

    .select(
        "tender_key",
        "source",
        "source_tender_id",
        "reference_number",
        "tender_name",
        "tender_name_en",
        "description",
        "agency_name",
        "source_tender_type",
        "tender_type",
        "source_status",
        "status",
        "publish_date",
        "closing_date",
        "opening_date",
        "categories",
        "regions",
        "estimated_value_min",
        "estimated_value_max",
        "document_price",
        "currency",
        "tender_url",
        "is_tech",
        "ingestion_timestamp"
    )
)

In [0]:
# Validate SAR Silver Transformation

print("Rows:", sar_silver.count())
print("Columns:", len(sar_silver.columns))

print(
    "Unique tender keys:",
    sar_silver.select("tender_key").distinct().count()
)

print(
    "Null tender keys:",
    sar_silver.filter(col("tender_key").isNull()).count()
)

print(
    "Null publish dates:",
    sar_silver.filter(col("publish_date").isNull()).count()
)

print(
    "Null closing dates:",
    sar_silver.filter(col("closing_date").isNull()).count()
)

sar_silver.select(
    "source_tender_id",
    "reference_number",
    "tender_name",
    "publish_date",
    "closing_date"
).show(truncate=False)

Rows: 3
Columns: 24
Unique tender keys: 3
Null tender keys: 0
Null publish dates: 0
Null closing dates: 0
+----------------+----------------+-----------------------------------------------------------------------------+------------+------------+
|source_tender_id|reference_number|tender_name                                                                  |publish_date|closing_date|
+----------------+----------------+-----------------------------------------------------------------------------+------------+------------+
|3149            |790770          |SAR Freight Locomotive - Technical Service Support & Supply Agreement (TSSSA)|2026-08-17  |2026-12-04  |
|3169            |795756          |Rerailing Equipment                                                          |2026-08-31  |2026-09-27  |
|3189            |799765          |Design Management and Review Consultancy for the GCC Railway Design          |2026-09-10  |2026-10-18  |
+----------------+----------------+-------------------

In [0]:
sar_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/"
    "tenders/sar"
)

merge_to_silver(
    sar_silver,
    sar_silver_path,
    "SAR"
)

SAR: Silver MERGE completed.


In [0]:
# This cell reloads the SAR Silver Delta table after the merge and verifies that the persisted data has the expected 24-column schema and unique tender keys.

sar_check = (
    spark.read
    .format("delta")
    .load(sar_silver_path)
)

print("Rows:", sar_check.count())
print("Columns:", len(sar_check.columns))

print(
    "Unique tender keys:",
    sar_check.select("tender_key").distinct().count()
)

Rows: 3
Columns: 24
Unique tender keys: 3


## tenders_unified

In [0]:
# Load All Source-Specific Silver Tables
# ============================================================
# STEP: Load all source-specific Silver Delta tables
# Purpose:
# Read each standardized tender source from the Silver layer
# and store the DataFrames for unified processing.
# ============================================================

silver_base = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/tenders/"
)

sources = [
    "nupco",
    "stc",
    "etimad",
    "cst",
    "tanafos",
    "taqeem",
    "tahakom",
    "forsah",
    "geo_resources",
    "sar"
]

silver_dfs = {}

for source in sources:
    path = silver_base + source

    df = spark.read.format("delta").load(path)
    silver_dfs[source] = df

    print(source, "->", df.count())

nupco -> 154
stc -> 2506
etimad -> 8257
cst -> 33
tanafos -> 30
taqeem -> 6
tahakom -> 160
forsah -> 1001
geo_resources -> 11
sar -> 3


In [0]:
# ============================================================
# STEP: Validate Silver schema consistency
# Purpose:
# Confirm that every source-specific Silver dataset uses the
# same 24-column structure and column order before unification.
# ============================================================

expected_columns = silver_dfs["nupco"].columns

for source, df in silver_dfs.items():
    same_schema = df.columns == expected_columns

    print(
        source,
        "columns:", len(df.columns),
        "same schema:", same_schema
    )

nupco columns: 24 same schema: True
stc columns: 24 same schema: True
etimad columns: 24 same schema: True
cst columns: 24 same schema: True
tanafos columns: 24 same schema: True
taqeem columns: 24 same schema: True
tahakom columns: 24 same schema: True
forsah columns: 24 same schema: True
geo_resources columns: 24 same schema: True
sar columns: 24 same schema: True


In [0]:
# ============================================================
# STEP: Combine all source-specific Silver datasets
# Purpose:
# Union the standardized Silver DataFrames into one unified
# tender dataset before cross-source deduplication.
# ============================================================

from functools import reduce

tenders_unified = reduce(
    lambda df1, df2: df1.unionByName(df2),
    silver_dfs.values()
)

print("Unified rows:", tenders_unified.count())
print("Unified columns:", len(tenders_unified.columns))

tenders_unified.groupBy("source") \
    .count() \
    .orderBy("source") \
    .show()

Unified rows: 12161
Unified columns: 24
+-------------+-----+
|       source|count|
+-------------+-----+
|          cst|   33|
|       etimad| 8257|
|       forsah| 1001|
|geo_resources|   11|
|        nupco|  154|
|          sar|    3|
|          stc| 2506|
|      tahakom|  160|
|      tanafos|   30|
|       taqeem|    6|
+-------------+-----+



In [0]:
# ============================================================
# STEP: Analyze duplicate reference numbers
# Purpose:
# Identify reference numbers that appear multiple times in the
# unified Silver data and determine whether they occur across
# multiple tender sources.
# ============================================================


from pyspark.sql.functions import (
    col, count, countDistinct
)

# How many records have a reference number?
print(
    "Rows with reference number:",
    tenders_unified.filter(
        col("reference_number").isNotNull()
    ).count()
)

print(
    "Distinct reference numbers:",
    tenders_unified.filter(
        col("reference_number").isNotNull()
    ).select("reference_number").distinct().count()
)

# Reference numbers appearing more than once
duplicate_refs = (
    tenders_unified
    .filter(col("reference_number").isNotNull())
    .groupBy("reference_number")
    .agg(
        count("*").alias("record_count"),
        countDistinct("source").alias("source_count")
    )
    .filter(col("record_count") > 1)
    .orderBy(col("record_count").desc())
)

print(
    "Duplicated reference numbers:",
    duplicate_refs.count()
)

duplicate_refs.show(30, truncate=False)

Rows with reference number: 11004
Distinct reference numbers: 10143
Duplicated reference numbers: 860
+----------------+------------+------------+
|reference_number|record_count|source_count|
+----------------+------------+------------+
|102001368       |3           |1           |
|260739002047    |2           |2           |
|260839006192    |2           |2           |
|260739009189    |2           |2           |
|260839008195    |2           |2           |
|260839009592    |2           |2           |
|260739007683    |2           |2           |
|260739003985    |2           |2           |
|260839004060    |2           |2           |
|260839001075    |2           |2           |
|260639008856    |2           |2           |
|260839010534    |2           |2           |
|260839006586    |2           |2           |
|260939000385    |2           |2           |
|260639004491    |2           |2           |
|2000002534      |2           |2           |
|260439009421    |2           |2           

In [0]:
# ============================================================
# STEP: Identify cross-source duplicate references
# Purpose:
# Isolate reference numbers that appear in more than one
# tender source for further deduplication analysis.
# ============================================================


cross_source_duplicates = (
    duplicate_refs
    .filter(col("source_count") > 1)
)

print(
    "Cross-source duplicate references:",
    cross_source_duplicates.count()
)

cross_source_duplicates.show(30, truncate=False)

Cross-source duplicate references: 847
+----------------+------------+------------+
|reference_number|record_count|source_count|
+----------------+------------+------------+
|260739010558    |2           |2           |
|260739002047    |2           |2           |
|260639004491    |2           |2           |
|260739009189    |2           |2           |
|260839008195    |2           |2           |
|260839009592    |2           |2           |
|260739007683    |2           |2           |
|260739003985    |2           |2           |
|260839004060    |2           |2           |
|260839001075    |2           |2           |
|260639008856    |2           |2           |
|260839010534    |2           |2           |
|260839006586    |2           |2           |
|260939000385    |2           |2           |
|260839006192    |2           |2           |
|2000002534      |2           |2           |
|260439009421    |2           |2           |
|260639008771    |2           |2           |
|260639008916   

In [0]:
# ============================================================
# STEP: Analyze cross-source duplicate patterns
# Purpose:
# Identify which combinations of tender sources share the
# same reference numbers before defining deduplication rules.
# ============================================================


from pyspark.sql.functions import collect_set, size, sort_array

cross_source_detail = (
    tenders_unified
    .filter(col("reference_number").isNotNull())
    .groupBy("reference_number")
    .agg(
        count("*").alias("record_count"),
        sort_array(collect_set("source")).alias("sources")
    )
    .withColumn("source_count", size(col("sources")))
    .filter(col("source_count") > 1)
)

print(
    "Cross-source duplicate references:",
    cross_source_detail.count()
)

cross_source_detail.groupBy("sources") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(30, truncate=False)

Cross-source duplicate references: 847
+-------------+-----+
|sources      |count|
+-------------+-----+
|[etimad, stc]|847  |
+-------------+-----+



In [0]:
# ============================================================
# STEP: Inspect cross-source duplicate tender records
# Purpose:
# Retrieve the actual tender records associated with duplicate
# reference numbers so their source, title, agency, dates,
# status, and URL can be compared before deduplication.
# ============================================================

duplicate_records = (
    tenders_unified.alias("t")
    .join(
        cross_source_detail.select("reference_number"),
        on="reference_number",
        how="inner"
    )
    .select(
        "reference_number",
        "source",
        "source_tender_id",
        "tender_name",
        "agency_name",
        "publish_date",
        "closing_date",
        "status",
        "tender_url"
    )
    .orderBy("reference_number", "source")
)

duplicate_records.show(40, truncate=False)

+----------------+------+----------------+----------------------------------------+----------------------------------------+------------+------------+------+---------------------------------------------------------------------------------------------+
|reference_number|source|source_tender_id|tender_name                             |agency_name                             |publish_date|closing_date|status|tender_url                                                                                   |
+----------------+------+----------------+----------------------------------------+----------------------------------------+------------+------------+------+---------------------------------------------------------------------------------------------+
|2000000878      |etimad|2000000878      |عمليات تقنية المعلومات                  |20250901AL-AHSA DEVELOPMENT AUTHORITY (A|2025-10-20  |2025-12-23  |معتمدة|NULL                                                                                   

In [0]:
# ============================================================
# STEP: Analyze same-source duplicate references
# Purpose:
# Identify reference numbers that appear multiple times within
# the same source and determine whether those records have
# different source tender IDs.
# ============================================================

same_source_duplicates = (
    tenders_unified
    .filter(col("reference_number").isNotNull())
    .groupBy("source", "reference_number")
    .agg(
        count("*").alias("record_count"),
        countDistinct("source_tender_id").alias("tender_id_count")
    )
    .filter(col("record_count") > 1)
    .orderBy(col("record_count").desc())
)

print(
    "Same-source duplicate groups:",
    same_source_duplicates.count()
)

same_source_duplicates.show(50, truncate=False)

Same-source duplicate groups: 13
+-------+------------------+------------+---------------+
|source |reference_number  |record_count|tender_id_count|
+-------+------------------+------------+---------------+
|tahakom|102001368         |3           |3              |
|tahakom|102000906         |2           |2              |
|tahakom|102001048         |2           |2              |
|tahakom|102001075         |2           |2              |
|tahakom|102001425         |2           |2              |
|tahakom|10009510          |2           |2              |
|tahakom|102001473         |2           |2              |
|tahakom|102001385         |2           |2              |
|tahakom|102001072         |2           |2              |
|tahakom|102001397         |2           |2              |
|tahakom|101000840         |2           |2              |
|tahakom|10009513-102000958|2           |2              |
|tahakom|102001116         |2           |2              |
+-------+------------------+-----------

In [0]:
# ============================================================
# STEP: Inspect same-source duplicate tender records
# Purpose:
# Retrieve the actual records belonging to repeated reference
# numbers within the same source so their IDs, titles, dates,
# agencies, and statuses can be compared before deduplication.
# ============================================================

same_source_duplicate_records = (
    tenders_unified.alias("t")
    .join(
        same_source_duplicates.alias("d"),
        (col("t.source") == col("d.source")) &
        (col("t.reference_number") == col("d.reference_number")),
        "inner"
    )
    .select(
        col("t.source"),
        col("t.reference_number"),
        col("t.source_tender_id"),
        col("t.tender_name"),
        col("t.agency_name"),
        col("t.publish_date"),
        col("t.closing_date"),
        col("t.status")
    )
    .orderBy("source", "reference_number", "source_tender_id")
)

same_source_duplicate_records.show(50, truncate=False)

+-------+------------------+----------------+-------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------+------+
|source |reference_number  |source_tender_id|tender_name                                                                                                                          |agency_name|publish_date|closing_date|status|
+-------+------------------+----------------+-------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------+------+
|tahakom|10009510          |2243            |مشروع توفير معدات مختبرات \nHW-V&V\n                                                                                                 |TAHAKOM    |2026-03-05  |2026-03-12  |OPEN  |
|tahakom|10009510          |2244            |مشروع توفير معدات مختبرات \nHW-V&V\n                   

In [0]:
# ============================================================
# STEP: Compare STC and Etimad overlap completeness
# Purpose:
# Isolate tenders that appear in both STC and Etimad and
# compare the availability of important fields in each source
# before defining the cross-source merge strategy.
# ============================================================

stc_etimad_overlap = (
    tenders_unified
    .join(
        cross_source_detail
        .filter(
            (col("source_count") == 2) &
            (col("sources") == lit(["etimad", "stc"]))
        )
        .select("reference_number"),
        on="reference_number",
        how="inner"
    )
    .filter(col("source").isin("stc", "etimad"))
)

stc_etimad_overlap.groupBy("source").agg(
    count("*").alias("rows"),
    count("tender_name").alias("has_name"),
    count("agency_name").alias("has_agency"),
    count("publish_date").alias("has_publish_date"),
    count("closing_date").alias("has_closing_date"),
    count("opening_date").alias("has_opening_date"),
    count("categories").alias("has_categories"),
    count("tender_url").alias("has_url"),
    count("status").alias("has_status"),
    count("document_price").alias("has_document_price")
).show(truncate=False)

+------+----+--------+----------+----------------+----------------+----------------+--------------+-------+----------+------------------+
|source|rows|has_name|has_agency|has_publish_date|has_closing_date|has_opening_date|has_categories|has_url|has_status|has_document_price|
+------+----+--------+----------+----------------+----------------+----------------+--------------+-------+----------+------------------+
|stc   |847 |847     |847       |847             |0               |0               |847           |847    |847       |0                 |
|etimad|847 |847     |847       |847             |841             |697             |847           |0      |30        |847               |
+------+----+--------+----------+----------------+----------------+----------------+--------------+-------+----------+------------------+



In [0]:
# ============================================================
# STEP: Merge overlapping STC and Etimad tenders
# Purpose:
# Create one canonical record for tenders found in both
# sources. Prefer Etimad values when available and use STC
# values as fallback to preserve complementary information.
# ============================================================


from pyspark.sql.functions import (
    col, coalesce, lit, sha2, concat_ws
)

stc_overlap = (
    stc_etimad_overlap
    .filter(col("source") == "stc")
    .alias("s")
)

etimad_overlap = (
    stc_etimad_overlap
    .filter(col("source") == "etimad")
    .alias("e")
)

merged_overlap = (
    etimad_overlap
    .join(
        stc_overlap,
        col("e.reference_number") == col("s.reference_number"),
        "inner"
    )

    # New canonical identity
    .select(
        sha2(
            concat_ws(
                "|",
                lit("etimad_stc"),
                col("e.reference_number")
            ),
            256
        ).alias("tender_key"),

        lit("etimad_stc").alias("source"),

        # Etimad ID is the canonical source ID
        col("e.source_tender_id").alias("source_tender_id"),
        col("e.reference_number").alias("reference_number"),

        # Core information
        coalesce(
            col("e.tender_name"),
            col("s.tender_name")
        ).alias("tender_name"),

        coalesce(
            col("e.tender_name_en"),
            col("s.tender_name_en")
        ).alias("tender_name_en"),

        coalesce(
            col("e.description"),
            col("s.description")
        ).alias("description"),

        coalesce(
            col("e.agency_name"),
            col("s.agency_name")
        ).alias("agency_name"),

        # Prefer Etimad tender type
        coalesce(
            col("e.source_tender_type"),
            col("s.source_tender_type")
        ).alias("source_tender_type"),

        coalesce(
            col("e.tender_type"),
            col("s.tender_type")
        ).alias("tender_type"),

        # Keep Etimad status for now
        coalesce(
            col("e.source_status"),
            col("s.source_status")
        ).alias("source_status"),

        coalesce(
            col("e.status"),
            col("s.status")
        ).alias("status"),

        # Dates
        coalesce(
            col("e.publish_date"),
            col("s.publish_date")
        ).alias("publish_date"),

        coalesce(
            col("e.closing_date"),
            col("s.closing_date")
        ).alias("closing_date"),

        coalesce(
            col("e.opening_date"),
            col("s.opening_date")
        ).alias("opening_date"),

        # Prefer Etimad categories
        coalesce(
            col("e.categories"),
            col("s.categories")
        ).alias("categories"),

        coalesce(
            col("e.regions"),
            col("s.regions")
        ).alias("regions"),

        coalesce(
            col("e.estimated_value_min"),
            col("s.estimated_value_min")
        ).alias("estimated_value_min"),

        coalesce(
            col("e.estimated_value_max"),
            col("s.estimated_value_max")
        ).alias("estimated_value_max"),

        # Etimad provides this
        coalesce(
            col("e.document_price"),
            col("s.document_price")
        ).alias("document_price"),

        coalesce(
            col("e.currency"),
            col("s.currency")
        ).alias("currency"),

        # STC provides the URL
        coalesce(
            col("e.tender_url"),
            col("s.tender_url")
        ).alias("tender_url"),

        coalesce(
            col("e.is_tech"),
            col("s.is_tech")
        ).alias("is_tech"),

        coalesce(
            col("e.ingestion_timestamp"),
            col("s.ingestion_timestamp")
        ).alias("ingestion_timestamp")
    )
)

In [0]:
# ============================================================
# STEP: Validate merged STC and Etimad records
# Purpose:
# Confirm that each overlapping reference produces one
# canonical record and check the completeness of important
# fields after combining information from both sources.
# ============================================================

print("Merged rows:", merged_overlap.count())
print("Columns:", len(merged_overlap.columns))

print(
    "Unique reference numbers:",
    merged_overlap.select("reference_number").distinct().count()
)

print(
    "Null URLs:",
    merged_overlap.filter(col("tender_url").isNull()).count()
)

print(
    "Null document prices:",
    merged_overlap.filter(col("document_price").isNull()).count()
)

print(
    "Null closing dates:",
    merged_overlap.filter(col("closing_date").isNull()).count()
)

print(
    "Null opening dates:",
    merged_overlap.filter(col("opening_date").isNull()).count()
)

Merged rows: 847
Columns: 24
Unique reference numbers: 847
Null URLs: 0
Null document prices: 0
Null closing dates: 6
Null opening dates: 150


In [0]:
# ============================================================
# STEP: Prepare STC-Etimad overlap reference list
# Purpose:
# Create a list of reference numbers shared specifically by
# STC and Etimad so the original duplicate records can be
# replaced by their canonical merged versions.
# ============================================================


overlap_refs = (
    cross_source_detail
    .filter(
        (col("source_count") == 2) &
        (col("sources") == lit(["etimad", "stc"]))
    )
    .select("reference_number")
)

In [0]:
# ============================================================
# STEP: Remove original STC-Etimad overlap records
# Purpose:
# Keep all unified Silver records except the original tenders
# whose reference numbers belong to the confirmed STC-Etimad
# overlap. These records will be replaced by merged versions.
# ============================================================

non_overlap_records = (
    tenders_unified.alias("t")
    .join(
        overlap_refs.alias("o"),
        col("t.reference_number") == col("o.reference_number"),
        "left_anti"
    )
)

In [0]:
# ============================================================
# STEP: Build the deduplicated unified Silver dataset
# Purpose:
# Combine all non-overlapping tender records with the
# canonical merged STC-Etimad records to produce the final
# deduplicated unified Silver DataFrame.
# ============================================================

tenders_unified_dedup = (
    non_overlap_records
    .unionByName(merged_overlap)
)

In [0]:
# ============================================================
# STEP: Validate the deduplicated unified Silver dataset
# Purpose:
# Confirm the effect of the STC-Etimad merge and verify that
# the final unified dataset preserves the 24-column schema
# with unique and non-null tender keys.
# ============================================================

print("Before merge:", tenders_unified.count())
print("Non-overlap records:", non_overlap_records.count())
print("Merged records:", merged_overlap.count())
print("Final records:", tenders_unified_dedup.count())

print("Columns:", len(tenders_unified_dedup.columns))

print(
    "Unique tender keys:",
    tenders_unified_dedup.select("tender_key").distinct().count()
)

print(
    "Null tender keys:",
    tenders_unified_dedup.filter(
        col("tender_key").isNull()
    ).count()
)

Before merge: 12161
Non-overlap records: 10467
Merged records: 847
Final records: 11314
Columns: 24
Unique tender keys: 11314
Null tender keys: 0


In [0]:
# ============================================================
# STEP: Validate final source distribution
# Purpose:
# Count the final unified Silver records by source after
# STC-Etimad deduplication and confirm that merged overlap
# records are represented by the new "etimad_stc" source.
# ============================================================

tenders_unified_dedup.groupBy("source") \
    .count() \
    .orderBy("source") \
    .show()
    

+-------------+-----+
|       source|count|
+-------------+-----+
|          cst|   33|
|       etimad| 7410|
|   etimad_stc|  847|
|       forsah| 1001|
|geo_resources|   11|
|        nupco|  154|
|          sar|    3|
|          stc| 1659|
|      tahakom|  160|
|      tanafos|   30|
|       taqeem|    6|
+-------------+-----+



In [0]:
# remaining_cross_source_refs = (
#     tenders_unified_dedup
#     .filter(col("reference_number").isNotNull())
#     .groupBy("reference_number")
#     .agg(
#         countDistinct("source").alias("source_count")
#     )
#     .filter(col("source_count") > 1)
# )

# print(
#     "Remaining cross-source reference overlaps:",
#     remaining_cross_source_refs.count()
# )

# new cell here:

# ============================================================
# STEP: Check remaining cross-source reference overlaps
# Purpose:
# Verify whether any reference numbers still appear across
# multiple sources after the STC-Etimad deduplication.
# ============================================================

remaining_cross_source_refs = (
    tenders_unified_dedup
    .filter(col("reference_number").isNotNull())
    .groupBy("reference_number")
    .agg(
        countDistinct("source").alias("source_count")
    )
    .filter(col("source_count") > 1)
)

print(
    "Remaining cross-source reference overlaps:",
    remaining_cross_source_refs.count()
)

# Display remaining overlaps for investigation
remaining_cross_source_refs.show(30, truncate=False)

Remaining cross-source reference overlaps: 0
+----------------+------------+
|reference_number|source_count|
+----------------+------------+
+----------------+------------+



In [0]:
# ============================================================
# STEP: Profile tender status values across sources
# Purpose:
# Review the original source statuses and their current
# standardized status values before applying final status
# normalization rules to the unified Silver dataset.
# ============================================================

tenders_unified_dedup.groupBy(
    "source",
    "source_status",
    "status"
).count().orderBy(
    "source",
    col("count").desc()
).show(200, truncate=False)

+-------------+-------------+---------+-----+
|source       |source_status|status   |count|
+-------------+-------------+---------+-----+
|cst          |current      |CURRENT  |33   |
|etimad       |NULL         |NULL     |7358 |
|etimad       |معتمدة       |معتمدة   |52   |
|etimad_stc   |open         |OPEN     |805  |
|etimad_stc   |معتمدة       |معتمدة   |30   |
|etimad_stc   |ended        |ENDED    |10   |
|etimad_stc   |returned     |RETURNED |2    |
|forsah       |pending      |PENDING  |648  |
|forsah       |awarded      |AWARDED  |127  |
|forsah       |completed    |COMPLETED|110  |
|forsah       |canceled     |CANCELED |71   |
|forsah       |open         |OPEN     |45   |
|geo_resources|NULL         |NULL     |11   |
|nupco        |NULL         |NULL     |154  |
|sar          |NULL         |NULL     |3    |
|stc          |open         |OPEN     |1196 |
|stc          |ended        |ENDED    |457  |
|stc          |returned     |RETURNED |6    |
|tahakom      |Open         |OPEN 

In [0]:
# ============================================================
# STEP: Review overall tender status distribution
# Purpose:
# Identify all current status values in the unified Silver
# dataset and measure their frequency before applying final
# status normalization rules.
# ============================================================

tenders_unified_dedup.groupBy("status") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(100, truncate=False)

+---------+-----+
|status   |count|
+---------+-----+
|NULL     |7526 |
|OPEN     |2224 |
|PENDING  |648  |
|ENDED    |467  |
|AWARDED  |127  |
|COMPLETED|110  |
|معتمدة   |82   |
|CANCELED |71   |
|CURRENT  |33   |
|CLOSED   |17   |
|RETURNED |8    |
|CANCELLED|1    |
+---------+-----+



In [0]:
# ============================================================
# STEP: Standardize tender status values
# Purpose:
# Map different source-specific status values into a common
# set of standardized statuses while preserving the original
# source value in the source_status column.
# ============================================================

from pyspark.sql.functions import (
    col, when, upper, trim, lit
)

tenders_status_clean = (
    tenders_unified_dedup
    .withColumn(
        "status",
        when(col("status").isNull(), lit("UNKNOWN"))

        .when(
            upper(trim(col("status"))).isin("OPEN", "CURRENT"),
            lit("OPEN")
        )

        .when(
            upper(trim(col("status"))).isin("CLOSED", "ENDED"),
            lit("CLOSED")
        )

        .when(
            upper(trim(col("status"))) == "PENDING",
            lit("PENDING")
        )

        .when(
            upper(trim(col("status"))) == "AWARDED",
            lit("AWARDED")
        )

        .when(
            upper(trim(col("status"))) == "COMPLETED",
            lit("COMPLETED")
        )

        .when(
            upper(trim(col("status"))).isin("CANCELED", "CANCELLED"),
            lit("CANCELLED")
        )

        .when(
            trim(col("status")) == "معتمدة",
            lit("APPROVED")
        )

        .when(
            upper(trim(col("status"))) == "RETURNED",
            lit("RETURNED")
        )

        .otherwise(lit("UNKNOWN"))
    )
)

In [0]:
# checking again

tenders_status_clean.groupBy("status") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+---------+-----+
|   status|count|
+---------+-----+
|  UNKNOWN| 7526|
|     OPEN| 2257|
|  PENDING|  648|
|   CLOSED|  484|
|  AWARDED|  127|
|COMPLETED|  110|
| APPROVED|   82|
|CANCELLED|   72|
| RETURNED|    8|
+---------+-----+



In [0]:
print("Rows:", tenders_status_clean.count())
print("Columns:", len(tenders_status_clean.columns))
print(
    "Unique keys:",
    tenders_status_clean.select("tender_key").distinct().count()
)

Rows: 11314
Columns: 24
Unique keys: 11314


In [0]:
# ============================================================
# STEP: Profile tender type values across sources
# Purpose:
# Compare the original source tender types with their current
# standardized tender_type values and identify differences
# that may require additional normalization.
# ============================================================

tenders_status_clean.groupBy(
    "source",
    "source_tender_type",
    "tender_type"
).count().orderBy(
    "source",
    col("count").desc()
).show(200, truncate=False)

+-------------+-------------------------------------+-------------------------------------+-----+
|source       |source_tender_type                   |tender_type                          |count|
+-------------+-------------------------------------+-------------------------------------+-----+
|cst          |NULL                                 |NULL                                 |33   |
|etimad       |شراء مباشر                           |شراء مباشر                           |4557 |
|etimad       |منافسة عامة                          |منافسة عامة                          |2720 |
|etimad       |منافسة إتفاقية إطارية                |منافسة إتفاقية إطارية                |120  |
|etimad       |المنافسة على مرحلتين (المرحلة الاولى)|المنافسة على مرحلتين (المرحلة الاولى)|10   |
|etimad       |منافسات الشركات الكبرى               |منافسات الشركات الكبرى               |1    |
|etimad       |مسابقة                               |مسابقة                               |1    |
|etimad       |المشا

In [0]:
tenders_status_clean.groupBy("tender_type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(100, truncate=False)

+-------------------------------------+-----+
|tender_type                          |count|
+-------------------------------------+-----+
|شراء مباشر                           |5883 |
|منافسة عامة                          |4617 |
|مناقصة                               |210  |
|منافسة إتفاقية إطارية                |173  |
|RFQ                                  |151  |
|NULL                                 |58   |
|Direct Purchase                      |56   |
|Final Results                        |41   |
|Under Studying                       |35   |
|المنافسة على مرحلتين (المرحلة الاولى)|24   |
|فورية                                |19   |
|عامة                                 |9    |
|RFP                                  |9    |
|منافسات الشركات الكبرى               |6    |
|Public tender                        |6    |
|Cancelled                            |5    |
|Available Updated                    |3    |
|Initial Results                      |3    |
|المشاركة فى الدخل                

In [0]:
# ============================================================
# STEP: Standardize tender type values
# Purpose:
# Map different Arabic and English source tender types into
# a consistent set of standardized tender type categories.
# Unmapped values are labeled UNKNOWN while the original
# value remains available in source_tender_type.
# ============================================================


from pyspark.sql.functions import col, when, trim, upper, lit

tenders_type_clean = (
    tenders_status_clean
    .withColumn(
        "tender_type",

        # Missing
        when(
            col("tender_type").isNull(),
            lit("UNKNOWN")
        )

        # Direct Purchase
        .when(
            trim(col("tender_type")).isin(
                "شراء مباشر",
                "Direct Purchase"
            ),
            lit("DIRECT_PURCHASE")
        )

        # Public Tender
        .when(
            trim(col("tender_type")).isin(
                "منافسة عامة",
                "مناقصة",
                "عامة",
                "Public tender"
            ),
            lit("PUBLIC_TENDER")
        )

        # Framework Agreement
        .when(
            trim(col("tender_type")) == "منافسة إتفاقية إطارية",
            lit("FRAMEWORK_AGREEMENT")
        )

        # Two-stage tender
        .when(
            trim(col("tender_type")) ==
            "المنافسة على مرحلتين (المرحلة الاولى)",
            lit("TWO_STAGE_TENDER")
        )

        # RFQ
        .when(
            upper(trim(col("tender_type"))) == "RFQ",
            lit("RFQ")
        )

        # RFP
        .when(
            upper(trim(col("tender_type"))) == "RFP",
            lit("RFP")
        )

        # Tanafos
        .when(
            trim(col("tender_type")) == "فورية",
            lit("IMMEDIATE")
        )

        .when(
            trim(col("tender_type")) == "محدودة",
            lit("LIMITED_TENDER")
        )

        # Other Etimad/STC types
        .when(
            trim(col("tender_type")) == "منافسات الشركات الكبرى",
            lit("MAJOR_COMPANIES")
        )

        .when(
            trim(col("tender_type")) == "المزايدة العكسية الالكترونية",
            lit("REVERSE_AUCTION")
        )

        .when(
            trim(col("tender_type")) == "المشاركة فى الدخل",
            lit("REVENUE_SHARING")
        )

        .when(
            trim(col("tender_type")) == "مسابقة",
            lit("COMPETITION")
        )

        # Anything else, including NUPCO lifecycle labels
        .otherwise(lit("UNKNOWN"))
    )
)

In [0]:
# checking the tender type

tenders_type_clean.groupBy("tender_type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(50, truncate=False)

+-------------------+-----+
|tender_type        |count|
+-------------------+-----+
|DIRECT_PURCHASE    |5939 |
|PUBLIC_TENDER      |4842 |
|FRAMEWORK_AGREEMENT|173  |
|RFQ                |151  |
|UNKNOWN            |145  |
|TWO_STAGE_TENDER   |24   |
|IMMEDIATE          |19   |
|RFP                |9    |
|MAJOR_COMPANIES    |6    |
|REVENUE_SHARING    |2    |
|LIMITED_TENDER     |2    |
|REVERSE_AUCTION    |1    |
|COMPETITION        |1    |
+-------------------+-----+



In [0]:
print("Rows:", tenders_type_clean.count())
print("Columns:", len(tenders_type_clean.columns))

print(
    "Unique keys:",
    tenders_type_clean.select("tender_key").distinct().count()
)

Rows: 11314
Columns: 24
Unique keys: 11314


In [0]:
# ============================================================
# STEP: Check NULL values across the unified Silver dataset
# Purpose:
# Measure data completeness by counting NULL values in every
# column after deduplication and standardization.
# ===========================================================

from pyspark.sql.functions import col, count, when

quality_check = tenders_type_clean.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in tenders_type_clean.columns
])

quality_check.show(vertical=True)

-RECORD 0--------------------
 tender_key          | 0     
 source              | 0     
 source_tender_id    | 0     
 reference_number    | 1157  
 tender_name         | 0     
 tender_name_en      | 11154 
 description         | 11107 
 agency_name         | 1155  
 source_tender_type  | 58    
 tender_type         | 0     
 source_status       | 7526  
 status              | 0     
 publish_date        | 204   
 closing_date        | 1698  
 opening_date        | 7456  
 categories          | 367   
 regions             | 10283 
 estimated_value_min | 10313 
 estimated_value_max | 10313 
 document_price      | 3018  
 currency            | 2015  
 tender_url          | 7617  
 is_tech             | 10277 
 ingestion_timestamp | 8461  



In [0]:
# ============================================================
# STEP: Validate critical fields and date consistency
# Purpose:
# Verify that essential tender identifiers are present and
# detect invalid date sequences where closing or opening dates
# occur before the tender publication date.
# ============================================================

print(
    "Null tender_key:",
    tenders_type_clean.filter(col("tender_key").isNull()).count()
)

print(
    "Null source:",
    tenders_type_clean.filter(col("source").isNull()).count()
)

print(
    "Null source_tender_id:",
    tenders_type_clean.filter(col("source_tender_id").isNull()).count()
)

print(
    "Null tender_name:",
    tenders_type_clean.filter(col("tender_name").isNull()).count()
)

print(
    "Closing before publish:",
    tenders_type_clean.filter(
        col("closing_date") < col("publish_date")
    ).count()
)

print(
    "Opening before publish:",
    tenders_type_clean.filter(
        col("opening_date") < col("publish_date")
    ).count()
)

Null tender_key: 0
Null source: 0
Null source_tender_id: 0
Null tender_name: 0
Closing before publish: 0
Opening before publish: 1


In [0]:
# check each source and it's render counts

tenders_type_clean.groupBy("source") \
    .count() \
    .orderBy("source") \
    .show()

+-------------+-----+
|       source|count|
+-------------+-----+
|          cst|   33|
|       etimad| 7410|
|   etimad_stc|  847|
|       forsah| 1001|
|geo_resources|   11|
|        nupco|  154|
|          sar|    3|
|          stc| 1659|
|      tahakom|  160|
|      tanafos|   30|
|       taqeem|    6|
+-------------+-----+



In [0]:
# checking data quality and if there is incorrect values like opening date < publish date

tenders_type_clean.filter(
    col("opening_date") < col("publish_date")
).select(
    "source",
    "source_tender_id",
    "reference_number",
    "tender_name",
    "publish_date",
    "closing_date",
    "opening_date"
).show(truncate=False)

+-------+----------------+----------------+----------------------------------------------------------------------+------------+------------+------------+
|source |source_tender_id|reference_number|tender_name                                                           |publish_date|closing_date|opening_date|
+-------+----------------+----------------+----------------------------------------------------------------------+------------+------------+------------+
|tahakom|2290            |101000589       |توريد نظام VIM - توسيع نطاق أنظمة المركبات المتحركة (VIM) حتى عام 2026|2026-04-26  |2026-05-21  |2026-04-21  |
+-------+----------------+----------------+----------------------------------------------------------------------+------------+------------+------------+



In [0]:
tenders_type_clean.groupBy("source") \
    .count() \
    .orderBy("source") \
    .show()

+-------------+-----+
|       source|count|
+-------------+-----+
|          cst|   33|
|       etimad| 7410|
|   etimad_stc|  847|
|       forsah| 1001|
|geo_resources|   11|
|        nupco|  154|
|          sar|    3|
|          stc| 1659|
|      tahakom|  160|
|      tanafos|   30|
|       taqeem|    6|
+-------------+-----+



## final unified Silver Delta

In [0]:
# ============================================================
# STEP: Write the final unified Silver dataset
# Purpose:
# Persist the cleaned, standardized, and deduplicated unified
# tender dataset to ADLS as a Delta table.
#
# The unified table is rebuilt using overwrite because
# cross-source deduplication relationships may change when
# new source data arrives.
# ============================================================

final_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/"
    "tenders_unified"
)

(
    tenders_type_clean
    .write
    .format("delta")
    .mode("overwrite")
    .save(final_silver_path)
)

print("Unified Silver dataset written successfully.")

Unified Silver dataset written successfully.


In [0]:
# checking the silver final rows & columns and unique keys

silver_final = (
    spark.read
    .format("delta")
    .load(final_silver_path)
)

print("Rows:", silver_final.count())
print("Columns:", len(silver_final.columns))
print(
    "Unique keys:",
    silver_final.select("tender_key").distinct().count()
)

Rows: 11314
Columns: 24
Unique keys: 11314


In [0]:
silver_final.groupBy("source") \
    .count() \
    .orderBy("source") \
    .show()

+-------------+-----+
|       source|count|
+-------------+-----+
|          cst|   33|
|       etimad| 7410|
|   etimad_stc|  847|
|       forsah| 1001|
|geo_resources|   11|
|        nupco|  154|
|          sar|    3|
|          stc| 1659|
|      tahakom|  160|
|      tanafos|   30|
|       taqeem|    6|
+-------------+-----+



In [0]:
# ============================================================
# STEP: Validate all source-specific Silver Delta tables
# Purpose:
# Read each persisted Silver source table and verify that
# tender_key values are unique after incremental processing.
# ============================================================

silver_sources = {
    "NUPCO": silver_nupco_path,
    "STC": silver_stc_path,
    "Etimad": silver_etimad_path,
    "CST": silver_cst_path,
    "Tanafos": tanafos_silver_path,
    "TAQEEM": taqeem_silver_path,
    "TAHAKOM": tahakom_silver_path,
    "FORSAH": forsah_silver_path,
    "Geo Resources": geo_silver_path,
    "SAR": sar_silver_path
}

print("Silver validation")
print("-" * 55)

for source_name, path in silver_sources.items():
    df = spark.read.format("delta").load(path)

    total = df.count()
    unique_keys = df.select("tender_key").distinct().count()
    duplicates = total - unique_keys

    print(
        f"{source_name:15} "
        f"rows={total:<6} "
        f"unique={unique_keys:<6} "
        f"duplicates={duplicates}"
    )

Silver validation
-------------------------------------------------------
NUPCO           rows=154    unique=154    duplicates=0
STC             rows=2506   unique=2506   duplicates=0
Etimad          rows=8257   unique=8257   duplicates=0
CST             rows=33     unique=33     duplicates=0
Tanafos         rows=30     unique=30     duplicates=0
TAQEEM          rows=6      unique=6      duplicates=0
TAHAKOM         rows=160    unique=160    duplicates=0
FORSAH          rows=1001   unique=1001   duplicates=0
Geo Resources   rows=11     unique=11     duplicates=0
SAR             rows=3      unique=3      duplicates=0


In [0]:
# ============================================================
# STEP: Validate the persisted unified Silver Delta table
# Purpose:
# Read the final unified dataset back from ADLS and verify

unified_check = (
    spark.read
    .format("delta")
    .load(final_silver_path)
)

print("Unified rows:", unified_check.count())
print(
    "Unique tender keys:",
    unified_check.select("tender_key").distinct().count()
)

print(
    "Duplicate keys:",
    unified_check.count()
    - unified_check.select("tender_key").distinct().count()
)

Unified rows: 11314
Unique tender keys: 11314
Duplicate keys: 0


## Checking before creating unity catalog

In [0]:
# ============================================================
# STEP: Inspect unified Silver tender dataset
# Purpose:
# Verify the existing unified Silver Delta dataset, its schema,
# and record count before registering it in Unity Catalog.
# ============================================================

unified_silver_path = (
    "abfss://silver@tenderdatalake11.dfs.core.windows.net/"
    "tenders_unified"
)

# Read the existing unified Silver Delta dataset.
unified_silver = (
    spark.read
    .format("delta")
    .load(unified_silver_path)
)

# Verify the standardized schema.
unified_silver.printSchema()

# Check the current number of unified tender records.
print(
    "Unified Silver record count:",
    unified_silver.count()
)

# Preview records from the unified dataset.
display(unified_silver.limit(10))

root
 |-- tender_key: string (nullable = true)
 |-- source: string (nullable = true)
 |-- source_tender_id: string (nullable = true)
 |-- reference_number: string (nullable = true)
 |-- tender_name: string (nullable = true)
 |-- tender_name_en: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- source_tender_type: string (nullable = true)
 |-- tender_type: string (nullable = true)
 |-- source_status: string (nullable = true)
 |-- status: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- closing_date: date (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- regions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- estimated_value_min: decimal(18,2) (nullable = true)
 |-- estimated_value_max: decimal(18,2) (nullable = true)
 |-- document_price: decimal(18,2) (nullable = true

tender_key,source,source_tender_id,reference_number,tender_name,tender_name_en,description,agency_name,source_tender_type,tender_type,source_status,status,publish_date,closing_date,opening_date,categories,regions,estimated_value_min,estimated_value_max,document_price,currency,tender_url,is_tech,ingestion_timestamp
f7c02d0d5698c7603370b4c730c056833dbcb8917fa03159fa279fb933202f54,tanafos,20794,2608149357,توريد وتركيب وتشغيل نظام كاميرات مراقبة لمبنى لجنة رعاية السجناء بمنطقة الرياض,null,null,اللجنة الوطنية لرعاية السجناء والمفرج عنهم وأسرهم بمنطقة الرياض,عامة,PUBLIC_TENDER,11,CLOSED,2026-08-06,2026-08-16,null,"List(الأمن والسلامة / أنشطة الأمن و السلامة, الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",List(الرياض),null,null,230.00,SAR,https://tanafos.sa/bids/view-bid-details/20794,true,null
debeec57edfbce9b7606e7f492bae7bb1caa6668e6effd39d3daff8fc4005658,tanafos,20773,2607149566,تطوير نظام إلكتروني ثنائي اللغة لإدارة وتسجيل دخول الزوار,null,null,مؤسسة الملك خالد الخيرية,عامة,PUBLIC_TENDER,11,CLOSED,2026-07-13,2026-07-30,null,"List(الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",List(الرياض),null,null,250.00,SAR,https://tanafos.sa/bids/view-bid-details/20773,true,null
3396998836e9923f68c9b24a3dae05d059060f89626f788985fbbfb7980b699e,tanafos,20737,2605153168,التسويق الإلكتروني للحملة الوطنية للتوعية بسرطان الثدي – أكتوبر 2026,null,null,جمعية زهرة لسرطان الثدي,فورية,IMMEDIATE,10,OPEN,2026-05-31,null,null,"List(الإعلام والنشر والتوزيع / خدمات النشر والطباعة والدعاية والاعلان والإنتاج الفني, الاتصالات وتقنية المعلومات / تقنية المعلومات)",List(جميع مناطق المملكة),null,null,1150.00,SAR,https://tanafos.sa/bids/view-bid-details/20737,true,null
5dcf07f9a0755dcecc5259ae39a6424019778251c47911cd73a4402aa5880707,tanafos,20711,2603137964,توريد وتركيب المستلزمات التقنية للمقر الرئيسي لجمعية العناية بمساجد الطرق,null,null,جمعية العناية بمساجد الطرق,محدودة,LIMITED_TENDER,10,CLOSED,2026-03-28,2026-04-06,null,"List(التجارة / تجارة الأدوات والآلات والأجهزة, الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",List(الرياض),null,null,1437.50,SAR,https://tanafos.sa/bids/view-bid-details/20711,true,null
522033cb6ac317d2a25b8c88a4b68337fbf5ffbf7af5b9a9fcbc16a52ef5fda8,tanafos,20688,2602156238,توريد جهاز تبرع ذاتي,null,null,الجمعية الخيرية لرعاية الأيتام بمنطقة حائل (رفاق),فورية,IMMEDIATE,10,OPEN,2026-02-11,null,null,"List(الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات, التجارة / تجارة الأدوات والآلات والأجهزة)",List(حائل),null,null,230.00,SAR,https://tanafos.sa/bids/view-bid-details/20688,true,null
77b9c11db7a3bfdc20db8e4b149a9f53c478277e921b95163f17320dc39ca97a,tanafos,20653,2601148826,"تنفيذ الحملة التسويقية ""رسالة تلهم وأثر مستدام""",null,null,جمعية تنمية الأيتام بمنطقة المدينة المنورة,عامة,PUBLIC_TENDER,10,CLOSED,2026-01-14,2026-01-27,null,"List(الإعلام والنشر والتوزيع / خدمات النشر والطباعة والدعاية والاعلان والإنتاج الفني, الاتصالات وتقنية المعلومات / تقنية المعلومات, الخدمات الإدارية وخدمات الدعم / تنظيم وإدارة المعارض والمؤتمرات)",List(المدينة المنورة),null,null,750.00,SAR,https://tanafos.sa/bids/view-bid-details/20653,true,null
c572a2da442a5f519b71d553ec3e2d5591bd28040a42df63c29e9530f8da9ccd,tanafos,20616,2512144518,تطوير واستضافة وتشغيل البوابة الإلكترونية وتطبيق صوت المستهلك,null,null,جمعية حماية المستهلك,عامة,PUBLIC_TENDER,10,CLOSED,2025-12-10,2025-12-22,null,"List(الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية المعلومات)",List(الرياض),null,null,999.99,SAR,https://tanafos.sa/bids/view-bid-details/20616,true,null
5555e0e5330476629b422693232114e801abebed00ddf08268efac4a713d2435,tanafos,20573,2510154676,توريد أجهزة حاسب آلي وملحقاتها,null,null,الجمعية الخيرية لرعاية الايتام بمنطقة المدينة المنورة تكافل,فورية,IMMEDIATE,10,OPEN,2025-10-30,null,null,"List(التجارة / تجارة الأدوات والآلات والأجهزة, الاتصالات وتقنية المعلومات / الإتصالات, الاتصالات وتقنية المعلومات / تقنية الم

In [0]:
# ============================================================
# STEP: Profile Silver data for Gold star-schema design
# Purpose:
# Understand the actual source, agency, category, and region
# values before creating Gold fact and dimension tables.
# ============================================================

silver_unified = spark.table(
    "tenderdatabricks.silver.tenders_unified"
)

# ------------------------------------------------------------
# 1. Sources
# Check the distinct sources and number of tenders per source.
# ------------------------------------------------------------

display(
    silver_unified
    .groupBy("source")
    .count()
    .orderBy("source")
)


# ------------------------------------------------------------
# 2. Agencies
# Check how many different agencies exist and how often
# agency_name is missing.
# ------------------------------------------------------------

silver_unified.select("agency_name").distinct().show(
    50,
    truncate=False
)

print(
    "Distinct agencies:",
    silver_unified.select("agency_name").distinct().count()
)

print(
    "Null agencies:",
    silver_unified.filter("agency_name IS NULL").count()
)


# ------------------------------------------------------------
# 3. Categories
# Inspect the array values currently stored in Silver.
# ------------------------------------------------------------

display(
    silver_unified
    .select("categories")
    .where("categories IS NOT NULL")
    .limit(20)
)


# ------------------------------------------------------------
# 4. Regions
# Inspect the array values currently stored in Silver.
# ------------------------------------------------------------

display(
    silver_unified
    .select("regions")
    .where("regions IS NOT NULL")
    .limit(20)
)

source,count
cst,33
etimad,7410
etimad_stc,847
forsah,1001
geo_resources,11
nupco,154
sar,3
stc,1659
tahakom,160
tanafos,30


+--------------------------------------------------------+
|agency_name                                             |
+--------------------------------------------------------+
|مدينة الملك عبدالعزيز للعلوم والتقنية                   |
|المديرية العامة للسجون                                  |
|الرئاسة العامة للبحوث العلمية والإفتاء                  |
|برنامج الخدمات المشتركة                                 |
|وزارة البيئة والمياه والزراعة                           |
|الهيئة العامة للعناية بشؤون المسجد الحرام والمسجد النبوي|
|الهيئة الوطنية للأمن السيبراني                          |
|صندوق التنمية الوطني                                    |
|مستشفى المدينة المنورة                                  |
|النيابة العامة                                          |
|202408011                                               |
|هيئة السوق المالية                                      |
|المراسم الملكية                                         |
|بلدية محافظة الشماسية                                  

categories
List(تقنية المعلومات)
List(تقنية المعلومات)
List(تقنية المعلومات)
List(تقنية المعلومات)
List(تقنية المعلومات)
List(تجارة الأدوات والآلات والأجهزة)
List(تقنية المعلومات)
List(تقنية المعلومات)
List(تقنية المعلومات)
List(الإتصالات)


regions
List(الرياض)
List(الرياض)
List(الرياض)
List(جدة)
List(الرياض)
List(العقيقة)
List(جدة)
List(جدة)
List(الدرعية)
List(جدة)
